# Block 1 - Research Configuration, Universe & Identity Engine

## Description

This notebook is the authoritative entry point for the global point-in-time equity research database. It establishes the research configuration, reconstructs the historical investible universe, resolves the global issuer/security/listing identity graph, performs semantic theme classification, assigns downstream source routes, validates the resulting contracts, and publishes the canonical Block 1 datasets used by all subsequent regional and research blocks.

The notebook is deliberately **industry-agnostic**. The current research theme, ETF universe, date range and other project-specific choices are supplied through configuration rather than embedded in the platform logic, allowing the same engine to be reused for different industries, themes and investment universes.

### Core responsibilities

- Load the research configuration, project metadata and run metadata.
- Reconstruct the historical investible universe from configured SEC N-PORT ETF filings, while allowing manually supplied securities to enter through the same canonical pipeline.
- Preserve point-in-time source availability, filing lineage and observation-level evidence so that later research can be reproduced without look-ahead bias.
- Resolve identity through the canonical hierarchy:

  **Economic Issuer → Legal Entity → Security → Listing**

- Maintain legal-entity, security, identifier and listing histories without collapsing legitimate corporate restructurings, successor entities, multiple securities or multiple listings into a single identity layer.
- Use deterministic identifier evidence and authoritative external reference data to reconcile identity, while quarantining genuine ambiguity rather than forcing a match.
- Use the **GLEIF public API** as the primary industry-agnostic external legal-entity reference layer. GLEIF responses are cached locally for reproducibility and efficiency. No GLEIF API key is required.
- Use GLEIF legal names, alternative/transliterated names, jurisdiction, entity status, successor relationships and other available reference evidence when validating source-reported LEIs.
- Recognise cross-language and non-comparable-script names so that low lexical similarity alone is not treated as evidence of an identity contradiction.
- Use **OpenAI semantic review** where semantic interpretation is appropriate, including cross-language or historical-name relationships, while keeping authoritative identity decisions deterministic and evidence-based.
- Use **OpenAI as the default semantic classifier for research-theme relevance and subindustry classification**. Classification is cache-first: a deterministic semantic cache is reused when available, and a cache miss invokes OpenAI automatically when an API key is available.
- Validate structured AI output and preserve abstentions or low-confidence cases as unclassified rather than forcing a semantic label.
- Resolve listing country and downstream source routing even where the exact historical exchange MIC cannot be established.
- Distinguish missing source-level issuer evidence from genuinely unresolved canonical identity when strong security identifiers already resolve the observation.
- Preserve unresolved or contradictory evidence in explicit quarantine/QC datasets for later review.
- Publish canonical Parquet outputs and a run manifest for downstream reproducibility and quality control.

### Identity and evidence policy

Identity is not inferred from a single company-name string. The engine combines deterministic identifier evidence, authoritative registry evidence and temporal/legal-entity relationships. LEIs identify legal entities rather than economic issuers; consequently, multiple legal entities may legitimately belong to one economic issuer through restructurings or succession. Likewise, a security may have legal-entity history while still mapping to exactly one economic issuer.

The critical canonical invariant is:

**one security → one economic issuer**

Source-reported LEIs, names, countries and identifiers are retained as evidence and are not silently overwritten. Invalid identifiers and genuine contradictions are excluded from authoritative identity construction or quarantined with explicit reason codes.

### AI policy

AI is used for **semantics, not truth**. It may interpret company-name relationships, classify research-theme relevance, identify subindustries and assist with other semantic tasks. It does not authoritatively assign internal IDs, LEIs, CIKs, ISIN ownership, ticker history, timestamps, point-in-time availability, arithmetic, observation IDs, version ordering, source routing or persistence.

For theme classification, the normal execution path is:

**resolved economic issuer → semantic cache → OpenAI on cache miss → structured-output validation → cache**

If OpenAI is required but unavailable, or if the model abstains or returns insufficient confidence, the issuer remains unclassified with an explicit reason rather than receiving a fabricated classification.

The OpenAI API key is loaded automatically from either the `OPENAI_API_KEY` environment variable or a Google Colab Secret named exactly `OPENAI_API_KEY`. There is no manual AI enable/disable switch.

### Design principles

1. **Configuration, not hard-coding.** Research themes, ETF universes and project-specific settings belong in configuration rather than executable platform logic.
2. **One global identity system.** All downstream blocks inherit the same issuer, legal-entity, security and listing identities.
3. **Point-in-time availability is distinct from economic period dates.** Public availability and source lineage are preserved independently of the period being described.
4. **Authoritative evidence and deterministic rules govern truth.** Identity, time, arithmetic, routing and persistence are not delegated to AI.
5. **AI handles semantic interpretation.** Semantic outputs are structured, validated, cached and allowed to abstain.
6. **Raw evidence remains separate from canonical truth.** Source observations, external-reference evidence and quarantine records are retained alongside canonical masters.
7. **Parquet is canonical.** Human-readable displays and other formats are secondary to the reproducible Parquet data products.

### External services

- **SEC EDGAR / N-PORT:** historical universe reconstruction and source filing evidence.
- **GLEIF:** authoritative external legal-entity reference and LEI validation; public access, no API key required.
- **OpenAI:** semantic name review and default research-theme classification; requires `OPENAI_API_KEY` when a new semantic classification or review is needed.

The notebook is designed to run deterministically from preserved source evidence and cached external/semantic responses wherever possible, while making every unresolved, quarantined or AI-dependent decision visible through explicit diagnostics and quality-control outputs.

In [48]:
# 1. INSTALL DEPENDENCIES
# Designed for Google Colab or a standard Python 3.11+ environment.

%pip -q install pandas pyarrow requests pyyaml openpyxl python-dateutil openai


In [94]:
# 2. IMPORTS, RESEARCH CONFIGURATION AND PROJECT PATHS

from __future__ import annotations

import io
import os
import re
import json
import time
import math
import yaml
import hashlib
import zipfile
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime, date, timezone
from typing import Any, Iterable, Optional
from itertools import combinations
from difflib import SequenceMatcher
from collections import Counter

import numpy as np
import pandas as pd
import requests
from dateutil.parser import parse as parse_datetime

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# RESEARCH CONFIGURATION
# ---------------------------------------------------------------------
# Edit this dictionary (or replace it with yaml.safe_load on a YAML file)
# to run the engine for another research theme or universe.

RESEARCH_CONFIG = {
    "project": {
        "project_name": "Global Automotive PIT Research Database",
        "project_version": "1.0",
        "research_theme": "Automotive",
    },
    "universe": {
        "etfs": ["DRIV", "CARZ", "IDRV", "KARS"],

        # Optional SEC N-PORT fund/series-name aliases.
        # These are project configuration, not platform logic.
        "fund_aliases": {
            "DRIV": [
                "Global X Autonomous & Electric Vehicles ETF",
                "Global X Autonomous and Electric Vehicles ETF",
            ],
            "CARZ": [
                "First Trust S-Network Future Vehicles & Technology ETF",
                "First Trust NASDAQ Global Auto Index Fund",
            ],
            "IDRV": [
                "iShares Self-Driving EV and Tech ETF",
                "iShares Self-Driving EV & Tech ETF",
            ],
            "KARS": [
                "KraneShares Electric Vehicles and Future Mobility Index ETF",
                "KraneShares Electric Vehicles & Future Mobility Index ETF",
            ],
        },

        "individual_securities": [],
        "start_date": "2019-10-01",
        "end_date": None,  # None = current date
        "include_historical_constituents": True,
    },
    "theme": {
        "classify_theme_relevance": True,
        "collect_theme_specific_kpis": True,
    },
    "fundamentals": {
        "collect_standard_financials": True,
        "preserve_versions": True,
        "point_in_time": True,
    },
    "identity_reference_data": {
        "enabled": True,
        "gleif_enabled": True,
        "network_enabled": True,
        "cache_days": 30,
        "request_delay_seconds": 0.08,
        "timeout_seconds": 20,
        "strict_external_validation": False,
    },
    "ai": {
        "provider": "openai",

        # Step 10 — semantic legal-name relationship review only.
        # Never authoritative for corporate identity.
        "identity_semantic_model": "gpt-5-mini",

        # Step 12 — gated theme classification.
        "theme_fast_model": "gpt-5-nano",
        "theme_web_model": "gpt-5.6-luna",
        "theme_fast_accept_confidence": 0.90,

        # Step 12C — taxonomy harmonisation.
        "taxonomy_harmonisation_model": "gpt-5.6-terra",

        "semantic_cache": True,
        "require_structured_output": True,
        "prompt_version": "theme_classifier_1.0",
        "fallback_policy": "AUTO_WHEN_NEEDED",
    },
    "output": {
        "parquet": True,
        "output_name": None,  # None = derive publication name from project configuration
    },
}

# ---------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine"
LOCAL_PROJECT_ROOT = "./research_database"
USE_CACHE = True
SEC_REQUEST_DELAY_SECONDS = 0.12
OVERWRITE_OUTPUTS = True
PERSIST_RAW_NPORT_HOLDINGS = True

SEC_USER_AGENT = os.getenv(
    "SEC_USER_AGENT",
    "PIT Equity Research Engine research@example.com"
)

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PROJECT_ROOT = Path(GOOGLE_DRIVE_ROOT)
    except Exception:
        PROJECT_ROOT = Path(LOCAL_PROJECT_ROOT)
else:
    PROJECT_ROOT = Path(LOCAL_PROJECT_ROOT)

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DIR = DATA_ROOT / "raw" / "block_1"
CACHE_DIR = DATA_ROOT / "raw" / "sec_nport_cache"
INTERIM_DIR = DATA_ROOT / "interim" / "block_1"
CANONICAL_DIR = DATA_ROOT / "canonical" / "block_1"
REGISTRY_DIR = DATA_ROOT / "registries"
REFERENCE_CACHE_DIR = DATA_ROOT / "reference_cache"
GLEIF_CACHE_DIR = REFERENCE_CACHE_DIR / "gleif"
MANIFEST_DIR = DATA_ROOT / "manifests"

for p in [RAW_DIR, CACHE_DIR, INTERIM_DIR, CANONICAL_DIR, REGISTRY_DIR, REFERENCE_CACHE_DIR, GLEIF_CACHE_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BLOCK_MANIFEST_PATH = MANIFEST_DIR / "block_1_manifest.json"

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def config_hash(config: dict) -> str:
    payload = json.dumps(config, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

RESEARCH_THEME = RESEARCH_CONFIG["project"]["research_theme"]
TARGET_FUNDS = [str(x).upper().strip() for x in RESEARCH_CONFIG["universe"]["etfs"]]
START_DATE = pd.Timestamp(RESEARCH_CONFIG["universe"]["start_date"]).normalize()
END_DATE = (
    pd.Timestamp(RESEARCH_CONFIG["universe"]["end_date"]).normalize()
    if RESEARCH_CONFIG["universe"]["end_date"]
    else pd.Timestamp.utcnow().tz_localize(None).normalize()
)

CONFIG_HASH = config_hash(RESEARCH_CONFIG)
PROJECT_NAMESPACE = (
    RESEARCH_CONFIG["project"]["project_name"].strip().upper()
    + "|" + CONFIG_HASH[:16]
)
RESEARCH_PROJECT_ID = "PRJ_" + hashlib.sha256(PROJECT_NAMESPACE.encode()).hexdigest()[:16].upper()
RUN_ID = "RUN_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + CONFIG_HASH[:8].upper()

print("Research project:", RESEARCH_CONFIG["project"]["project_name"])
print("Theme:", RESEARCH_THEME)
print("Funds:", TARGET_FUNDS)
print("Date range:", START_DATE.date(), "to", END_DATE.date())
print("Research project ID:", RESEARCH_PROJECT_ID)
print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Research project: Global Automotive PIT Research Database
Theme: Automotive
Funds: ['DRIV', 'CARZ', 'IDRV', 'KARS']
Date range: 2019-10-01 to 2026-08-30
Research project ID: PRJ_C8F90B7B41558D68
Run ID: RUN_20260830T043356Z_1A2DA849
Project root: /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine


In [50]:
# 3. CANONICAL SCHEMAS, ENUMS AND VALIDATION CONTRACTS

IDENTITY_NAMESPACE_VERSION = "IDENTITY_NAMESPACE_1"
SCHEMA_VERSION = "1.0.0"

ECONOMIC_ISSUER_COLUMNS = [
    "economic_issuer_id",
    "issuer_name",
    "issuer_name_normalised",
    "legal_name",
    "country_of_domicile",
    "headquarters_country",
    "industry_theme",
    "theme_relevance",
    "theme_subindustry",
    "theme_classification_method",
    "theme_confidence",
    "active_from",
    "active_to",
    "predecessor_issuer_id",
    "successor_issuer_id",
]

LEGAL_ENTITY_COLUMNS = [
    "legal_entity_id",
    "economic_issuer_id",
    "legal_entity_name",
    "entity_country",
    "lei",
    "cik",
    "edinet_code",
    "dart_corp_code",
    "cninfo_entity_id",
    "valid_from",
    "valid_to",
]

SECURITY_COLUMNS = [
    "security_id",
    "economic_issuer_id",
    "legal_entity_id",
    "security_name",
    "security_type",
    "share_class",
    "isin",
    "cusip",
    "sedol",
    "figi",
    "currency",
    "valid_from",
    "valid_to",
]

LISTING_COLUMNS = [
    "listing_id",
    "security_id",
    "economic_issuer_id",
    "ticker",
    "exchange_name",
    "exchange_mic",
    "listing_country",
    "trading_currency",
    "primary_listing_flag",
    "valid_from",
    "valid_to",
    "source_route",
    "source_system",
]

UNIVERSE_MEMBERSHIP_COLUMNS = [
    "research_project_id",
    "fund_id",
    "fund_ticker",
    "security_id",
    "listing_id",
    "economic_issuer_id",
    "membership_start_date",
    "membership_end_date",
    "first_observed_date",
    "last_observed_date",
    "weight",
    "shares",
    "market_value",
    "source_filing_id",
    "source_available_datetime",
    "source_observation_id",
    "universe_source_type",
]

ROUTING_COLUMNS = [
    "economic_issuer_id",
    "security_id",
    "listing_id",
    "ticker",
    "exchange_mic",
    "listing_country",
    "primary_source_engine",
    "primary_source_system",
    "secondary_source_system",
    "processing_priority",
    "route_reason",
    "valid_from",
    "valid_to",
]

THEME_CLASSIFICATION_COLUMNS = [
    "economic_issuer_id",
    "industry_theme",
    "theme_relevance",
    "theme_subindustry",
    "theme_classification_method",
    "theme_confidence",
    "classification_reason",
    "ai_assisted",
    "ai_task_id",
]

IDENTIFIER_HISTORY_COLUMNS = [
    "entity_type",
    "entity_id",
    "identifier_type",
    "identifier_value",
    "valid_from",
    "valid_to",
    "first_observed_date",
    "last_observed_date",
    "source_system",
    "source_observation_id",
    "is_primary",
]

VALID_THEME_RELEVANCE = {"Core", "High", "Moderate", "Peripheral", "Unclassified"}
VALID_SOURCE_ENGINES = {
    "STRUCTURED_XBRL",
    "EUROPE",
    "EAST_ASIA",
    "CHINA_HK",
    "RESIDUAL_GLOBAL",
}

def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[columns]

def assert_unique_non_null(df: pd.DataFrame, column: str, table_name: str) -> None:
    if column not in df.columns:
        raise AssertionError(f"{table_name}: missing key column {column}")
    if df[column].isna().any():
        raise AssertionError(f"{table_name}: {column} contains nulls")
    if df[column].duplicated().any():
        dupes = df.loc[df[column].duplicated(keep=False), column].astype(str).head(10).tolist()
        raise AssertionError(f"{table_name}: duplicate {column}: {dupes}")

def assert_foreign_key(
    child: pd.DataFrame,
    child_column: str,
    parent: pd.DataFrame,
    parent_column: str,
    child_name: str,
    allow_null: bool = True,
) -> None:
    vals = child[child_column]
    if allow_null:
        vals = vals.dropna()
    missing = set(vals.astype(str)) - set(parent[parent_column].dropna().astype(str))
    if missing:
        raise AssertionError(
            f"{child_name}: {len(missing)} {child_column} values absent from parent; "
            f"examples={list(sorted(missing))[:10]}"
        )

print("Canonical schema version:", SCHEMA_VERSION)
print("Identity namespace:", IDENTITY_NAMESPACE_VERSION)


Canonical schema version: 1.0.0
Identity namespace: IDENTITY_NAMESPACE_1


In [51]:
# 4. NORMALISATION, STABLE IDS, EXCHANGE REGISTRY AND SOURCE ROUTING REGISTRY

NULL_STRINGS = {"", "NONE", "NULL", "NAN", "N/A", "NA", "<NA>"}

COUNTRY_ALIASES = {
    "UNITED STATES": "US", "USA": "US", "U.S.": "US", "US": "US",
    "UNITED KINGDOM": "GB", "UK": "GB", "GREAT BRITAIN": "GB", "GB": "GB",
    "GERMANY": "DE", "FRANCE": "FR", "ITALY": "IT", "SPAIN": "ES",
    "NETHERLANDS": "NL", "SWEDEN": "SE", "NORWAY": "NO", "DENMARK": "DK",
    "FINLAND": "FI", "SWITZERLAND": "CH", "AUSTRIA": "AT", "BELGIUM": "BE",
    "JAPAN": "JP", "SOUTH KOREA": "KR", "KOREA": "KR",
    "CHINA": "CN", "MAINLAND CHINA": "CN", "HONG KONG": "HK",
    "TAIWAN": "TW", "AUSTRALIA": "AU", "CANADA": "CA",
    "INDIA": "IN", "SINGAPORE": "SG", "INDONESIA": "ID",
}

EXCHANGE_REGISTRY = {
    "XNYS": {"exchange_name": "New York Stock Exchange", "country": "US", "engine": "STRUCTURED_XBRL", "system": "SEC"},
    "XNAS": {"exchange_name": "Nasdaq", "country": "US", "engine": "STRUCTURED_XBRL", "system": "SEC"},
    "ARCX": {"exchange_name": "NYSE Arca", "country": "US", "engine": "STRUCTURED_XBRL", "system": "SEC"},
    "XLON": {"exchange_name": "London Stock Exchange", "country": "GB", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XETR": {"exchange_name": "Xetra", "country": "DE", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XFRA": {"exchange_name": "Frankfurt Stock Exchange", "country": "DE", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XPAR": {"exchange_name": "Euronext Paris", "country": "FR", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XMIL": {"exchange_name": "Borsa Italiana", "country": "IT", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XMAD": {"exchange_name": "Bolsa de Madrid", "country": "ES", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XAMS": {"exchange_name": "Euronext Amsterdam", "country": "NL", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XSTO": {"exchange_name": "Nasdaq Stockholm", "country": "SE", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XSWX": {"exchange_name": "SIX Swiss Exchange", "country": "CH", "engine": "EUROPE", "system": "ESEF_OR_NATIONAL"},
    "XTKS": {"exchange_name": "Tokyo Stock Exchange", "country": "JP", "engine": "EAST_ASIA", "system": "EDINET"},
    "XKRX": {"exchange_name": "Korea Exchange", "country": "KR", "engine": "EAST_ASIA", "system": "OPENDART"},
    "XKOS": {"exchange_name": "KOSDAQ", "country": "KR", "engine": "EAST_ASIA", "system": "OPENDART"},
    "XSHG": {"exchange_name": "Shanghai Stock Exchange", "country": "CN", "engine": "CHINA_HK", "system": "CNINFO"},
    "XSHE": {"exchange_name": "Shenzhen Stock Exchange", "country": "CN", "engine": "CHINA_HK", "system": "CNINFO"},
    "XBSE": {"exchange_name": "Beijing Stock Exchange", "country": "CN", "engine": "CHINA_HK", "system": "CNINFO"},
    "XHKG": {"exchange_name": "Hong Kong Exchanges and Clearing", "country": "HK", "engine": "CHINA_HK", "system": "HKEX"},
    "XASX": {"exchange_name": "Australian Securities Exchange", "country": "AU", "engine": "RESIDUAL_GLOBAL", "system": "ASX_OR_IR"},
    "XTSE": {"exchange_name": "Toronto Stock Exchange", "country": "CA", "engine": "RESIDUAL_GLOBAL", "system": "SEDAR_PLUS_OR_IR"},
    "XTAI": {"exchange_name": "Taiwan Stock Exchange", "country": "TW", "engine": "RESIDUAL_GLOBAL", "system": "TWSE_OR_IR"},
    "XSES": {"exchange_name": "Singapore Exchange", "country": "SG", "engine": "RESIDUAL_GLOBAL", "system": "SGX_OR_IR"},
    "XBOM": {"exchange_name": "BSE India", "country": "IN", "engine": "RESIDUAL_GLOBAL", "system": "BSE_OR_IR"},
    "XNSE": {"exchange_name": "National Stock Exchange of India", "country": "IN", "engine": "RESIDUAL_GLOBAL", "system": "NSE_OR_IR"},
}

EXCHANGE_ALIASES = {
    "NYSE": "XNYS", "NEW YORK STOCK EXCHANGE": "XNYS",
    "NASDAQ": "XNAS", "NASDAQ GLOBAL SELECT MARKET": "XNAS",
    "LSE": "XLON", "LONDON STOCK EXCHANGE": "XLON",
    "XETRA": "XETR", "FRANKFURT": "XFRA",
    "EURONEXT PARIS": "XPAR", "BORSA ITALIANA": "XMIL",
    "TOKYO": "XTKS", "TOKYO STOCK EXCHANGE": "XTKS", "TSE": "XTKS",
    "KOREA EXCHANGE": "XKRX", "KRX": "XKRX", "KOSDAQ": "XKOS",
    "SHANGHAI": "XSHG", "SHANGHAI STOCK EXCHANGE": "XSHG",
    "SHENZHEN": "XSHE", "SHENZHEN STOCK EXCHANGE": "XSHE",
    "HONG KONG": "XHKG", "HONG KONG STOCK EXCHANGE": "XHKG", "HKEX": "XHKG",
    "ASX": "XASX", "AUSTRALIAN SECURITIES EXCHANGE": "XASX",
    "TORONTO": "XTSE", "TORONTO STOCK EXCHANGE": "XTSE",
}

TICKER_SUFFIX_TO_MIC = {
    ".L": "XLON", ".DE": "XETR", ".F": "XFRA", ".PA": "XPAR", ".MI": "XMIL",
    ".MC": "XMAD", ".AS": "XAMS", ".ST": "XSTO", ".SW": "XSWX",
    ".T": "XTKS", ".KS": "XKRX", ".KQ": "XKOS",
    ".SS": "XSHG", ".SZ": "XSHE", ".BJ": "XBSE", ".HK": "XHKG",
    ".AX": "XASX", ".TO": "XTSE", ".TW": "XTAI", ".SI": "XSES",
}

# Country-level MIC inference is deliberately conservative. These are markets
# where an ordinary equity can be assigned to a primary exchange with high
# practical confidence from listing-country evidence alone. Ambiguous markets
# such as the US, Canada, India and Korea remain MIC-unresolved unless stronger
# evidence is available, while still receiving an authoritative source route.
COUNTRY_DEFAULT_MIC = {
    "GB": "XLON", "DE": "XETR", "FR": "XPAR", "IT": "XMIL", "ES": "XMAD",
    "NL": "XAMS", "SE": "XSTO", "CH": "XSWX", "JP": "XTKS", "HK": "XHKG",
    "AU": "XASX", "TW": "XTAI", "SG": "XSES",
}

EUROPE_COUNTRIES = {"GB","DE","FR","IT","ES","NL","SE","NO","DK","FI","CH","AT","BE","IE","PT"}

def clean_string(value: Any) -> Optional[str]:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    s = str(value).strip()
    return None if s.upper() in NULL_STRINGS else s

def normalise_identifier(value: Any) -> Optional[str]:
    s = clean_string(value)
    if not s:
        return None
    return re.sub(r"[^A-Z0-9]", "", s.upper())

def normalise_ticker(value: Any) -> Optional[str]:
    s = clean_string(value)
    if not s:
        return None
    return re.sub(r"\s+", "", s.upper())

def normalise_name(value: Any) -> Optional[str]:
    s = clean_string(value)
    if not s:
        return None
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper().replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    legal_suffixes = (
        r"\b(INCORPORATED|INC|CORPORATION|CORP|COMPANY|CO|LIMITED|LTD|PLC|"
        r"AG|SA|SE|NV|BV|SPA|S P A|AB|ASA|OYJ|KK|K K|CO LTD|GROUP|HOLDINGS?)\b"
    )
    s = re.sub(legal_suffixes, " ", s)
    return re.sub(r"\s+", " ", s).strip() or None

def normalise_country(value: Any) -> Optional[str]:
    s = clean_string(value)
    if not s:
        return None
    u = s.upper()
    return COUNTRY_ALIASES.get(u, u if len(u) == 2 else u)

def normalise_currency(value: Any) -> Optional[str]:
    s = clean_string(value)
    return s.upper() if s else None

def extract_isin_country(isin: Any) -> Optional[str]:
    s = normalise_identifier(isin)
    if s and len(s) == 12 and s[:2].isalpha():
        return s[:2]
    return None

def stable_hash(*parts: Any, length: int = 20) -> str:
    payload = "|".join("" if p is None else str(p).strip().upper() for p in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:length].upper()

def make_entity_id(prefix: str, entity_type: str, canonical_key: str) -> str:
    return f"{prefix}_{stable_hash(IDENTITY_NAMESPACE_VERSION, entity_type, canonical_key)}"

def infer_mic_from_ticker(ticker: Any) -> Optional[str]:
    t = normalise_ticker(ticker)
    if not t:
        return None
    for suffix, mic in sorted(TICKER_SUFFIX_TO_MIC.items(), key=lambda x: -len(x[0])):
        if t.endswith(suffix):
            return mic
    return None

def infer_mic_from_numeric_ticker(ticker: Any, country: Any, currency: Any = None) -> Optional[str]:
    t = normalise_ticker(ticker)
    c = normalise_country(country)
    cur = normalise_currency(currency)
    if not t:
        return None
    numeric = re.sub(r"\D", "", t)
    if c == "JP" and len(numeric) == 4:
        return "XTKS"
    if (c == "HK" or cur == "HKD") and numeric and 1 <= len(numeric) <= 5:
        return "XHKG"
    if c == "CN" and len(numeric) == 6:
        if numeric.startswith(("5", "6", "9")):
            return "XSHG"
        if numeric.startswith(("0", "1", "2", "3")):
            return "XSHE"
        if numeric.startswith(("4", "8")):
            return "XBSE"
    # A six-digit Korean code does not by itself distinguish KRX from KOSDAQ.
    return None

def infer_listing_country(row: pd.Series) -> tuple[Optional[str], str, float]:
    source_country = normalise_country(row.get("listing_country"))
    if source_country:
        return source_country, "SOURCE_LISTING_COUNTRY", 1.00

    source_mic = normalise_identifier(row.get("source_mic"))
    if source_mic in EXCHANGE_REGISTRY:
        return EXCHANGE_REGISTRY[source_mic]["country"], "SOURCE_MIC_COUNTRY", 1.00

    source_exchange = clean_string(row.get("source_exchange"))
    if source_exchange:
        mic = EXCHANGE_ALIASES.get(source_exchange.upper())
        if mic in EXCHANGE_REGISTRY:
            return EXCHANGE_REGISTRY[mic]["country"], "SOURCE_EXCHANGE_COUNTRY", 0.99

    ticker_mic = infer_mic_from_ticker(row.get("ticker"))
    if ticker_mic in EXCHANGE_REGISTRY:
        return EXCHANGE_REGISTRY[ticker_mic]["country"], "TICKER_SUFFIX_COUNTRY", 0.95

    isin_country = normalise_country(row.get("isin_country") or extract_isin_country(row.get("isin")))
    issuer_country = normalise_country(row.get("issuer_country") or row.get("investment_country"))
    currency = normalise_currency(row.get("currency"))
    cusip = normalise_identifier(row.get("cusip"))

    # USD + CUSIP is strong route-level evidence for a US-listed security, but
    # deliberately does not pretend to identify NYSE versus Nasdaq.
    if currency == "USD" and cusip:
        return "US", "USD_CUSIP_LISTING_COUNTRY", 0.90
    if isin_country and issuer_country and isin_country == issuer_country:
        return isin_country, "ISIN_ISSUER_COUNTRY", 0.88
    if issuer_country:
        return issuer_country, "ISSUER_COUNTRY_PROXY", 0.75
    if isin_country:
        return isin_country, "ISIN_COUNTRY_PROXY", 0.70
    return None, "UNRESOLVED_COUNTRY", 0.0

def resolve_listing_mic(row: pd.Series) -> tuple[Optional[str], str, float]:
    source_mic = normalise_identifier(row.get("source_mic"))
    if source_mic in EXCHANGE_REGISTRY:
        return source_mic, "SOURCE_MIC", 1.00

    source_exchange = clean_string(row.get("source_exchange"))
    if source_exchange:
        alias = EXCHANGE_ALIASES.get(source_exchange.upper())
        if alias:
            return alias, "SOURCE_EXCHANGE", 0.99

    ticker_mic = infer_mic_from_ticker(row.get("ticker"))
    if ticker_mic:
        return ticker_mic, "TICKER_SUFFIX", 0.95

    listing_country, _, _ = infer_listing_country(row)
    inferred = infer_mic_from_numeric_ticker(row.get("ticker"), listing_country, row.get("currency"))
    if inferred:
        return inferred, "COUNTRY_TICKER_STRUCTURE", 0.90

    default_mic = COUNTRY_DEFAULT_MIC.get(listing_country)
    if default_mic:
        return default_mic, "COUNTRY_PRIMARY_MARKET", 0.78

    return None, "MIC_UNRESOLVED_COUNTRY_KNOWN" if listing_country else "UNRESOLVED", 0.0

def route_from_mic(mic: Any, listing_country: Any = None) -> dict:
    m = normalise_identifier(mic)
    if m in EXCHANGE_REGISTRY:
        x = EXCHANGE_REGISTRY[m]
        return {
            "primary_source_engine": x["engine"],
            "primary_source_system": x["system"],
            "secondary_source_system": "COMPANY_IR",
            "processing_priority": 1,
            "route_reason": f"MIC_{m}",
        }

    country = normalise_country(listing_country)
    if country == "US":
        engine, system = "STRUCTURED_XBRL", "SEC"
    elif country in EUROPE_COUNTRIES:
        engine, system = "EUROPE", "ESEF_OR_NATIONAL"
    elif country in {"JP", "KR"}:
        engine, system = "EAST_ASIA", "EDINET" if country == "JP" else "OPENDART"
    elif country in {"CN", "HK"}:
        engine, system = "CHINA_HK", "CNINFO" if country == "CN" else "HKEX"
    else:
        engine, system = "RESIDUAL_GLOBAL", "MARKET_ADAPTER_OR_IR"

    return {
        "primary_source_engine": engine,
        "primary_source_system": system,
        "secondary_source_system": "COMPANY_IR",
        "processing_priority": 2 if country else 3,
        "route_reason": f"COUNTRY_{country or 'UNKNOWN'}",
    }

print(f"Exchange registry loaded: {len(EXCHANGE_REGISTRY)} MICs")


Exchange registry loaded: 25 MICs


In [52]:
# 5. SEC N-PORT HTTP, CATALOGUE AND ARCHIVE HELPERS

NPORT_DATASET_PAGE = "https://www.sec.gov/dera/data/form-n-port-data-sets"

SEC_SESSION = requests.Session()
SEC_SESSION.headers.update({
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
})

def sec_get(url: str, timeout: int = 90, retries: int = 4) -> requests.Response:
    last_error = None
    for attempt in range(retries):
        try:
            r = SEC_SESSION.get(url, timeout=timeout)
            if r.status_code == 200:
                time.sleep(SEC_REQUEST_DELAY_SECONDS)
                return r
            if r.status_code in {403, 429, 500, 502, 503, 504}:
                time.sleep((attempt + 1) * 1.5)
                continue
            r.raise_for_status()
        except Exception as exc:
            last_error = exc
            time.sleep((attempt + 1) * 1.5)
    raise RuntimeError(f"SEC request failed after {retries} attempts: {url}") from last_error

def quarter_start(ts: pd.Timestamp) -> tuple[int, int]:
    q = ((ts.month - 1) // 3) + 1
    return int(ts.year), int(q)

def iter_quarters(start_date: pd.Timestamp, end_date: pd.Timestamp) -> list[tuple[int, int]]:
    out = []
    y, q = quarter_start(start_date)
    end_y, end_q = quarter_start(end_date)
    while (y, q) <= (end_y, end_q):
        out.append((y, q))
        q += 1
        if q == 5:
            y, q = y + 1, 1
    return out

def nport_bulk_url(year: int, quarter: int) -> str:
    # SEC bulk archive naming convention.
    return f"https://www.sec.gov/files/dera/data/form-n-port-data-sets/{year}q{quarter}_nport.zip"

def build_nport_catalogue(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    rows = [
        {"year": y, "quarter": q, "url": nport_bulk_url(y, q)}
        for y, q in iter_quarters(start_date, end_date)
    ]
    return pd.DataFrame(rows)

def download_quarter(year: int, quarter: int, force: bool = False) -> Path:
    target = CACHE_DIR / f"{year}q{quarter}_nport.zip"
    if target.exists() and USE_CACHE and not force:
        return target
    url = nport_bulk_url(year, quarter)
    response = sec_get(url, timeout=180)
    target.write_bytes(response.content)
    return target

def archive_members(path: Path) -> list[str]:
    with zipfile.ZipFile(path) as zf:
        return zf.namelist()

def find_archive_member(path: Path, candidates: Iterable[str]) -> Optional[str]:
    names = archive_members(path)
    lowered = {n.lower(): n for n in names}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    for n in names:
        base = Path(n).name.lower()
        for candidate in candidates:
            if base == candidate.lower():
                return n
    return None

def read_archive_table(path: Path, candidates: Iterable[str]) -> pd.DataFrame:
    member = find_archive_member(path, candidates)
    if not member:
        return pd.DataFrame()
    with zipfile.ZipFile(path) as zf:
        raw = zf.read(member)
    # SEC bulk tables are tab-delimited.
    return pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str, low_memory=False)

nport_dataset_catalog_df = build_nport_catalogue(START_DATE, END_DATE)
display(nport_dataset_catalog_df.head())
print("Quarters in configured range:", len(nport_dataset_catalog_df))


,year,quarter,url
0,2019,4,https://www.sec.gov/files/dera/data/form-n-por...
1,2020,1,https://www.sec.gov/files/dera/data/form-n-por...
2,2020,2,https://www.sec.gov/files/dera/data/form-n-por...
3,2020,3,https://www.sec.gov/files/dera/data/form-n-por...
4,2020,4,https://www.sec.gov/files/dera/data/form-n-por...


Quarters in configured range: 28


In [53]:
# 6. ETF REGISTRY AND TARGET-FILING DISCOVERY

# Fund aliases are configuration data, not platform logic.
# Every configured ticker is always included as an exact ticker candidate.
# Optional legal/series-name aliases may be supplied under:
# RESEARCH_CONFIG["universe"]["fund_aliases"].

def configured_fund_aliases() -> dict[str, list[str]]:
    raw = RESEARCH_CONFIG.get("universe", {}).get("fund_aliases", {}) or {}
    out = {}
    for ticker, aliases in raw.items():
        ticker_norm = str(ticker).upper().strip()
        vals = aliases if isinstance(aliases, list) else [aliases]
        out[ticker_norm] = [
            str(v).strip()
            for v in vals
            if v is not None and str(v).strip()
        ]
    return out

def compact_text(value: Any) -> str:
    s = clean_string(value) or ""
    return re.sub(r"[^A-Z0-9]", "", s.upper())

def build_etf_registry(target_funds: list[str]) -> pd.DataFrame:
    alias_config = configured_fund_aliases()
    rows = []

    for ticker in target_funds:
        ticker = str(ticker).upper().strip()
        aliases = [ticker] + alias_config.get(ticker, [])

        for alias in dict.fromkeys(aliases):
            rows.append({
                "fund_id": "FUND_" + stable_hash("FUND", ticker, length=16),
                "fund_ticker": ticker,
                "alias": alias,
                "alias_normalised": compact_text(alias),
            })

    return pd.DataFrame(rows).drop_duplicates()

etf_registry_df = build_etf_registry(TARGET_FUNDS)

def identify_target_filings(
    submissions: pd.DataFrame,
    registrants: pd.DataFrame,
    fund_reports: pd.DataFrame,
    registry: pd.DataFrame,
    year: int,
    quarter: int,
) -> pd.DataFrame:
    if submissions.empty:
        return pd.DataFrame()

    s = submissions.copy()
    s.columns = [str(c).upper() for c in s.columns]
    r = registrants.copy()
    r.columns = [str(c).upper() for c in r.columns]
    f = fund_reports.copy()
    f.columns = [str(c).upper() for c in f.columns]

    accession_col = "ACCESSION_NUMBER"
    if accession_col not in s.columns:
        return pd.DataFrame()

    # Build a broad searchable series/registrant name from available tables.
    merged = s.copy()
    if not r.empty and accession_col in r.columns:
        keep = [c for c in [accession_col, "REGISTRANT_NAME", "CIK"] if c in r.columns]
        merged = merged.merge(r[keep].drop_duplicates(), on=accession_col, how="left")
    if not f.empty and accession_col in f.columns:
        keep = [c for c in [accession_col, "SERIES_NAME", "SERIES_ID", "REPORT_DATE",
                             "REPORT_ENDING_PERIOD"] if c in f.columns]
        merged = merged.merge(f[keep].drop_duplicates(), on=accession_col, how="left")

    text_cols = [c for c in ["SERIES_NAME", "REGISTRANT_NAME"] if c in merged.columns]
    merged["_search"] = ""
    for c in text_cols:
        merged["_search"] = merged["_search"] + " " + merged[c].fillna("").astype(str)
    merged["_search_norm"] = merged["_search"].map(compact_text)

    matches = []
    for row in registry.itertuples(index=False):
        mask = merged["_search_norm"].str.contains(row.alias_normalised, na=False, regex=False)
        if row.alias_normalised == compact_text(row.fund_ticker):
            # Avoid matching a short ticker inside arbitrary legal names.
            mask = pd.Series(False, index=merged.index)
            ticker_cols = [c for c in merged.columns if "TICKER" in c]
            for c in ticker_cols:
                mask |= merged[c].fillna("").astype(str).str.upper().eq(row.fund_ticker)
        hit = merged.loc[mask].copy()
        if not hit.empty:
            hit["fund_id"] = row.fund_id
            hit["fund_ticker"] = row.fund_ticker
            hit["matched_alias"] = row.alias
            matches.append(hit)

    if not matches:
        return pd.DataFrame()

    out = pd.concat(matches, ignore_index=True)
    out["dataset_year"] = year
    out["dataset_quarter"] = quarter
    return out.drop_duplicates(subset=["ACCESSION_NUMBER", "fund_ticker"], keep="first")

display(etf_registry_df)


,fund_id,fund_ticker,alias,alias_normalised
0,FUND_9157AA533FFA0FAB,DRIV,DRIV,DRIV
1,FUND_9157AA533FFA0FAB,DRIV,Global X Autonomous & Electric Vehicles ETF,GLOBALXAUTONOMOUSELECTRICVEHICLESETF
2,FUND_9157AA533FFA0FAB,DRIV,Global X Autonomous and Electric Vehicles ETF,GLOBALXAUTONOMOUSANDELECTRICVEHICLESETF
3,FUND_63E9D13B0BA3FFE8,CARZ,CARZ,CARZ
4,FUND_63E9D13B0BA3FFE8,CARZ,First Trust S-Network Future Vehicles & Techno...,FIRSTTRUSTSNETWORKFUTUREVEHICLESTECHNOLOGYETF
5,FUND_63E9D13B0BA3FFE8,CARZ,First Trust NASDAQ Global Auto Index Fund,FIRSTTRUSTNASDAQGLOBALAUTOINDEXFUND
6,FUND_CD45AAC0B678329D,IDRV,IDRV,IDRV
7,FUND_CD45AAC0B678329D,IDRV,iShares Self-Driving EV and Tech ETF,ISHARESSELFDRIVINGEVANDTECHETF
8,FUND_CD45AAC0B678329D,IDRV,iShares Self-Driving EV & Tech ETF,ISHARESSELFDRIVINGEVTECHETF
9,FUND_1A02192D2C6BD726,KARS,KARS,KARS


In [54]:
# 7. QUARTER PROCESSING AND RAW HOLDINGS ACQUISITION

def first_existing_column(df: pd.DataFrame, names: Iterable[str]) -> Optional[str]:
    lookup = {str(c).upper(): c for c in df.columns}
    for name in names:
        if name.upper() in lookup:
            return lookup[name.upper()]
    return None


def process_quarter(
    year: int,
    quarter: int,
    registry: pd.DataFrame,
) -> dict[str, pd.DataFrame]:

    archive_path = download_quarter(year, quarter)

    submissions = read_archive_table(
        archive_path,
        ["SUBMISSION.tsv", "SUBMISSION.txt", "submission.tsv", "submission.txt"],
    )

    registrants = read_archive_table(
        archive_path,
        ["REGISTRANT.tsv", "REGISTRANT.txt", "registrant.tsv", "registrant.txt"],
    )

    fund_reports = read_archive_table(
        archive_path,
        [
            "FUND_REPORTED_INFO.tsv",
            "FUND_REPORTED_INFO.txt",
            "fund_reported_info.tsv",
            "fund_reported_info.txt",
        ],
    )

    holdings = read_archive_table(
        archive_path,
        [
            "FUND_REPORTED_HOLDING.tsv",
            "FUND_REPORTED_HOLDING.txt",
            "fund_reported_holding.tsv",
            "fund_reported_holding.txt",
        ],
    )

    identifiers = read_archive_table(
        archive_path,
        [
            "IDENTIFIERS.tsv",
            "IDENTIFIERS.txt",
            "identifiers.tsv",
            "identifiers.txt",
        ],
    )

    # --------------------------------------------------------
    # Identify target ETF filings
    # --------------------------------------------------------

    filings = identify_target_filings(
        submissions,
        registrants,
        fund_reports,
        registry,
        year,
        quarter,
    )

    if filings.empty or holdings.empty:
        return {
            "filings": filings,
            "holdings": pd.DataFrame(),
        }

    # --------------------------------------------------------
    # Standardise holding-table column names
    # --------------------------------------------------------

    h = holdings.copy()
    h.columns = [str(c).upper() for c in h.columns]

    if "ACCESSION_NUMBER" not in h.columns:
        return {
            "filings": filings,
            "holdings": pd.DataFrame(),
        }

    # --------------------------------------------------------
    # Attach ETF identity AND filing/report metadata
    #
    # These dates are essential for PIT snapshot construction.
    # --------------------------------------------------------

    filing_metadata_columns = [
        "ACCESSION_NUMBER",
        "fund_id",
        "fund_ticker",
        "matched_alias",
        "FILING_DATE",
        "SUB_TYPE",
        "REPORT_DATE",
        "REPORT_ENDING_PERIOD",
        "SERIES_NAME",
        "SERIES_ID",
    ]

    filing_metadata_columns = [
        c for c in filing_metadata_columns
        if c in filings.columns
    ]

    wanted = (
        filings[filing_metadata_columns]
        .drop_duplicates(
            subset=["ACCESSION_NUMBER", "fund_ticker"],
            keep="first",
        )
    )

    h = h.merge(
        wanted,
        on="ACCESSION_NUMBER",
        how="inner",
    )

    # --------------------------------------------------------
    # Attach security identifiers
    #
    # SEC N-PORT bulk archives normally expose these as wide
    # fields:
    #
    #   IDENTIFIER_ISIN
    #   IDENTIFIER_TICKER
    #   OTHER_IDENTIFIER
    #   OTHER_IDENTIFIER_DESC
    #
    # A type/value fallback is retained for compatibility with
    # any alternative archive layout.
    # --------------------------------------------------------

    if not identifiers.empty:

        ids = identifiers.copy()
        ids.columns = [str(c).upper() for c in ids.columns]

        join_cols = [
            c
            for c in ["ACCESSION_NUMBER", "HOLDING_ID"]
            if c in ids.columns and c in h.columns
        ]

        if join_cols:

            # -----------------------------------------------
            # Preferred: wide-format SEC identifier table
            # -----------------------------------------------

            wide_identifier_candidates = {
                "IDENTIFIER_ISIN": [
                    "IDENTIFIER_ISIN",
                    "ISIN",
                ],
                "IDENTIFIER_TICKER": [
                    "IDENTIFIER_TICKER",
                    "TICKER",
                ],
                "OTHER_IDENTIFIER": [
                    "OTHER_IDENTIFIER",
                ],
                "OTHER_IDENTIFIER_DESC": [
                    "OTHER_IDENTIFIER_DESC",
                    "OTHER_IDENTIFIER_DESCRIPTION",
                ],
            }

            selected = ids[join_cols].copy()
            found_wide_identifier = False

            for target, candidates in wide_identifier_candidates.items():

                source = first_existing_column(ids, candidates)

                if source is not None:
                    selected[target] = ids[source]
                    found_wide_identifier = True

            if found_wide_identifier:

                selected = selected.drop_duplicates(
                    subset=join_cols,
                    keep="first",
                )

                h = h.merge(
                    selected,
                    on=join_cols,
                    how="left",
                    suffixes=("", "_IDENTIFIER"),
                )

            # -----------------------------------------------
            # Fallback: identifier-type / identifier-value
            # layout
            # -----------------------------------------------

            else:

                value_col = first_existing_column(
                    ids,
                    [
                        "IDENTIFIER_VALUE",
                        "IDENTIFIER",
                        "VALUE",
                    ],
                )

                type_col = first_existing_column(
                    ids,
                    [
                        "IDENTIFIER_TYPE",
                        "TYPE",
                    ],
                )

                if value_col is not None and type_col is not None:

                    ids["_type"] = (
                        ids[type_col]
                        .fillna("")
                        .astype(str)
                        .str.upper()
                        .str.strip()
                    )

                    pivot = (
                        ids.pivot_table(
                            index=join_cols,
                            columns="_type",
                            values=value_col,
                            aggfunc="first",
                        )
                        .reset_index()
                    )

                    pivot.columns = [
                        str(c).upper()
                        for c in pivot.columns
                    ]

                    rename_map = {}

                    for col in pivot.columns:

                        uc = str(col).upper()

                        if uc == "ISIN":
                            rename_map[col] = "IDENTIFIER_ISIN"

                        elif uc == "TICKER":
                            rename_map[col] = "IDENTIFIER_TICKER"

                    pivot = pivot.rename(columns=rename_map)

                    h = h.merge(
                        pivot,
                        on=join_cols,
                        how="left",
                    )

    # --------------------------------------------------------
    # Dataset lineage
    # --------------------------------------------------------

    h["dataset_year"] = year
    h["dataset_quarter"] = quarter

    return {
        "filings": filings,
        "holdings": h,
    }


# ============================================================
# PROCESS ALL N-PORT QUARTERS WITH LIVE PROGRESS
# ============================================================

import time
from datetime import datetime, timedelta

download_log = []
filing_parts = []
holding_parts = []

records = list(nport_dataset_catalog_df.itertuples(index=False))
total_quarters = len(records)

cell7_start = time.time()

print("=" * 72)
print("CELL 7 — N-PORT ACQUISITION STARTED")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
print(f"Quarters to process: {total_quarters}")
print("=" * 72)

for i, rec in enumerate(records, start=1):

    quarter_start = time.time()

    try:

        result = process_quarter(
            int(rec.year),
            int(rec.quarter),
            etf_registry_df,
        )

        if not result["filings"].empty:
            filing_parts.append(result["filings"])

        if not result["holdings"].empty:
            holding_parts.append(result["holdings"])

        download_log.append({
            "dataset_year": rec.year,
            "dataset_quarter": rec.quarter,
            "url": rec.url,
            "status": "OK",
            "target_filing_rows": len(result["filings"]),
            "target_holding_rows": len(result["holdings"]),
            "error": None,
        })

        status = "✓"

    except Exception as exc:

        download_log.append({
            "dataset_year": rec.year,
            "dataset_quarter": rec.quarter,
            "url": rec.url,
            "status": "ERROR",
            "target_filing_rows": 0,
            "target_holding_rows": 0,
            "error": repr(exc),
        })

        status = "✗"

    # --------------------------------------------------------
    # LIVE PROGRESS REPORT
    # --------------------------------------------------------

    elapsed = time.time() - cell7_start
    average_per_quarter = elapsed / i

    remaining_quarters = total_quarters - i
    eta_seconds = average_per_quarter * remaining_quarters

    quarter_elapsed = time.time() - quarter_start

    progress_pct = 100 * i / total_quarters

    cumulative_filings = sum(
        len(x) for x in filing_parts
    )

    cumulative_holdings = sum(
        len(x) for x in holding_parts
    )

    elapsed_td = timedelta(seconds=int(elapsed))
    eta_td = timedelta(seconds=int(eta_seconds))

    print(
        f"[{datetime.now().strftime('%H:%M:%S')}] "
        f"{status} {rec.year} Q{rec.quarter} | "
        f"{i}/{total_quarters} "
        f"({progress_pct:5.1f}%)"
    )

    print(
        f"    Quarter runtime: {quarter_elapsed:6.1f}s | "
        f"Elapsed: {elapsed_td} | "
        f"ETA: ~{eta_td}"
    )

    print(
        f"    Cumulative filings: {cumulative_filings:,} | "
        f"Cumulative holdings: {cumulative_holdings:,}"
    )

    print("-" * 72)


nport_download_log_df = pd.DataFrame(download_log)

etf_filing_history_df = (
    pd.concat(
        filing_parts,
        ignore_index=True,
        sort=False,
    )
    if filing_parts
    else pd.DataFrame()
)

etf_holdings_raw_df = (
    pd.concat(
        holding_parts,
        ignore_index=True,
        sort=False,
    )
    if holding_parts
    else pd.DataFrame()
)


# ============================================================
# CELL 7 COMPLETION SUMMARY
# ============================================================

cell7_elapsed = time.time() - cell7_start

successful_quarters = int(
    (nport_download_log_df["status"] == "OK").sum()
)

failed_quarters = int(
    (nport_download_log_df["status"] == "ERROR").sum()
)

print()
print("=" * 72)
print("CELL 7 — N-PORT ACQUISITION COMPLETE")
print(f"Total runtime: {timedelta(seconds=int(cell7_elapsed))}")
print(f"Successful quarters: {successful_quarters}/{total_quarters}")
print(f"Failed quarters: {failed_quarters}")
print(f"Filings retrieved: {len(etf_filing_history_df):,}")
print(f"Raw holdings retrieved: {len(etf_holdings_raw_df):,}")
print("=" * 72)


# ============================================================
# ACQUISITION VALIDATION
# ============================================================

print("Filings:", len(etf_filing_history_df))
print("Raw target holdings:", len(etf_holdings_raw_df))

required_raw_fields = [
    "FILING_DATE",
    "REPORT_DATE",
    "REPORT_ENDING_PERIOD",
    "IDENTIFIER_ISIN",
    "IDENTIFIER_TICKER",
]

print("\nCritical raw-field coverage:")

for col in required_raw_fields:

    if col in etf_holdings_raw_df.columns:

        populated = int(
            etf_holdings_raw_df[col]
            .notna()
            .sum()
        )

        rate = (
            populated / len(etf_holdings_raw_df)
            if len(etf_holdings_raw_df)
            else 0
        )

        print(
            f"  {col:<24}"
            f"{populated:>7,} / "
            f"{len(etf_holdings_raw_df):>7,} "
            f"({rate:>6.2%})"
        )

    else:
        print(
            f"  {col:<24}MISSING COLUMN"
        )


# Dates must exist before PIT construction is allowed to proceed.

if len(etf_holdings_raw_df):

    if "FILING_DATE" not in etf_holdings_raw_df.columns:
        raise AssertionError(
            "FILING_DATE was not attached to N-PORT holdings."
        )

    if "REPORT_DATE" not in etf_holdings_raw_df.columns:
        raise AssertionError(
            "REPORT_DATE was not attached to N-PORT holdings."
        )

    filing_dates = pd.to_datetime(
        etf_holdings_raw_df["FILING_DATE"],
        errors="coerce",
    )

    report_dates = pd.to_datetime(
        etf_holdings_raw_df["REPORT_DATE"],
        errors="coerce",
    )

    if filing_dates.notna().sum() == 0:
        raise AssertionError(
            "All N-PORT FILING_DATE values are null."
        )

    if report_dates.notna().sum() == 0:
        raise AssertionError(
            "All N-PORT REPORT_DATE values are null."
        )


display(
    nport_download_log_df.tail()
)

CELL 7 — N-PORT ACQUISITION STARTED
Start time: 00:58:18
Quarters to process: 28
[00:59:35] ✓ 2019 Q4 | 1/28 (  3.6%)
    Quarter runtime:   77.7s | Elapsed: 0:01:17 | ETA: ~0:34:56
    Cumulative filings: 3 | Cumulative holdings: 188
------------------------------------------------------------------------
[01:01:10] ✓ 2020 Q1 | 2/28 (  7.1%)
    Quarter runtime:   95.1s | Elapsed: 0:02:52 | ETA: ~0:37:26
    Cumulative filings: 7 | Cumulative holdings: 462
------------------------------------------------------------------------
[01:02:33] ✓ 2020 Q2 | 3/28 ( 10.7%)
    Quarter runtime:   82.6s | Elapsed: 0:04:15 | ETA: ~0:35:28
    Cumulative filings: 11 | Cumulative holdings: 733
------------------------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# 8. STANDARDISE N-PORT HOLDINGS, FILTER EQUITIES AND BUILD PIT SNAPSHOTS

def series_from(df: pd.DataFrame, candidates: Iterable[str], default=pd.NA) -> pd.Series:
    col = first_existing_column(df, candidates)
    if col is None:
        return pd.Series(default, index=df.index)
    return df[col]

def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False), errors="coerce")

def standardise_holdings(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()

    df = raw.copy()
    df.columns = [str(c).upper() for c in df.columns]

    out = pd.DataFrame(index=df.index)
    mappings = {
        "accession_number": ["ACCESSION_NUMBER"],
        "holding_id": ["HOLDING_ID"],
        "issuer_name": ["ISSUER_NAME"],
        "issuer_lei": ["ISSUER_LEI", "LEI"],
        "security_name": ["ISSUER_TITLE", "SECURITY_TITLE"],
        "cusip": ["ISSUER_CUSIP", "CUSIP"],
        "isin": ["IDENTIFIER_ISIN", "ISIN"],
        "ticker": ["IDENTIFIER_TICKER", "TICKER"],
        "other_identifier": ["OTHER_IDENTIFIER"],
        "other_identifier_description": ["OTHER_IDENTIFIER_DESC"],
        "currency": ["CURRENCY_CODE", "CURRENCY"],
        "asset_category": ["ASSET_CAT", "ASSET_CATEGORY"],
        "issuer_type": ["ISSUER_TYPE"],
        "investment_country": ["INVESTMENT_COUNTRY", "COUNTRY"],
        "unit": ["UNIT"],
        "balance": ["BALANCE"],
        "market_value": ["CURRENCY_VALUE", "MARKET_VALUE"],
        "reported_weight": ["PERCENTAGE", "WEIGHT"],
        "fund_id": ["FUND_ID"],
        "fund_ticker": ["FUND_TICKER"],
        "matched_alias": ["MATCHED_ALIAS"],
        "dataset_year": ["DATASET_YEAR"],
        "dataset_quarter": ["DATASET_QUARTER"],
    }

    for dst, srcs in mappings.items():
        out[dst] = series_from(df, srcs)

    # Filing/snapshot fields are best recovered from merged source columns if present.
    out["filing_date"] = pd.to_datetime(
        series_from(df, ["FILING_DATE"]), errors="coerce"
    )
    out["snapshot_date"] = pd.to_datetime(
        series_from(df, ["REPORT_DATE", "REPORT_ENDING_PERIOD"]), errors="coerce"
    )
    out["fiscal_period_end"] = pd.to_datetime(
        series_from(df, ["REPORT_ENDING_PERIOD", "REPORT_DATE"]), errors="coerce"
    )

    out["balance"] = safe_numeric(out["balance"])
    out["market_value"] = safe_numeric(out["market_value"])
    out["reported_weight"] = safe_numeric(out["reported_weight"])

    # N-PORT filing date is the earliest defensible public-availability date here.
    out["available_datetime"] = out["filing_date"]
    out["effective_date"] = out["available_datetime"].dt.normalize()
    out["information_lag_days"] = (
        out["available_datetime"].dt.normalize() - out["snapshot_date"].dt.normalize()
    ).dt.days

    # Normalised identity evidence.
    out["issuer_name_normalised"] = out["issuer_name"].map(normalise_name)
    out["security_name_normalised"] = out["security_name"].map(normalise_name)
    out["ticker_normalised"] = out["ticker"].map(normalise_ticker)
    out["isin_normalised"] = out["isin"].map(normalise_identifier)
    out["cusip_normalised"] = out["cusip"].map(normalise_identifier)
    out["lei_normalised"] = out["issuer_lei"].map(normalise_identifier)
    out["issuer_country"] = out["investment_country"].map(normalise_country)
    out["currency"] = out["currency"].map(normalise_currency)
    out["isin_country"] = out["isin"].map(extract_isin_country)

    out["source_system"] = "SEC_NPORT"
    out["source_filing_id"] = out["accession_number"]
    out["source_observation_id"] = [
        "NPORT_" + stable_hash(a, h, f, length=24)
        for a, h, f in zip(out["accession_number"], out["holding_id"], out["fund_ticker"])
    ]
    return out

def filter_equity_holdings(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    category = df["asset_category"].fillna("").astype(str).str.upper()
    issuer_type = df["issuer_type"].fillna("").astype(str).str.upper()
    security_name = df["security_name"].fillna("").astype(str).str.upper()

    # Broad but conservative equity screen; source evidence is preserved for audit.
    equity_mask = (
        category.str.contains("EC|EQUITY|COMMON|PREFERRED", regex=True)
        | issuer_type.str.contains("CORP", regex=False)
    )
    obvious_non_equity = security_name.str.contains(
        r"\b(BOND|NOTE|TREASURY|SWAP|FUTURE|OPTION|WARRANT)\b", regex=True
    )
    return df.loc[equity_mask & ~obvious_non_equity].copy()

def resolve_snapshot_versions(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.sort_values(
        ["fund_ticker", "snapshot_date", "available_datetime", "accession_number"]
    ).copy()
    # Retain the latest publicly filed version for an identical fund/snapshot/holding.
    key = ["fund_ticker", "snapshot_date", "holding_id"]
    x["_version_rank"] = x.groupby(key)["available_datetime"].rank(method="first")
    max_rank = x.groupby(key)["_version_rank"].transform("max")
    x["is_latest_snapshot_version"] = x["_version_rank"].eq(max_rank)
    return x

def build_snapshot_index(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    idx = (
        df.loc[df["is_latest_snapshot_version"]]
        .groupby(["fund_id", "fund_ticker", "snapshot_date", "accession_number"], dropna=False)
        .agg(
            filing_date=("filing_date", "min"),
            available_datetime=("available_datetime", "min"),
            constituent_rows=("source_observation_id", "nunique"),
        )
        .reset_index()
        .sort_values(["fund_ticker", "available_datetime", "snapshot_date"])
    )
    idx["next_available_datetime"] = idx.groupby("fund_ticker")["available_datetime"].shift(-1)
    idx["is_latest_public_snapshot"] = idx["next_available_datetime"].isna()
    return idx

def build_constituent_intervals(df: pd.DataFrame, snapshot_index: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    latest = df.loc[df["is_latest_snapshot_version"]].copy()
    interval_cols = [
        "fund_id", "fund_ticker", "snapshot_date", "accession_number",
        "available_datetime", "next_available_datetime", "is_latest_public_snapshot"
    ]
    idx = snapshot_index[interval_cols].drop_duplicates()
    latest = latest.drop(columns=["available_datetime"], errors="ignore").merge(
        idx,
        on=["fund_id", "fund_ticker", "snapshot_date", "accession_number"],
        how="left",
    )
    latest["membership_start_date"] = latest["available_datetime"]
    latest["membership_end_date"] = latest["next_available_datetime"]
    return latest

etf_holdings_standardised_df = standardise_holdings(etf_holdings_raw_df)
etf_equity_holdings_df = filter_equity_holdings(etf_holdings_standardised_df)
etf_equity_holdings_df = resolve_snapshot_versions(etf_equity_holdings_df)
etf_snapshot_index_df = build_snapshot_index(etf_equity_holdings_df)
etf_constituent_intervals_df = build_constituent_intervals(
    etf_equity_holdings_df, etf_snapshot_index_df
)

print("Standardised holdings:", len(etf_holdings_standardised_df))
print("Equity observations:", len(etf_equity_holdings_df))
print("PIT interval observations:", len(etf_constituent_intervals_df))
display(etf_snapshot_index_df.tail())


In [55]:
# 9. ADD OPTIONAL MANUAL SECURITIES THROUGH THE SAME IDENTITY PIPELINE

MANUAL_SECURITY_COLUMNS = [
    "ticker", "security_name", "issuer_name", "isin", "cusip", "lei",
    "listing_country", "exchange_mic", "currency", "valid_from", "valid_to"
]

def build_manual_security_observations(items: list[dict]) -> pd.DataFrame:
    if not items:
        return pd.DataFrame()

    rows = []
    for i, item in enumerate(items):
        row = {c: item.get(c) for c in MANUAL_SECURITY_COLUMNS}
        row["fund_id"] = pd.NA
        row["fund_ticker"] = pd.NA
        row["snapshot_date"] = pd.to_datetime(item.get("valid_from") or START_DATE)
        row["available_datetime"] = pd.to_datetime(item.get("valid_from") or START_DATE)
        row["membership_start_date"] = pd.to_datetime(item.get("valid_from") or START_DATE)
        row["membership_end_date"] = pd.to_datetime(item.get("valid_to"), errors="coerce")
        row["issuer_country"] = normalise_country(item.get("listing_country"))
        row["investment_country"] = row["issuer_country"]
        row["issuer_lei"] = item.get("lei")
        row["lei_normalised"] = normalise_identifier(item.get("lei"))
        row["isin_normalised"] = normalise_identifier(item.get("isin"))
        row["cusip_normalised"] = normalise_identifier(item.get("cusip"))
        row["ticker_normalised"] = normalise_ticker(item.get("ticker"))
        row["issuer_name_normalised"] = normalise_name(item.get("issuer_name"))
        row["security_name_normalised"] = normalise_name(item.get("security_name"))
        row["source_mic"] = item.get("exchange_mic")
        row["source_exchange"] = pd.NA
        row["source_system"] = "MANUAL_CONFIG"
        row["source_filing_id"] = pd.NA
        row["source_observation_id"] = "MANUAL_" + stable_hash(
            item.get("ticker"), item.get("isin"), i, length=24
        )
        row["balance"] = pd.NA
        row["market_value"] = pd.NA
        row["reported_weight"] = pd.NA
        rows.append(row)
    return pd.DataFrame(rows)

manual_security_observations_df = build_manual_security_observations(
    RESEARCH_CONFIG["universe"].get("individual_securities", [])
)

identity_evidence_df = pd.concat(
    [etf_constituent_intervals_df, manual_security_observations_df],
    ignore_index=True,
    sort=False,
)

for col in [
    "source_mic", "source_exchange", "listing_country", "security_name",
    "issuer_name", "issuer_country", "currency", "isin_normalised",
    "cusip_normalised", "lei_normalised", "ticker_normalised"
]:
    if col not in identity_evidence_df.columns:
        identity_evidence_df[col] = pd.NA

print("Identity evidence rows:", len(identity_evidence_df))
print("Manual observations:", len(manual_security_observations_df))


Identity evidence rows: 8144
Manual observations: 0


In [56]:
# 10. FOUR-LAYER IDENTITY RESOLUTION WITH EXTERNAL REFERENCE EVIDENCE
# Economic Issuer → Legal Entity → Security → Listing
#
# The resolver is industry-agnostic. Authoritative external reference data is
# used as evidence, never as a hard-coded company exception. GLEIF is queried
# only for LEIs actually observed in the selected universe and responses are
# cached on Drive. OpenAI is not used to assign authoritative identity.

def first_present(*values: Any) -> Any:
    for value in values:
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except Exception:
            pass
        if str(value).strip():
            return value
    return None

INVALID_IDENTIFIER_TOKENS = {
    "", "0", "00", "000", "0000", "00000", "000000", "0000000",
    "00000000", "000000000", "0000000000", "NA", "N/A", "NONE",
    "NULL", "NAN", "UNKNOWN", "NOTAVAILABLE",
}

def usable_identifier(value: Any, identifier_type: str) -> Optional[str]:
    value = normalise_identifier(value)
    if not value:
        return None
    token = str(value).strip().upper().replace(" ", "")
    if token in INVALID_IDENTIFIER_TOKENS or (token and set(token) == {"0"}):
        return None
    ident_type = identifier_type.upper()
    if ident_type == "ISIN" and (len(token) != 12 or not token[:2].isalpha() or not token.isalnum()):
        return None
    if ident_type == "CUSIP" and (len(token) != 9 or not all(ch.isalnum() or ch in "*@#" for ch in token)):
        return None
    if ident_type == "LEI" and (len(token) != 20 or not token.isalnum()):
        return None
    return token

def usable_isin(row: pd.Series) -> Optional[str]:
    return usable_identifier(first_present(row.get("isin_normalised"), row.get("isin")), "ISIN")

def usable_cusip(row: pd.Series) -> Optional[str]:
    return usable_identifier(first_present(row.get("cusip_normalised"), row.get("cusip")), "CUSIP")

def usable_lei(row: pd.Series) -> Optional[str]:
    return usable_identifier(first_present(row.get("lei_normalised"), row.get("issuer_lei")), "LEI")

class UnionFind:
    def __init__(self):
        self.parent, self.rank = {}, {}
    def add(self, x):
        if x not in self.parent:
            self.parent[x], self.rank[x] = x, 0
    def find(self, x):
        self.add(x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

def build_security_identifier_components(x: pd.DataFrame) -> tuple[dict, dict, set[str]]:
    uf, row_tokens, cusip_to_isins = UnionFind(), {}, {}
    for idx, row in x.iterrows():
        isin, cusip = usable_isin(row), usable_cusip(row)
        if cusip and isin:
            cusip_to_isins.setdefault(cusip, set()).add(isin)
    ambiguous_cusips = {c for c, vals in cusip_to_isins.items() if len(vals) > 1}
    for idx, row in x.iterrows():
        isin, cusip = usable_isin(row), usable_cusip(row)
        tokens = []
        if isin: tokens.append(("ISIN", isin)); uf.add(("ISIN", isin))
        if cusip: tokens.append(("CUSIP", cusip))
        if cusip and cusip not in ambiguous_cusips: uf.add(("CUSIP", cusip))
        if isin and cusip and cusip not in ambiguous_cusips: uf.union(("ISIN", isin), ("CUSIP", cusip))
        row_tokens[idx] = tokens
    components = {}
    for token in list(uf.parent):
        components.setdefault(uf.find(token), set()).add(token)
    token_to_anchor = {}
    for members in components.values():
        isins = sorted(v for t, v in members if t == "ISIN")
        cusips = sorted(v for t, v in members if t == "CUSIP")
        anchor = ("ISIN", isins[0]) if isins else ("CUSIP", cusips[0])
        for token in members: token_to_anchor[token] = anchor
    return row_tokens, token_to_anchor, ambiguous_cusips

# ----------------------------- GLEIF adapter -----------------------------
GLEIF_API_BASE = "https://api.gleif.org/api/v1"
IDENTITY_REF_CFG = RESEARCH_CONFIG.get("identity_reference_data", {})
GLEIF_ENABLED = bool(IDENTITY_REF_CFG.get("enabled", True) and IDENTITY_REF_CFG.get("gleif_enabled", True))
GLEIF_NETWORK_ENABLED = bool(IDENTITY_REF_CFG.get("network_enabled", True))
GLEIF_CACHE_DAYS = int(IDENTITY_REF_CFG.get("cache_days", 30))
GLEIF_TIMEOUT = float(IDENTITY_REF_CFG.get("timeout_seconds", 20))
GLEIF_DELAY = float(IDENTITY_REF_CFG.get("request_delay_seconds", 0.08))

def _cache_fresh(path: Path, days: int) -> bool:
    if not path.exists(): return False
    age = time.time() - path.stat().st_mtime
    return age <= days * 86400

def gleif_get_lei_record(lei: str) -> dict:
    """Retrieve one GLEIF LEI record. Public API; no API key is required."""
    lei = usable_identifier(lei, "LEI")
    if not lei:
        return {"lei": None, "status": "INVALID_LEI"}

    cache_path = GLEIF_CACHE_DIR / f"lei_{lei}.json"
    payload = None
    source = "CACHE"

    if _cache_fresh(cache_path, GLEIF_CACHE_DAYS):
        try:
            payload = json.loads(cache_path.read_text(encoding="utf-8"))
        except Exception:
            payload = None

    if payload is None and GLEIF_ENABLED and GLEIF_NETWORK_ENABLED:
        source = "GLEIF_API"
        try:
            r = requests.get(
                f"{GLEIF_API_BASE}/lei-records/{lei}",
                timeout=GLEIF_TIMEOUT,
                headers={"Accept": "application/vnd.api+json"},
            )
            if r.status_code == 200:
                payload = r.json()
                cache_path.write_text(
                    json.dumps(payload, ensure_ascii=False),
                    encoding="utf-8",
                )
            elif r.status_code == 404:
                return {"lei": lei, "status": "NOT_FOUND", "source": source}
            else:
                return {
                    "lei": lei,
                    "status": f"HTTP_{r.status_code}",
                    "source": source,
                }
        except Exception as exc:
            return {
                "lei": lei,
                "status": "UNAVAILABLE",
                "source": source,
                "error": str(exc)[:300],
            }
        finally:
            time.sleep(GLEIF_DELAY)

    if payload is None:
        return {
            "lei": lei,
            "status": "UNAVAILABLE",
            "source": "DISABLED_OR_NO_CACHE",
        }

    try:
        data = payload.get("data", {}) or {}
        attrs = data.get("attributes", {}) or {}
        entity = attrs.get("entity", {}) or {}
        registration = attrs.get("registration", {}) or {}

        legal_name = ((entity.get("legalName") or {}).get("name"))
        legal_address = entity.get("legalAddress") or {}
        hq_address = entity.get("headquartersAddress") or {}

        # GLEIF can expose alternative and transliterated names. These are
        # critical when the source filing uses English but GLEIF uses the local script.
        alternative_names = []
        for field in ("otherNames", "transliteratedOtherNames"):
            vals = entity.get(field) or []
            if isinstance(vals, dict):
                vals = [vals]
            for item in vals:
                if isinstance(item, dict):
                    name = item.get("name")
                else:
                    name = item
                if name and str(name).strip():
                    alternative_names.append(str(name).strip())

        successors = entity.get("successorEntities") or []
        if not successors and entity.get("successorEntity"):
            successors = [entity.get("successorEntity")]
        successor_leis = sorted({
            str(s.get("lei")).upper()
            for s in successors
            if isinstance(s, dict) and s.get("lei")
        })

        isins = attrs.get("isins") or []
        if isinstance(isins, str):
            isins = [isins]

        return {
            "lei": lei,
            "status": "FOUND",
            "source": source,
            "legal_name": legal_name,
            "alternative_names": sorted(set(alternative_names)),
            "entity_status": entity.get("status"),
            "registration_status": registration.get("status"),
            "legal_jurisdiction": entity.get("legalJurisdiction"),
            "legal_address_country": legal_address.get("country"),
            "hq_country": hq_address.get("country"),
            "successor_leis": successor_leis,
            "mapped_isins": sorted({str(v).upper() for v in isins if v}),
            "retrieved_datetime": utc_now_iso(),
        }
    except Exception as exc:
        return {
            "lei": lei,
            "status": "PARSE_ERROR",
            "source": source,
            "error": str(exc)[:300],
        }

def name_similarity(a: Any, b: Any) -> float:
    aa, bb = normalise_name(a) or "", normalise_name(b) or ""
    if not aa or not bb:
        return 0.0
    if aa == bb:
        return 1.0
    return SequenceMatcher(None, aa, bb).ratio()

def _script_family(value: Any) -> str:
    """Coarse script family used only to decide whether string similarity is meaningful."""
    text = str(value or "").strip()
    if not text:
        return "EMPTY"
    has_latin = False
    has_cjk = False
    has_hangul = False
    has_other_letters = False
    for ch in text:
        if not ch.isalpha():
            continue
        name = unicodedata.name(ch, "")
        if "LATIN" in name:
            has_latin = True
        elif any(k in name for k in ("CJK", "HIRAGANA", "KATAKANA", "IDEOGRAPH")):
            has_cjk = True
        elif "HANGUL" in name:
            has_hangul = True
        else:
            has_other_letters = True
    active = sum([has_latin, has_cjk, has_hangul, has_other_letters])
    if active > 1:
        return "MIXED"
    if has_latin:
        return "LATIN"
    if has_cjk:
        return "CJK"
    if has_hangul:
        return "HANGUL"
    if has_other_letters:
        return "OTHER"
    return "NONLETTER"

def names_are_directly_comparable(source_name: Any, candidate_names: list[Any]) -> bool:
    source_family = _script_family(source_name)
    if source_family in {"EMPTY", "NONLETTER"}:
        return False
    candidate_families = {_script_family(v) for v in candidate_names if v}
    if source_family == "MIXED":
        return True
    return source_family in candidate_families or "MIXED" in candidate_families

def best_name_similarity(source_name: Any, candidate_names: list[Any]) -> tuple[float, Optional[str]]:
    best_score, best_name = 0.0, None
    for candidate in candidate_names:
        if not candidate:
            continue
        score = name_similarity(source_name, candidate)
        if score > best_score:
            best_score, best_name = score, str(candidate)
    return best_score, best_name

IDENTITY_AI_CACHE_PATH = REGISTRY_DIR / "identity_semantic_cache.jsonl"
IDENTITY_AI_PROMPT_VERSION = "legal_name_relationship_1.0"

def _identity_openai_api_key() -> Optional[str]:
    key = os.getenv("OPENAI_API_KEY")
    if key and str(key).strip():
        return str(key).strip()
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key and str(key).strip():
            os.environ["OPENAI_API_KEY"] = str(key).strip()
            return str(key).strip()
    except Exception:
        pass
    return None

def _identity_ai_cache_key(payload: dict) -> str:
    normalized = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(
        f"LEGAL_NAME_RELATIONSHIP|{SCHEMA_VERSION}|{IDENTITY_AI_PROMPT_VERSION}|{normalized}".encode("utf-8")
    ).hexdigest()

def _load_identity_ai_cache() -> dict[str, dict]:
    cache = {}
    if IDENTITY_AI_CACHE_PATH.exists():
        for line in IDENTITY_AI_CACHE_PATH.read_text(encoding="utf-8").splitlines():
            try:
                rec = json.loads(line)
                cache[rec["input_hash"]] = rec
            except Exception:
                continue
    return cache

def semantic_name_relationship_review(
    source_name: Any,
    gleif_names: list[Any],
    source_country: Any,
    gleif_country: Any,
) -> Optional[dict]:
    """
    Optional semantic evidence only. The result never assigns or changes an LEI,
    legal_entity_id, security_id, economic_issuer_id, or any other canonical identity.
    """
    api_key = _identity_openai_api_key()
    if not api_key:
        return None

    payload = {
        "source_name": source_name,
        "gleif_names": [str(v) for v in gleif_names if v],
        "source_country": source_country,
        "gleif_country": gleif_country,
    }
    key = _identity_ai_cache_key(payload)
    cache = _load_identity_ai_cache()
    if key in cache:
        result = dict(cache[key]["response"])
        result["cache_hit"] = True
        return result

    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    schema = {
        "type": "object",
        "properties": {
            "relationship": {
                "type": "string",
                "enum": ["SAME_ENTITY_NAME_VARIANT", "NAME_CHANGE_OR_SUCCESSOR", "DIFFERENT_ENTITY", "UNCERTAIN"],
            },
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "reasoning_summary": {"type": "string"},
        },
        "required": ["relationship", "confidence", "reasoning_summary"],
        "additionalProperties": False,
    }
    response = client.responses.create(
        model=RESEARCH_CONFIG["ai"]["identity_semantic_model"],
        input=[
            {
                "role": "system",
                "content": (
                    "Classify only the semantic relationship between a source issuer name and names returned by GLEIF. "
                    "Account for translation, transliteration, abbreviations, punctuation, corporate suffixes and historical name changes. "
                    "Do not assign, validate, merge, split or infer any identifier or canonical corporate identity."
                ),
            },
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
        ],
        text={"format": {"type": "json_schema", "name": "legal_name_relationship", "schema": schema, "strict": True}},
    )
    result = json.loads(response.output_text)
    record = {
        "input_hash": key,
        "task_type": "LEGAL_NAME_RELATIONSHIP",
        "prompt_version": IDENTITY_AI_PROMPT_VERSION,
        "model": RESEARCH_CONFIG["ai"]["identity_semantic_model"],
        "input_payload": payload,
        "response": result,
        "created_datetime": utc_now_iso(),
    }
    with IDENTITY_AI_CACHE_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    result["cache_hit"] = False
    return result

def build_gleif_reference_table(x: pd.DataFrame) -> pd.DataFrame:
    leis = sorted({usable_lei(r) for _, r in x.iterrows() if usable_lei(r)})
    records = [gleif_get_lei_record(lei) for lei in leis] if leis else []
    return pd.DataFrame(records)

def fallback_security_key(row: pd.Series) -> tuple[str, str]:
    ticker = normalise_ticker(first_present(row.get("ticker_normalised"), row.get("ticker")))
    currency = normalise_currency(row.get("currency"))
    sec_name = normalise_name(row.get("security_name"))
    issuer = normalise_name(row.get("issuer_name"))
    if issuer and ticker: return "ISSUER_TICKER", f"{issuer}|{ticker}|{currency or ''}"
    if issuer and sec_name: return "ISSUER_SECURITY_NAME", f"{issuer}|{sec_name}|{currency or ''}"
    return "SOURCE_OBSERVATION", str(row.get("source_observation_id"))

def resolve_identity_evidence(evidence: pd.DataFrame) -> pd.DataFrame:
    x = evidence.copy()

    # 1) Resolve securities independently of legal-entity completeness.
    row_tokens, token_to_anchor, ambiguous_cusips = build_security_identifier_components(x)
    sec_types, sec_values, sec_methods = [], [], []
    for idx, row in x.iterrows():
        tokens, anchor = row_tokens.get(idx, []), None
        for token in tokens:
            if token[0] == "ISIN" and token in token_to_anchor:
                anchor = token_to_anchor[token]
                break
        if anchor is None:
            for token in tokens:
                if token[0] == "CUSIP" and token[1] in ambiguous_cusips:
                    continue
                if token in token_to_anchor:
                    anchor = token_to_anchor[token]
                    break
        if anchor:
            t, v = anchor
            sec_types.append(t)
            sec_values.append(v)
            sec_methods.append(
                "CUSIP_TO_ISIN_BRIDGE"
                if t == "ISIN" and ("ISIN", v) not in tokens
                else f"DIRECT_{t}"
            )
        else:
            t, v = fallback_security_key(row)
            sec_types.append(t)
            sec_values.append(v)
            sec_methods.append(t)

    x["security_identity_type"], x["security_identity_value"] = sec_types, sec_values
    x["security_identity_resolution_method"] = sec_methods
    x["security_id"] = [make_entity_id("SEC", "SECURITY", key) for key in sec_values]

    # 2) Enrich observed valid LEIs from GLEIF. Name comparison is multilingual-aware.
    # Low lexical similarity across different scripts is NOT treated as contradiction.
    global identity_reference_evidence_df, identity_quarantine_df
    identity_reference_evidence_df = build_gleif_reference_table(x)
    gleif_map = {}
    if not identity_reference_evidence_df.empty and "lei" in identity_reference_evidence_df.columns:
        gleif_map = identity_reference_evidence_df.set_index("lei").to_dict("index")

    legal_ids, legal_types, legal_values, legal_methods = [], [], [], []
    legal_conf, accepted_lei, quarantine_rows = [], [], []
    semantic_review_count = 0
    semantic_review_cache_hits = 0

    for idx, row in x.iterrows():
        lei, isin = usable_lei(row), usable_isin(row)
        issuer_name = row.get("issuer_name")
        record = gleif_map.get(lei, {}) if lei else {}
        decision, reason, confidence = "NO_LEI", "", 0.60
        semantic_review = None

        if lei:
            if record.get("status") == "FOUND":
                candidate_names = [record.get("legal_name")] + list(record.get("alternative_names") or [])
                sim, best_match_name = best_name_similarity(issuer_name, candidate_names)
                comparable = names_are_directly_comparable(issuer_name, candidate_names)

                mapped_isins = set(record.get("mapped_isins") or [])
                isin_match = bool(isin and isin in mapped_isins)

                record_country = normalise_country(
                    first_present(record.get("legal_jurisdiction"), record.get("legal_address_country"))
                )
                source_country = normalise_country(
                    first_present(row.get("issuer_country"), row.get("investment_country"))
                )
                isin_country = normalise_country(row.get("isin_country"))
                source_country_candidates = {c for c in [source_country, isin_country] if c}
                country_match = bool(record_country and record_country in source_country_candidates)
                country_conflict = bool(record_country and source_country_candidates and record_country not in source_country_candidates)

                if isin_match:
                    decision, reason, confidence = "ACCEPT", "GLEIF_ISIN_MATCH", 0.99
                elif sim >= 0.55:
                    decision, reason, confidence = "ACCEPT", "GLEIF_NAME_MATCH", 0.94
                elif sim >= 0.40 and country_match:
                    decision, reason, confidence = "ACCEPT", "GLEIF_NAME_COUNTRY_CONSISTENT", 0.90
                elif not comparable and country_match:
                    # Cross-script names are not contradictory. The LEI is source-reported
                    # and GLEIF confirms that the LEI exists in the expected jurisdiction.
                    decision, reason, confidence = "PROVISIONAL", "GLEIF_NAME_NOT_COMPARABLE", 0.86
                    semantic_review = semantic_name_relationship_review(
                        issuer_name, candidate_names, source_country, record_country
                    )
                elif country_match:
                    # Same jurisdiction but weak lexical match can be a historical rename.
                    decision, reason, confidence = "PROVISIONAL", "GLEIF_POSSIBLE_NAME_CHANGE", 0.80
                    semantic_review = semantic_name_relationship_review(
                        issuer_name, candidate_names, source_country, record_country
                    )
                elif comparable and sim < 0.30 and country_conflict:
                    # Strong deterministic contradiction: same-script name mismatch plus
                    # incompatible jurisdiction evidence. This catches source LEIs that
                    # clearly belong to another company without relying on AI.
                    decision, reason, confidence = "REJECT", "GLEIF_DETERMINISTIC_CONTRADICTION", 0.98
                else:
                    decision, reason, confidence = "QUARANTINE", "GLEIF_AMBIGUOUS", 0.55
                    semantic_review = semantic_name_relationship_review(
                        issuer_name, candidate_names, source_country, record_country
                    )

                if semantic_review is not None:
                    semantic_review_count += 1
                    semantic_review_cache_hits += int(bool(semantic_review.get("cache_hit")))
                    # AI is semantic evidence only. It never changes decision or canonical IDs.
                    relation = semantic_review.get("relationship")
                    reason = f"{reason}|AI_SEMANTIC_{relation}"

            elif record.get("status") in {"NOT_FOUND", "PARSE_ERROR"}:
                decision, reason, confidence = "QUARANTINE", "GLEIF_RECORD_NOT_VALIDATED", 0.40
            else:
                decision, reason, confidence = "PROVISIONAL", "GLEIF_UNAVAILABLE", 0.65

        if lei and decision in {"ACCEPT", "PROVISIONAL"}:
            ltype, lval = "LEI", lei
            if decision == "ACCEPT":
                method = "GLEIF_VALIDATED_LEI"
            elif reason.startswith("GLEIF_NAME_NOT_COMPARABLE"):
                method = "GLEIF_PROVISIONAL_CROSS_SCRIPT_LEI"
            elif reason.startswith("GLEIF_POSSIBLE_NAME_CHANGE"):
                method = "GLEIF_PROVISIONAL_NAME_CHANGE_LEI"
            else:
                method = "SOURCE_LEI_EXTERNAL_UNAVAILABLE"
            accepted_lei.append(lei)
        else:
            # Country is deliberately NOT part of fallback legal identity. N-PORT
            # investment country is supporting evidence and can vary by filer.
            name = normalise_name(issuer_name)
            if name:
                ltype, lval, method = "NAME", name, "NAME_FALLBACK"
            else:
                ltype, lval, method = (
                    "SOURCE_OBSERVATION",
                    str(row.get("source_observation_id")),
                    "SOURCE_OBSERVATION",
                )
            accepted_lei.append(None)

        legal_types.append(ltype)
        legal_values.append(lval)
        legal_methods.append(method)
        legal_conf.append(confidence)
        legal_ids.append(make_entity_id("LE", "LEGAL_ENTITY", lval))

        if lei and decision in {"REJECT", "QUARANTINE"}:
            quarantine_rows.append({
                "source_observation_id": row.get("source_observation_id"),
                "security_id": x.at[idx, "security_id"],
                "observed_lei": lei,
                "issuer_name": issuer_name,
                "isin": isin,
                "decision": decision,
                "reason": reason,
                "confidence": confidence,
                "gleif_legal_name": record.get("legal_name"),
                "gleif_alternative_names": record.get("alternative_names"),
                "gleif_status": record.get("status"),
                "semantic_relationship": (semantic_review or {}).get("relationship"),
                "semantic_confidence": (semantic_review or {}).get("confidence"),
            })

    x["legal_entity_id"], x["legal_identity_type"], x["legal_identity_value"] = legal_ids, legal_types, legal_values
    x["legal_identity_resolution_method"], x["legal_identity_confidence"] = legal_methods, legal_conf
    x["accepted_lei"] = accepted_lei
    identity_quarantine_df = pd.DataFrame(quarantine_rows)

    # 3) Build economic-issuer components as a graph. A persistent security may
    # legitimately pass through multiple legal entities; those legal entities remain
    # distinct while sharing one economic issuer.
    euf = UnionFind()
    for _, row in x.iterrows():
        euf.union(("SEC", row["security_id"]), ("LE", row["legal_entity_id"]))

    # Connect explicit GLEIF successor relationships where both LEIs are observed.
    lei_to_legal = (
        x.dropna(subset=["accepted_lei"])[["accepted_lei", "legal_entity_id"]]
        .drop_duplicates()
        .groupby("accepted_lei")["legal_entity_id"]
        .apply(list)
        .to_dict()
    )
    for lei, rec in gleif_map.items():
        for succ in rec.get("successor_leis") or []:
            if lei in lei_to_legal and succ in lei_to_legal:
                for a in lei_to_legal[lei]:
                    for b in lei_to_legal[succ]:
                        euf.union(("LE", a), ("LE", b))

    # Same accepted LEI always means the same legal entity/economic issuer.
    for lei, legal_list in lei_to_legal.items():
        for lid in legal_list[1:]:
            euf.union(("LE", legal_list[0]), ("LE", lid))

    component_key = {}
    for node in list(euf.parent):
        root = euf.find(node)
        component_key.setdefault(root, []).append(node)

    root_to_key = {}
    for root, members in component_key.items():
        leis = sorted({
            x.loc[x["legal_entity_id"].eq(v), "accepted_lei"].dropna().iloc[0]
            for t, v in members
            if t == "LE"
            and not x.loc[x["legal_entity_id"].eq(v), "accepted_lei"].dropna().empty
        })
        secs = sorted(v for t, v in members if t == "SEC")
        root_to_key[root] = (
            f"LEI_COMPONENT|{leis[0]}"
            if leis
            else f"SECURITY_COMPONENT|{secs[0]}"
        )

    x["economic_issuer_key"] = [
        root_to_key[euf.find(("SEC", sid))]
        for sid in x["security_id"]
    ]
    x["economic_issuer_id"] = [
        make_entity_id("EI", "ECONOMIC_ISSUER", key)
        for key in x["economic_issuer_key"]
    ]
    x["economic_issuer_resolution_method"] = np.where(
        x["accepted_lei"].notna(),
        "IDENTITY_GRAPH_WITH_GLEIF",
        "IDENTITY_GRAPH_SECURITY_CONTINUITY",
    )

    # 4) Choose one canonical legal parent per security for the Security Master,
    # while preserving every observed legal entity in legal history.
    canonical_parent = {}
    for sid, g in x.groupby("security_id", dropna=False):
        candidates = []
        for lid, lg in g.groupby("legal_entity_id", dropna=False):
            has_validated = lg["legal_identity_resolution_method"].eq("GLEIF_VALIDATED_LEI").any()
            has_lei = lg["accepted_lei"].notna().any()
            active_score = 0
            for accepted in lg["accepted_lei"].dropna().unique():
                rec = gleif_map.get(accepted, {})
                if str(rec.get("entity_status", "")).upper() == "ACTIVE":
                    active_score = max(active_score, 2)
                if str(rec.get("registration_status", "")).upper() == "ISSUED":
                    active_score = max(active_score, 3)
            candidates.append((
                int(has_validated),
                active_score,
                int(has_lei),
                len(lg),
                str(lid),
                lid,
            ))
        canonical_parent[sid] = sorted(candidates, reverse=True)[0][-1]

    x["canonical_security_legal_entity_id"] = x["security_id"].map(canonical_parent)

    # 5) Listing resolution remains independent of legal identity.
    resolutions = x.apply(resolve_listing_mic, axis=1)
    x["exchange_mic"] = [r[0] for r in resolutions]
    x["listing_resolution_method"] = [r[1] for r in resolutions]
    x["listing_resolution_confidence"] = [r[2] for r in resolutions]

    country_results = x.apply(infer_listing_country, axis=1)
    inferred_country = [r[0] for r in country_results]
    x["listing_country_resolution_method"] = [r[1] for r in country_results]
    x["listing_country_confidence"] = [r[2] for r in country_results]
    x["listing_country"] = [
        EXCHANGE_REGISTRY[m]["country"] if m in EXCHANGE_REGISTRY else c
        for m, c in zip(x["exchange_mic"], inferred_country)
    ]
    x["exchange_name"] = [
        EXCHANGE_REGISTRY.get(m, {}).get("exchange_name") if m else None
        for m in x["exchange_mic"]
    ]
    x["listing_ticker"] = x["ticker"].map(normalise_ticker)
    x["listing_key"] = [
        f"{sec}|{mic or 'UNRESOLVED'}|{tic or ''}|{cur or ''}"
        for sec, mic, tic, cur in zip(
            x["security_id"],
            x["exchange_mic"],
            x["listing_ticker"],
            x["currency"],
        )
    ]
    x["listing_id"] = [
        make_entity_id("LST", "LISTING", key)
        for key in x["listing_key"]
    ]

    # Runtime stats used by downstream QC.
    x.attrs["identity_semantic_review_count"] = semantic_review_count
    x.attrs["identity_semantic_review_cache_hits"] = semantic_review_cache_hits
    return x

resolved_identity_df = resolve_identity_evidence(identity_evidence_df)
_, _, _ambiguous_cusips_for_qc = build_security_identifier_components(resolved_identity_df)

identity_repair_stats = {
    "ambiguous_cusip_values_quarantined": len(_ambiguous_cusips_for_qc),
    "security_rows_cusip_to_isin_bridge": int(
        resolved_identity_df["security_identity_resolution_method"].eq("CUSIP_TO_ISIN_BRIDGE").sum()
    ),
    "external_lei_records": int(len(identity_reference_evidence_df)),
    "external_lei_validated_rows": int(
        resolved_identity_df["legal_identity_resolution_method"].eq("GLEIF_VALIDATED_LEI").sum()
    ),
    "external_lei_provisional_rows": int(
        resolved_identity_df["legal_identity_resolution_method"].isin([
            "GLEIF_PROVISIONAL_CROSS_SCRIPT_LEI",
            "GLEIF_PROVISIONAL_NAME_CHANGE_LEI",
            "SOURCE_LEI_EXTERNAL_UNAVAILABLE",
        ]).sum()
    ),
    "external_lei_quarantined_rows": int(len(identity_quarantine_df)),
    "identity_semantic_review_count": int(resolved_identity_df.attrs.get("identity_semantic_review_count", 0)),
    "identity_semantic_review_cache_hits": int(resolved_identity_df.attrs.get("identity_semantic_review_cache_hits", 0)),
    "invalid_cusip_rows_excluded_from_identity": int(
        resolved_identity_df.apply(
            lambda r: bool(normalise_identifier(first_present(r.get("cusip_normalised"), r.get("cusip"))))
            and usable_cusip(r) is None,
            axis=1,
        ).sum()
    ),
    "invalid_isin_rows_excluded_from_identity": int(
        resolved_identity_df.apply(
            lambda r: bool(normalise_identifier(first_present(r.get("isin_normalised"), r.get("isin"))))
            and usable_isin(r) is None,
            axis=1,
        ).sum()
    ),
}

print("Resolved rows:", len(resolved_identity_df))
print("Economic issuers:", resolved_identity_df["economic_issuer_id"].nunique())
print("Legal entities:", resolved_identity_df["legal_entity_id"].nunique())
print("Securities:", resolved_identity_df["security_id"].nunique())
print("Listings:", resolved_identity_df["listing_id"].nunique())
print("GLEIF records retrieved/cached:", identity_repair_stats["external_lei_records"])
print("GLEIF-validated rows:", identity_repair_stats["external_lei_validated_rows"])
print("GLEIF-provisional rows:", identity_repair_stats["external_lei_provisional_rows"])
print("Identity quarantine rows:", identity_repair_stats["external_lei_quarantined_rows"])
print("Identity semantic reviews:", identity_repair_stats["identity_semantic_review_count"])
print("Identity semantic review cache hits:", identity_repair_stats["identity_semantic_review_cache_hits"])


Resolved rows: 8144
Economic issuers: 421
Legal entities: 493
Securities: 493
Listings: 538
GLEIF records retrieved/cached: 321
GLEIF-validated rows: 6488
GLEIF-provisional rows: 251
Identity quarantine rows: 2
Identity semantic reviews: 253
Identity semantic review cache hits: 253


In [77]:
# 11. BUILD CANONICAL IDENTITY MASTERS AND TEMPORAL IDENTIFIER HISTORY

# PURPOSE
# -------
# Build the canonical:
#
#   Economic Issuer
#       ↓
#   Legal Entity
#       ↓
#   Security
#       ↓
#   Listing
#
# identity hierarchy from resolved_identity_df.
#
# Then:
#
# 11B
#   Detect possible duplicate economic issuers.
#
# 11C
#   Resolve only duplicates supported by authoritative
#   external reference evidence.
#
# IMPORTANT
# ---------
# - Fresh masters built here are authoritative for each run.
# - Stale in-memory aliases from earlier executions are reset.
# - Names alone never merge economic issuers.
# - AI never authoritatively changes identity.
# - Legal entities and securities remain distinct where required.
# - Authoritative economic-issuer continuity is preserved.
# - Step 11 is safe to rerun in the same Colab runtime.
# ============================================================


# ============================================================
# 11.0 — HELPERS
# ============================================================

def first_non_null(values: pd.Series):

    x = values.dropna()

    x = x[
        x.astype(str)
        .str.strip()
        .ne("")
    ]

    return (
        x.iloc[0]
        if len(x)
        else pd.NA
    )


def min_date(values: pd.Series):

    x = (
        pd.to_datetime(
            values,
            errors="coerce",
        )
        .dropna()
    )

    return (
        x.min()
        if len(x)
        else pd.NaT
    )


def max_date(values: pd.Series):

    x = (
        pd.to_datetime(
            values,
            errors="coerce",
        )
        .dropna()
    )

    return (
        x.max()
        if len(x)
        else pd.NaT
    )


# ============================================================
# 11.1 — LEGAL ENTITY MASTER
# ============================================================

def build_legal_entity_master(
    x: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for legal_id, g in x.groupby(
        "legal_entity_id",
        dropna=False,
    ):

        if "accepted_lei" in g.columns:

            lei_value = first_non_null(
                g["accepted_lei"]
            )

        else:

            lei_value = first_non_null(
                g["lei_normalised"]
            )

        rows.append(
            {
                "legal_entity_id":
                    legal_id,

                "economic_issuer_id":
                    first_non_null(
                        g["economic_issuer_id"]
                    ),

                "legal_entity_name":
                    first_non_null(
                        g["issuer_name"]
                    ),

                "entity_country":
                    first_non_null(
                        g["issuer_country"]
                    ),

                "lei":
                    lei_value,

                "cik":
                    pd.NA,

                "edinet_code":
                    pd.NA,

                "dart_corp_code":
                    pd.NA,

                "cninfo_entity_id":
                    pd.NA,

                "valid_from":
                    min_date(
                        g["snapshot_date"]
                    ),

                "valid_to":
                    pd.NaT,
            }
        )

    return ensure_columns(
        pd.DataFrame(rows),
        LEGAL_ENTITY_COLUMNS,
    )


# ============================================================
# 11.2 — SECURITY HELPERS
# ============================================================

def classify_security_type(
    name: Any,
) -> str:

    n = (
        clean_string(name)
        or ""
    ).upper()

    if (
        "ADR" in n
        or "AMERICAN DEPOSIT" in n
    ):
        return "ADR"

    if (
        "GDR" in n
        or "GLOBAL DEPOSIT" in n
    ):
        return "GDR"

    if (
        "PREFERRED" in n
        or "PREFERENCE" in n
    ):
        return "PREFERRED"

    return "COMMON"


def classify_share_class(
    name: Any,
    ticker: Any,
) -> Optional[str]:

    n = (
        clean_string(name)
        or ""
    ).upper()

    t = (
        normalise_ticker(ticker)
        or ""
    )

    match = re.search(
        r"\bCLASS\s+([A-Z0-9]+)\b",
        n,
    )

    if match:
        return match.group(1)

    match = re.search(
        r"[.-]([A-Z])$",
        t,
    )

    return (
        match.group(1)
        if match
        else None
    )


# ============================================================
# 11.3 — SECURITY MASTER
# ============================================================

def build_security_master(
    x: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for sec_id, g in x.groupby(
        "security_id",
        dropna=False,
    ):

        name = first_non_null(
            g["security_name"]
        )

        ticker = first_non_null(
            g["ticker"]
        )

        if (
            "canonical_security_legal_entity_id"
            in g.columns
        ):

            legal_entity_id = (
                first_non_null(
                    g[
                        "canonical_security_legal_entity_id"
                    ]
                )
            )

        else:

            legal_entity_id = (
                first_non_null(
                    g["legal_entity_id"]
                )
            )

        isin_value = first_non_null(
            g["isin_normalised"]
            .map(
                lambda v:
                    usable_identifier(
                        v,
                        "ISIN",
                    )
            )
        )

        cusip_value = first_non_null(
            g["cusip_normalised"]
            .map(
                lambda v:
                    usable_identifier(
                        v,
                        "CUSIP",
                    )
            )
        )

        rows.append(
            {
                "security_id":
                    sec_id,

                "economic_issuer_id":
                    first_non_null(
                        g["economic_issuer_id"]
                    ),

                "legal_entity_id":
                    legal_entity_id,

                "security_name":
                    name,

                "security_type":
                    classify_security_type(
                        name
                    ),

                "share_class":
                    classify_share_class(
                        name,
                        ticker,
                    ),

                "isin":
                    isin_value,

                "cusip":
                    cusip_value,

                "sedol":
                    pd.NA,

                "figi":
                    pd.NA,

                "currency":
                    first_non_null(
                        g["currency"]
                    ),

                "valid_from":
                    min_date(
                        g["snapshot_date"]
                    ),

                "valid_to":
                    pd.NaT,
            }
        )

    return ensure_columns(
        pd.DataFrame(rows),
        SECURITY_COLUMNS,
    )


# ============================================================
# 11.4 — LISTING MASTER
# ============================================================

def build_listing_master(
    x: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for listing_id, g in x.groupby(
        "listing_id",
        dropna=False,
    ):

        mic = first_non_null(
            g["exchange_mic"]
        )

        listing_country = (
            first_non_null(
                g["listing_country"]
            )
        )

        route = route_from_mic(
            mic,
            listing_country,
        )

        rows.append(
            {
                "listing_id":
                    listing_id,

                "security_id":
                    first_non_null(
                        g["security_id"]
                    ),

                "economic_issuer_id":
                    first_non_null(
                        g["economic_issuer_id"]
                    ),

                "ticker":
                    first_non_null(
                        g["listing_ticker"]
                    ),

                "exchange_name":
                    first_non_null(
                        g["exchange_name"]
                    ),

                "exchange_mic":
                    mic,

                "listing_country":
                    listing_country,

                "trading_currency":
                    first_non_null(
                        g["currency"]
                    ),

                "primary_listing_flag":
                    pd.NA,

                "valid_from":
                    min_date(
                        g["snapshot_date"]
                    ),

                "valid_to":
                    pd.NaT,

                "source_route":
                    route[
                        "primary_source_engine"
                    ],

                "source_system":
                    route[
                        "primary_source_system"
                    ],
            }
        )

    out = ensure_columns(
        pd.DataFrame(rows),
        LISTING_COLUMNS,
    )

    # Conservative rule:
    # if a security has exactly one listing,
    # that listing may safely be marked primary.
    counts = (
        out
        .groupby(
            "security_id"
        )[
            "listing_id"
        ]
        .transform(
            "count"
        )
    )

    out.loc[
        counts.eq(1),
        "primary_listing_flag",
    ] = True

    return out


# ============================================================
# 11.5 — ECONOMIC ISSUER MASTER
# ============================================================

def build_economic_issuer_master(
    x: pd.DataFrame,
) -> pd.DataFrame:

    rows = []

    for issuer_id, g in x.groupby(
        "economic_issuer_id",
        dropna=False,
    ):

        legal_name = first_non_null(
            g["issuer_name"]
        )

        rows.append(
            {
                "economic_issuer_id":
                    issuer_id,

                "issuer_name":
                    legal_name,

                "issuer_name_normalised":
                    normalise_name(
                        legal_name
                    ),

                "legal_name":
                    legal_name,

                "country_of_domicile":
                    first_non_null(
                        g["issuer_country"]
                    ),

                "headquarters_country":
                    pd.NA,

                "industry_theme":
                    RESEARCH_THEME,

                "theme_relevance":
                    "Unclassified",

                "theme_subindustry":
                    pd.NA,

                "theme_classification_method":
                    "UNCLASSIFIED",

                "theme_confidence":
                    pd.NA,

                "active_from":
                    min_date(
                        g["snapshot_date"]
                    ),

                "active_to":
                    pd.NaT,

                "predecessor_issuer_id":
                    pd.NA,

                "successor_issuer_id":
                    pd.NA,
            }
        )

    return ensure_columns(
        pd.DataFrame(rows),
        ECONOMIC_ISSUER_COLUMNS,
    )


# ============================================================
# 11.6 — TEMPORAL IDENTIFIER HISTORY
# ============================================================

def build_identifier_history(
    x: pd.DataFrame,
) -> pd.DataFrame:

    specs = [
        (
            "LEGAL_ENTITY",
            "legal_entity_id",
            "LEI",
            "accepted_lei",
        ),
        (
            "SECURITY",
            "security_id",
            "ISIN",
            "isin_normalised",
        ),
        (
            "SECURITY",
            "security_id",
            "CUSIP",
            "cusip_normalised",
        ),
        (
            "LISTING",
            "listing_id",
            "TICKER",
            "listing_ticker",
        ),
        (
            "LISTING",
            "listing_id",
            "MIC",
            "exchange_mic",
        ),
    ]

    parts = []

    for (
        entity_type,
        id_col,
        ident_type,
        value_col,
    ) in specs:

        if value_col not in x.columns:
            continue

        temp = x[
            [
                id_col,
                value_col,
                "snapshot_date",
                "source_system",
                "source_observation_id",
            ]
        ].copy()

        if ident_type in {
            "LEI",
            "ISIN",
            "CUSIP",
        }:

            temp[
                value_col
            ] = temp[
                value_col
            ].map(
                lambda v:
                    usable_identifier(
                        v,
                        ident_type,
                    )
            )

        temp = temp.dropna(
            subset=[
                id_col,
                value_col,
            ]
        )

        temp = temp.loc[
            temp[
                value_col
            ]
            .astype(str)
            .str.strip()
            .ne("")
        ]

        if temp.empty:
            continue

        hist = (
            temp
            .groupby(
                [
                    id_col,
                    value_col,
                    "source_system",
                ],
                dropna=False,
            )
            .agg(
                first_observed_date=(
                    "snapshot_date",
                    "min",
                ),

                last_observed_date=(
                    "snapshot_date",
                    "max",
                ),

                source_observation_id=(
                    "source_observation_id",
                    "first",
                ),
            )
            .reset_index()
        )

        hist[
            "entity_type"
        ] = entity_type

        hist[
            "entity_id"
        ] = hist[
            id_col
        ]

        hist[
            "identifier_type"
        ] = ident_type

        hist[
            "identifier_value"
        ] = hist[
            value_col
        ]

        hist[
            "valid_from"
        ] = hist[
            "first_observed_date"
        ]

        hist[
            "valid_to"
        ] = pd.NaT

        hist[
            "is_primary"
        ] = ident_type in {
            "LEI",
            "ISIN",
            "CUSIP",
        }

        parts.append(
            hist
        )

    if not parts:

        return pd.DataFrame(
            columns=
                IDENTIFIER_HISTORY_COLUMNS
        )

    return ensure_columns(
        pd.concat(
            parts,
            ignore_index=True,
            sort=False,
        ),
        IDENTIFIER_HISTORY_COLUMNS,
    )


# ============================================================
# 11.7 — BUILD FRESH MASTERS
# ============================================================
#
# These objects are freshly reconstructed from resolved_identity_df
# EVERY time Step 11 runs.
#
# They deliberately overwrite any stale in-memory aliases from a
# previous execution.
# ============================================================

economic_issuer_master_df = (
    build_economic_issuer_master(
        resolved_identity_df
    )
)

legal_entity_master_df = (
    build_legal_entity_master(
        resolved_identity_df
    )
)

security_master_df = (
    build_security_master(
        resolved_identity_df
    )
)

listing_master_df = (
    build_listing_master(
        resolved_identity_df
    )
)

identifier_history_df = (
    build_identifier_history(
        resolved_identity_df
    )
)


# ------------------------------------------------------------
# IMPORTANT:
# Reset canonical aliases explicitly.
#
# Do NOT search globals() for an older version first.
# ------------------------------------------------------------

economic_issuer_master = (
    economic_issuer_master_df.copy()
)

legal_entity_master = (
    legal_entity_master_df.copy()
)

security_master = (
    security_master_df.copy()
)

listing_master = (
    listing_master_df.copy()
)


print(
    "Economic issuer master:",
    len(economic_issuer_master),
)

print(
    "Legal entity master:",
    len(legal_entity_master),
)

print(
    "Security master:",
    len(security_master),
)

print(
    "Listing master:",
    len(listing_master),
)


# ============================================================
# 11.8 — CANONICAL BLOCK 1 OUTPUT DIRECTORY
# ============================================================

from pathlib import Path


def _resolve_block1_canonical_dir():

    candidate_names = [
        "BLOCK1_CANONICAL_DIR",
        "CANONICAL_DIR",
        "canonical_dir",
        "CANONICAL_OUTPUT_DIR",
        "canonical_output_dir",
        "CANONICAL_PATH",
        "canonical_path",
    ]

    for name in candidate_names:

        value = globals().get(
            name
        )

        if value is None:
            continue

        path = Path(
            value
        )

        # If the supplied path is the parent canonical directory,
        # descend into the Block 1 namespace.
        if (
            path.name.lower()
            == "canonical"
        ):

            path = (
                path
                / "block_1"
            )

        path.mkdir(
            parents=True,
            exist_ok=True,
        )

        return path

    path = Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "AI-Powered Global PIT Equity Research Engine/"
        "data/canonical/block_1"
    )

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    return path


BLOCK1_CANONICAL_DIR = (
    _resolve_block1_canonical_dir()
)


# ============================================================
# 11B. ECONOMIC-ISSUER DUPLICATE CANDIDATE AUDIT
# ============================================================
#
# Candidate detection only.
#
# 11B NEVER mutates canonical identity.
# ============================================================

DUPLICATE_NAME_STRONG_THRESHOLD = 0.94
DUPLICATE_NAME_REVIEW_THRESHOLD = 0.84


# ------------------------------------------------------------
# Generic entity-name normalisation
# ------------------------------------------------------------

LEGAL_SUFFIX_TOKENS = {
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "co",
    "company",
    "ltd",
    "limited",
    "plc",
    "llc",
    "lp",
    "sa",
    "sas",
    "ag",
    "se",
    "nv",
    "bv",
    "spa",
    "srl",
    "oyj",
    "ab",
    "asa",
    "as",
    "pte",
    "pty",
    "holdings",
    "holding",
    "group",
}


def normalise_entity_name_for_duplicate_detection(
    value,
):

    if (
        value is None
        or pd.isna(value)
    ):
        return None

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE,
    )

    value = re.sub(
        r"_+",
        " ",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    if not value:
        return None

    tokens = value.split()

    while (
        tokens
        and tokens[-1]
        in LEGAL_SUFFIX_TOKENS
    ):

        tokens.pop()

    while (
        tokens
        and tokens[0] == "the"
    ):

        tokens.pop(0)

    result = (
        " ".join(tokens)
        .strip()
    )

    return (
        result
        or None
    )


def name_similarity(
    a,
    b,
):

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


print()

print(
    "Canonical dataframe aliases reset from fresh Step 11 output:"
)

print(
    f"  Economic issuers: "
    f"{len(economic_issuer_master):,}"
)

print(
    f"  Legal entities:   "
    f"{len(legal_entity_master):,}"
)

print(
    f"  Securities:       "
    f"{len(security_master):,}"
)

print(
    f"  Listings:         "
    f"{len(listing_master):,}"
)

print()


# ------------------------------------------------------------
# Build issuer evidence
# ------------------------------------------------------------

issuer_base = (
    economic_issuer_master
    .copy()
)

issuer_base[
    "duplicate_name_normalised"
] = (
    issuer_base[
        "issuer_name"
    ]
    .map(
        normalise_entity_name_for_duplicate_detection
    )
)


legal_evidence = (
    legal_entity_master
    .groupby(
        "economic_issuer_id",
        dropna=False,
    )
    .agg(
        legal_entity_ids=(
            "legal_entity_id",
            lambda s:
                sorted(
                    {
                        str(x)
                        for x
                        in s.dropna()
                    }
                ),
        ),

        leis=(
            "lei",
            lambda s:
                sorted(
                    {
                        str(x)
                        .upper()
                        .strip()
                        for x
                        in s.dropna()
                        if str(x).strip()
                    }
                ),
        ),

        legal_entity_names=(
            "legal_entity_name",
            lambda s:
                sorted(
                    {
                        str(x)
                        .strip()
                        for x
                        in s.dropna()
                        if str(x).strip()
                    }
                ),
        ),
    )
    .reset_index()
)


security_evidence = (
    security_master
    .groupby(
        "economic_issuer_id",
        dropna=False,
    )
    .agg(
        security_ids=(
            "security_id",
            lambda s:
                sorted(
                    {
                        str(x)
                        for x
                        in s.dropna()
                    }
                ),
        ),

        isins=(
            "isin",
            lambda s:
                sorted(
                    {
                        str(x)
                        .upper()
                        .strip()
                        for x
                        in s.dropna()
                        if str(x).strip()
                    }
                ),
        ),

        cusips=(
            "cusip",
            lambda s:
                sorted(
                    {
                        str(x)
                        .upper()
                        .strip()
                        for x
                        in s.dropna()
                        if str(x).strip()
                    }
                ),
        ),
    )
    .reset_index()
)


issuer_evidence = (
    issuer_base
    .merge(
        legal_evidence,
        on="economic_issuer_id",
        how="left",
    )
    .merge(
        security_evidence,
        on="economic_issuer_id",
        how="left",
    )
)


for col in [
    "legal_entity_ids",
    "leis",
    "legal_entity_names",
    "security_ids",
    "isins",
    "cusips",
]:

    issuer_evidence[
        col
    ] = issuer_evidence[
        col
    ].apply(
        lambda x:
            x
            if isinstance(
                x,
                list,
            )
            else []
    )


# ------------------------------------------------------------
# Generate candidate pairs
# ------------------------------------------------------------

DUPLICATE_CANDIDATE_COLUMNS = [
    "economic_issuer_id_left",
    "economic_issuer_id_right",
    "issuer_name_left",
    "issuer_name_right",
    "normalised_name_left",
    "normalised_name_right",
    "name_similarity",
    "shared_lei",
    "shared_isin",
    "shared_cusip",
    "shared_security_id",
    "candidate_strength",
    "authoritative_merge_allowed",
    "resolution_status",
    "resolution_reason",
]


candidate_rows = []

records = (
    issuer_evidence
    .to_dict(
        "records"
    )
)


for left, right in combinations(
    records,
    2,
):

    left_id = left[
        "economic_issuer_id"
    ]

    right_id = right[
        "economic_issuer_id"
    ]

    if left_id == right_id:
        continue

    left_name = left.get(
        "duplicate_name_normalised"
    )

    right_name = right.get(
        "duplicate_name_normalised"
    )

    similarity = name_similarity(
        left_name,
        right_name,
    )

    shared_lei = sorted(
        set(
            left.get(
                "leis",
                [],
            )
        )
        &
        set(
            right.get(
                "leis",
                [],
            )
        )
    )

    shared_isin = sorted(
        set(
            left.get(
                "isins",
                [],
            )
        )
        &
        set(
            right.get(
                "isins",
                [],
            )
        )
    )

    shared_cusip = sorted(
        set(
            left.get(
                "cusips",
                [],
            )
        )
        &
        set(
            right.get(
                "cusips",
                [],
            )
        )
    )

    shared_security_id = sorted(
        set(
            left.get(
                "security_ids",
                [],
            )
        )
        &
        set(
            right.get(
                "security_ids",
                [],
            )
        )
    )

    exact_normalised_name = bool(
        left_name
        and right_name
        and (
            left_name
            == right_name
        )
    )

    strong_identifier_overlap = bool(
        shared_lei
        or shared_isin
        or shared_cusip
        or shared_security_id
    )

    candidate = bool(
        exact_normalised_name
        or strong_identifier_overlap
        or (
            similarity
            >= DUPLICATE_NAME_REVIEW_THRESHOLD
        )
    )

    if not candidate:
        continue

    if strong_identifier_overlap:

        candidate_strength = (
            "STRONG_REFERENCE_EVIDENCE"
        )

    elif exact_normalised_name:

        candidate_strength = (
            "EXACT_NORMALISED_NAME"
        )

    elif (
        similarity
        >= DUPLICATE_NAME_STRONG_THRESHOLD
    ):

        candidate_strength = (
            "STRONG_NAME_SIMILARITY"
        )

    else:

        candidate_strength = (
            "SEMANTIC_REVIEW"
        )

    if shared_security_id:

        resolution_reason = (
            "SHARED_CANONICAL_SECURITY"
        )

    elif shared_isin:

        resolution_reason = (
            "SHARED_ISIN"
        )

    elif shared_cusip:

        resolution_reason = (
            "SHARED_CUSIP"
        )

    elif shared_lei:

        resolution_reason = (
            "SHARED_LEI_REQUIRES_TEMPORAL_"
            "LEGAL_ENTITY_REVIEW"
        )

    elif (
        candidate_strength
        == "EXACT_NORMALISED_NAME"
    ):

        resolution_reason = (
            "EXACT_NORMALISED_NAME_"
            "REQUIRES_REFERENCE_REVIEW"
        )

    elif (
        candidate_strength
        == "STRONG_NAME_SIMILARITY"
    ):

        resolution_reason = (
            "STRONG_NAME_SIMILARITY_"
            "REQUIRES_REFERENCE_REVIEW"
        )

    else:

        resolution_reason = (
            "SEMANTIC_DUPLICATE_CANDIDATE"
        )

    candidate_rows.append(
        {
            "economic_issuer_id_left":
                left_id,

            "economic_issuer_id_right":
                right_id,

            "issuer_name_left":
                left.get(
                    "issuer_name"
                ),

            "issuer_name_right":
                right.get(
                    "issuer_name"
                ),

            "normalised_name_left":
                left_name,

            "normalised_name_right":
                right_name,

            "name_similarity":
                round(
                    float(
                        similarity
                    ),
                    6,
                ),

            "shared_lei":
                shared_lei,

            "shared_isin":
                shared_isin,

            "shared_cusip":
                shared_cusip,

            "shared_security_id":
                shared_security_id,

            "candidate_strength":
                candidate_strength,

            "authoritative_merge_allowed":
                False,

            "resolution_status":
                "REVIEW_REQUIRED",

            "resolution_reason":
                resolution_reason,
        }
    )


economic_issuer_duplicate_candidates = pd.DataFrame(
    candidate_rows,
    columns=
        DUPLICATE_CANDIDATE_COLUMNS,
)


# ------------------------------------------------------------
# Save pre-resolution audit
# ------------------------------------------------------------

duplicate_audit_path = (
    BLOCK1_CANONICAL_DIR
    / "economic_issuer_duplicate_candidates.parquet"
)


economic_issuer_duplicate_candidates.to_parquet(
    duplicate_audit_path,
    index=False,
)


print(
    f"Duplicate audit saved to: "
    f"{duplicate_audit_path}"
)

print("=" * 72)

print(
    "ECONOMIC ISSUER DUPLICATE AUDIT"
)

print("=" * 72)

print(
    f"Economic issuers: "
    f"{economic_issuer_master['economic_issuer_id'].nunique():,}"
)

print(
    f"Duplicate candidates: "
    f"{len(economic_issuer_duplicate_candidates):,}"
)


if not economic_issuer_duplicate_candidates.empty:

    display_cols = [
        "economic_issuer_id_left",
        "issuer_name_left",
        "economic_issuer_id_right",
        "issuer_name_right",
        "name_similarity",
        "candidate_strength",
        "resolution_reason",
    ]

    display(
        economic_issuer_duplicate_candidates[
            display_cols
        ]
        .sort_values(
            [
                "candidate_strength",
                "name_similarity",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )


print()

print(
    "NOTE: This audit does not automatically merge economic issuers."
)

print(
    "Canonical identity changes remain "
    "deterministic/reference-data decisions."
)


# ============================================================
# 11C. AUTHORITATIVE ECONOMIC-ISSUER DUPLICATE RESOLUTION
# ============================================================
#
# Authoritative evidence only.
#
# Company-specific facts below are reference-data records.
# The executable resolution engine remains generic.
# ============================================================


# ============================================================
# 11C.1 — AUTHORITATIVE RESOLUTION EVIDENCE
# ============================================================

authoritative_issuer_resolutions = pd.DataFrame(
    [
        {
            "absorbed_economic_issuer_id":
                "EI_171C89C6D6C7F8CF4413",

            "surviving_economic_issuer_id":
                "EI_37751A68347964AB667F",

            "resolution_type":
                "SAME_ECONOMIC_ISSUER",

            "evidence_type":
                "AUTHORITATIVE_LEGAL_ENTITY_CONTINUITY",

            "effective_date":
                None,

            "evidence_authority":
                "UK Companies House / SEC",

            "evidence_reference":
                (
                    "UK company 13624182; current issuer "
                    "Polestar Automotive Holding UK PLC"
                ),

            "resolution_reason":
                (
                    "Authoritative registry and SEC evidence "
                    "support one economic issuer; alternate "
                    "source name is not a separate economic "
                    "business."
                ),
        },

        {
            "absorbed_economic_issuer_id":
                "EI_3107FA0761A1BE682FAA",

            "surviving_economic_issuer_id":
                "EI_9873ADE3C58BBD64103C",

            "resolution_type":
                "SUCCESSOR_CONTINUITY",

            "evidence_type":
                "SEC_SUCCESSOR_ISSUER",

            "effective_date":
                "2021-04-20",

            "evidence_authority":
                "U.S. SEC",

            "evidence_reference":
                (
                    "Marvell Technology Inc established as "
                    "successor issuer to "
                    "Marvell Technology Group Ltd."
                ),

            "resolution_reason":
                (
                    "Corporate reorganisation changed the "
                    "parent legal entity while preserving "
                    "economic-issuer continuity."
                ),
        },

        {
            "absorbed_economic_issuer_id":
                "EI_AD9BEACD02BCD45CA36F",

            "surviving_economic_issuer_id":
                "EI_3C406EAC76F5E8EF71CA",

            "resolution_type":
                "SAME_ECONOMIC_ISSUER",

            "evidence_type":
                "AUTHORITATIVE_MULTI_LISTING_CONFIRMATION",

            "effective_date":
                None,

            "evidence_authority":
                "Hesai / SEC / HKEX",

            "evidence_reference":
                (
                    "Hesai Group Nasdaq HSAI ADS and HKEX "
                    "2525 ordinary shares are securities "
                    "of the same issuer."
                ),

            "resolution_reason":
                (
                    "Separate securities/listings were "
                    "incorrectly represented as separate "
                    "economic issuers."
                ),
        },

        {
            "absorbed_economic_issuer_id":
                "EI_71A9094755A6CEBE1BE0",

            "surviving_economic_issuer_id":
                "EI_828A0E111BDC0BDAE69E",

            "resolution_type":
                "SUCCESSOR_CONTINUITY",

            "evidence_type":
                "SEC_REDOMICILIATION_MERGER",

            "effective_date":
                "2024-09-30",

            "evidence_authority":
                "U.S. SEC",

            "evidence_reference":
                (
                    "TE Connectivity Ltd merged into "
                    "TE Connectivity plc; plc survived "
                    "and shareholders received one share "
                    "for each former Ltd share."
                ),

            "resolution_reason":
                (
                    "Legal-parent redomiciliation does not "
                    "create a new economic issuer."
                ),
        },
    ]
)


authoritative_issuer_resolutions[
    "effective_date"
] = pd.to_datetime(
    authoritative_issuer_resolutions[
        "effective_date"
    ],
    errors="coerce",
)


authoritative_issuer_resolutions[
    "resolution_datetime"
] = pd.Timestamp.now(
    tz="UTC"
)


authoritative_issuer_resolutions[
    "resolution_method"
] = (
    "AUTHORITATIVE_REFERENCE_DETERMINISTIC"
)


# ============================================================
# 11C.2 — VALIDATE RESOLUTION EVIDENCE
#         AND DETECT APPLICATION STATE
# ============================================================

required_resolution_columns = {
    "absorbed_economic_issuer_id",
    "surviving_economic_issuer_id",
    "resolution_type",
    "evidence_type",
    "evidence_authority",
    "evidence_reference",
    "resolution_reason",
}


missing_resolution_columns = (
    required_resolution_columns
    - set(
        authoritative_issuer_resolutions.columns
    )
)


if missing_resolution_columns:

    raise RuntimeError(
        "Resolution evidence table missing columns: "
        f"{sorted(missing_resolution_columns)}"
    )


if authoritative_issuer_resolutions[
    "absorbed_economic_issuer_id"
].duplicated().any():

    raise RuntimeError(
        "An absorbed economic issuer appears "
        "more than once."
    )


if (
    authoritative_issuer_resolutions[
        "absorbed_economic_issuer_id"
    ]
    ==
    authoritative_issuer_resolutions[
        "surviving_economic_issuer_id"
    ]
).any():

    raise RuntimeError(
        "Resolution table contains a self-merge."
    )


existing_issuer_ids = set(
    economic_issuer_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


resolution_states = []


for _, row in (
    authoritative_issuer_resolutions
    .iterrows()
):

    absorbed_id = str(
        row[
            "absorbed_economic_issuer_id"
        ]
    )

    surviving_id = str(
        row[
            "surviving_economic_issuer_id"
        ]
    )

    absorbed_exists = (
        absorbed_id
        in existing_issuer_ids
    )

    survivor_exists = (
        surviving_id
        in existing_issuer_ids
    )

    if (
        absorbed_exists
        and survivor_exists
    ):

        state = "PENDING"

    elif (
        not absorbed_exists
        and survivor_exists
    ):

        state = "ALREADY_APPLIED"

    else:

        state = "INVALID"

    resolution_states.append(
        state
    )


authoritative_issuer_resolutions[
    "application_state"
] = resolution_states


invalid_resolutions = (
    authoritative_issuer_resolutions[
        authoritative_issuer_resolutions[
            "application_state"
        ]
        .eq(
            "INVALID"
        )
    ]
    .copy()
)


if not invalid_resolutions.empty:

    raise RuntimeError(
        "Authoritative issuer resolution table "
        "contains invalid references.\n\n"
        +
        invalid_resolutions[
            [
                "absorbed_economic_issuer_id",
                "surviving_economic_issuer_id",
                "application_state",
            ]
        ]
        .to_string(
            index=False
        )
    )


pending_issuer_resolutions = (
    authoritative_issuer_resolutions[
        authoritative_issuer_resolutions[
            "application_state"
        ]
        .eq(
            "PENDING"
        )
    ]
    .copy()
)


already_applied_issuer_resolutions = (
    authoritative_issuer_resolutions[
        authoritative_issuer_resolutions[
            "application_state"
        ]
        .eq(
            "ALREADY_APPLIED"
        )
    ]
    .copy()
)


# ============================================================
# 11C.3 — REQUIRE PENDING RESOLUTIONS TO MATCH
#         CURRENT 11B CANDIDATES
# ============================================================

candidate_pairs = set()


for _, row in (
    economic_issuer_duplicate_candidates
    .iterrows()
):

    pair = frozenset(
        [
            str(
                row[
                    "economic_issuer_id_left"
                ]
            ),
            str(
                row[
                    "economic_issuer_id_right"
                ]
            ),
        ]
    )

    candidate_pairs.add(
        pair
    )


for _, row in (
    pending_issuer_resolutions
    .iterrows()
):

    resolution_pair = frozenset(
        [
            str(
                row[
                    "absorbed_economic_issuer_id"
                ]
            ),
            str(
                row[
                    "surviving_economic_issuer_id"
                ]
            ),
        ]
    )

    if (
        resolution_pair
        not in candidate_pairs
    ):

        raise RuntimeError(
            "Pending authoritative resolution does "
            "not correspond to a current Step 11B "
            "duplicate candidate: "
            f"{sorted(resolution_pair)}"
        )


# ============================================================
# 11C.4 — BUILD TRANSITIVE MERGE MAP
# ============================================================

direct_merge_map = dict(
    zip(
        authoritative_issuer_resolutions[
            "absorbed_economic_issuer_id"
        ],
        authoritative_issuer_resolutions[
            "surviving_economic_issuer_id"
        ],
    )
)


def resolve_surviving_issuer_id(
    issuer_id,
):

    if pd.isna(
        issuer_id
    ):
        return issuer_id

    current = str(
        issuer_id
    )

    seen = set()

    while (
        current
        in direct_merge_map
    ):

        if current in seen:

            raise RuntimeError(
                "Cycle detected in economic "
                "issuer merge map."
            )

        seen.add(
            current
        )

        current = (
            direct_merge_map[
                current
            ]
        )

    return current


economic_issuer_merge_map = (
    authoritative_issuer_resolutions
    .copy()
)


economic_issuer_merge_map[
    "canonical_economic_issuer_id"
] = (
    economic_issuer_merge_map[
        "surviving_economic_issuer_id"
    ]
    .map(
        resolve_surviving_issuer_id
    )
)


# ============================================================
# 11C.5 — PRESERVE PRE-MERGE SNAPSHOTS
# ============================================================

economic_issuer_master_pre_11c = (
    economic_issuer_master.copy()
)

legal_entity_master_pre_11c = (
    legal_entity_master.copy()
)

security_master_pre_11c = (
    security_master.copy()
)

listing_master_pre_11c = (
    listing_master.copy()
)


# ============================================================
# 11C.6 — GENERIC ECONOMIC-ISSUER REMAPPER
# ============================================================

def remap_economic_issuer_column(
    df,
):

    if not isinstance(
        df,
        pd.DataFrame,
    ):
        return df

    if (
        "economic_issuer_id"
        not in df.columns
    ):
        return df

    out = df.copy()

    out[
        "economic_issuer_id"
    ] = (
        out[
            "economic_issuer_id"
        ]
        .map(
            resolve_surviving_issuer_id
        )
    )

    return out


# ------------------------------------------------------------
# Remap canonical identity masters
# ------------------------------------------------------------

legal_entity_master = (
    remap_economic_issuer_column(
        legal_entity_master
    )
)

security_master = (
    remap_economic_issuer_column(
        security_master
    )
)

listing_master = (
    remap_economic_issuer_column(
        listing_master
    )
)


# ============================================================
# 11C.7 — REBUILD ECONOMIC ISSUER MASTER
# ============================================================

absorbed_ids = set(
    authoritative_issuer_resolutions[
        "absorbed_economic_issuer_id"
    ]
    .astype(str)
)


economic_issuer_master = (
    economic_issuer_master[
        ~economic_issuer_master[
            "economic_issuer_id"
        ]
        .astype(str)
        .isin(
            absorbed_ids
        )
    ]
    .copy()
)


economic_issuer_master = (
    economic_issuer_master
    .drop_duplicates(
        subset=[
            "economic_issuer_id"
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11C.8 — REMAP EXISTING UPSTREAM IDENTITY-BEARING TABLES
# ============================================================
#
# Deliberately excludes theme_classification_df.
#
# Theme classification belongs to Step 12 and should be generated
# from the final post-11C issuer master on a fresh pipeline run.
# ============================================================

upstream_candidate_names = [
    "historical_universe_membership_df",
    "historical_universe_membership",

    "universe_membership_df",
    "universe_membership",

    "identity_evidence_df",
    "identity_reference_evidence_df",
]


remapped_upstream_tables = []


for object_name in (
    upstream_candidate_names
):

    obj = globals().get(
        object_name
    )

    if not isinstance(
        obj,
        pd.DataFrame,
    ):
        continue

    if (
        "economic_issuer_id"
        not in obj.columns
    ):
        continue

    globals()[
        object_name
    ] = (
        remap_economic_issuer_column(
            obj
        )
    )

    remapped_upstream_tables.append(
        object_name
    )


# ============================================================
# 11C.9 — ALIAS / RESOLUTION LINEAGE
# ============================================================

economic_issuer_alias_history = (
    authoritative_issuer_resolutions[
        [
            "absorbed_economic_issuer_id",
            "surviving_economic_issuer_id",
            "effective_date",
            "resolution_type",
            "evidence_type",
            "evidence_authority",
            "evidence_reference",
            "resolution_reason",
            "resolution_method",
            "resolution_datetime",
            "application_state",
        ]
    ]
    .copy()
)


economic_issuer_alias_history[
    "canonical_economic_issuer_id"
] = (
    economic_issuer_alias_history[
        "surviving_economic_issuer_id"
    ]
    .map(
        resolve_surviving_issuer_id
    )
)


# ============================================================
# 11C.10 — CRITICAL POST-MERGE INVARIANTS
# ============================================================

security_parent_counts = (
    security_master
    .groupby(
        "security_id",
        dropna=False,
    )[
        "economic_issuer_id"
    ]
    .nunique(
        dropna=True
    )
)


security_parent_conflicts = (
    security_parent_counts[
        security_parent_counts
        > 1
    ]
)


if len(
    security_parent_conflicts
) > 0:

    raise AssertionError(
        "Step 11C created security → "
        "economic issuer conflicts."
    )


legal_parent_ids = set(
    legal_entity_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


security_parent_ids = set(
    security_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


listing_parent_ids = set(
    listing_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


canonical_parent_ids = set(
    economic_issuer_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


orphan_parent_ids = (
    (
        legal_parent_ids
        | security_parent_ids
        | listing_parent_ids
    )
    - canonical_parent_ids
)


if orphan_parent_ids:

    raise AssertionError(
        "Step 11C created orphan economic "
        "issuer references: "
        f"{sorted(orphan_parent_ids)}"
    )


remaining_absorbed_refs = 0


for df in [
    legal_entity_master,
    security_master,
    listing_master,
]:

    remaining_absorbed_refs += int(
        df[
            "economic_issuer_id"
        ]
        .astype(str)
        .isin(
            absorbed_ids
        )
        .sum()
    )


if (
    remaining_absorbed_refs
    > 0
):

    raise AssertionError(
        "Absorbed economic issuer IDs remain "
        "in canonical identity masters."
    )


# ============================================================
# 11C.11 — MARK RESOLVED DUPLICATE PAIRS
# ============================================================

resolved_pair_keys = {
    frozenset(
        [
            str(
                row[
                    "absorbed_economic_issuer_id"
                ]
            ),
            str(
                row[
                    "surviving_economic_issuer_id"
                ]
            ),
        ]
    )
    for _, row
    in authoritative_issuer_resolutions
    .iterrows()
}


def candidate_resolution_status(
    row,
):

    pair = frozenset(
        [
            str(
                row[
                    "economic_issuer_id_left"
                ]
            ),
            str(
                row[
                    "economic_issuer_id_right"
                ]
            ),
        ]
    )

    if pair in resolved_pair_keys:

        return (
            "RESOLVED_AUTHORITATIVE"
        )

    return (
        "REVIEW_REQUIRED"
    )


if not economic_issuer_duplicate_candidates.empty:

    economic_issuer_duplicate_candidates[
        "resolution_status"
    ] = (
        economic_issuer_duplicate_candidates
        .apply(
            candidate_resolution_status,
            axis=1,
        )
    )


# ============================================================
# 11C.12 — SYNCHRONISE FINAL CANONICAL OBJECT NAMES
# ============================================================
#
# Downstream code may use either *_master or *_master_df.
#
# After this point both representations contain the same
# post-resolution canonical state.
# ============================================================

economic_issuer_master_df = (
    economic_issuer_master.copy()
)

legal_entity_master_df = (
    legal_entity_master.copy()
)

security_master_df = (
    security_master.copy()
)

listing_master_df = (
    listing_master.copy()
)


# ============================================================
# 11C.13 — PERSIST RESOLUTION LINEAGE
# ============================================================

resolution_path = (
    BLOCK1_CANONICAL_DIR
    / "economic_issuer_resolution_evidence.parquet"
)


alias_history_path = (
    BLOCK1_CANONICAL_DIR
    / "economic_issuer_alias_history.parquet"
)


duplicate_candidates_path = (
    BLOCK1_CANONICAL_DIR
    / "economic_issuer_duplicate_candidates.parquet"
)


economic_issuer_merge_map.to_parquet(
    resolution_path,
    index=False,
)


economic_issuer_alias_history.to_parquet(
    alias_history_path,
    index=False,
)


economic_issuer_duplicate_candidates.to_parquet(
    duplicate_candidates_path,
    index=False,
)


# ============================================================
# 11C.14 — FINAL REPORT
# ============================================================

pre_count = (
    economic_issuer_master_pre_11c[
        "economic_issuer_id"
    ]
    .nunique()
)


post_count = (
    economic_issuer_master[
        "economic_issuer_id"
    ]
    .nunique()
)


pending_resolution_count = len(
    pending_issuer_resolutions
)


already_applied_resolution_count = len(
    already_applied_issuer_resolutions
)


total_resolution_count = len(
    authoritative_issuer_resolutions
)


if economic_issuer_duplicate_candidates.empty:

    remaining_strong_candidates = (
        economic_issuer_duplicate_candidates
        .copy()
    )

else:

    remaining_strong_candidates = (
        economic_issuer_duplicate_candidates[
            (
                economic_issuer_duplicate_candidates[
                    "candidate_strength"
                ]
                .isin(
                    [
                        "STRONG_REFERENCE_EVIDENCE",
                        "EXACT_NORMALISED_NAME",
                        "STRONG_NAME_SIMILARITY",
                    ]
                )
            )
            &
            (
                economic_issuer_duplicate_candidates[
                    "resolution_status"
                ]
                .ne(
                    "RESOLVED_AUTHORITATIVE"
                )
            )
        ]
        .copy()
    )


print()

print("=" * 80)

print(
    "STEP 11C — AUTHORITATIVE ECONOMIC-ISSUER "
    "DUPLICATE RESOLUTION"
)

print("=" * 80)


print(
    f"Economic issuers before resolution: "
    f"{pre_count:,}"
)


print(
    f"Authoritative resolutions registered: "
    f"{total_resolution_count:,}"
)


print(
    f"Resolutions applied this run: "
    f"{pending_resolution_count:,}"
)


print(
    f"Resolutions already present upstream: "
    f"{already_applied_resolution_count:,}"
)


print(
    f"Economic issuers after resolution: "
    f"{post_count:,}"
)


print(
    f"Legal entities: "
    f"{len(legal_entity_master):,}"
)


print(
    f"Securities: "
    f"{len(security_master):,}"
)


print(
    f"Listings: "
    f"{len(listing_master):,}"
)


print(
    f"Security → economic issuer conflicts: "
    f"{len(security_parent_conflicts):,}"
)


print(
    f"Orphan economic issuer references: "
    f"{len(orphan_parent_ids):,}"
)


print(
    f"Remaining strong duplicate candidates: "
    f"{len(remaining_strong_candidates):,}"
)


print(
    f"Upstream tables remapped in memory: "
    f"{len(remapped_upstream_tables):,}"
)


if remapped_upstream_tables:

    print(
        "Remapped tables: "
        + ", ".join(
            remapped_upstream_tables
        )
    )


print()

print(
    "Resolution evidence:"
)


display(
    authoritative_issuer_resolutions[
        [
            "absorbed_economic_issuer_id",
            "surviving_economic_issuer_id",
            "resolution_type",
            "evidence_type",
            "effective_date",
            "evidence_authority",
            "application_state",
        ]
    ]
)


if len(
    remaining_strong_candidates
) > 0:

    print()

    print(
        "STRONG DUPLICATE CANDIDATES "
        "STILL REQUIRING REVIEW:"
    )

    display(
        remaining_strong_candidates[
            [
                "economic_issuer_id_left",
                "issuer_name_left",
                "economic_issuer_id_right",
                "issuer_name_right",
                "candidate_strength",
                "resolution_status",
            ]
        ]
    )


print()

print(
    f"Saved resolution evidence: "
    f"{resolution_path}"
)

print(
    f"Saved issuer alias history: "
    f"{alias_history_path}"
)

print(
    f"Saved duplicate audit: "
    f"{duplicate_candidates_path}"
)

print("=" * 80)


# ============================================================
# 11C.15 — FINAL ASSERTIONS
# ============================================================

assert (
    post_count
    ==
    (
        pre_count
        - pending_resolution_count
    )
), (
    "Economic issuer count did not fall by the "
    "expected number of pending authoritative "
    "resolutions."
)


assert len(
    security_parent_conflicts
) == 0, (
    "Security → economic issuer invariant failed."
)


assert len(
    orphan_parent_ids
) == 0, (
    "Orphan economic issuer references remain."
)


assert (
    remaining_absorbed_refs
    == 0
), (
    "Absorbed economic issuer IDs remain "
    "in canonical masters."
)


assert len(
    remaining_strong_candidates
) == 0, (
    "Strong economic-issuer duplicate "
    "candidates remain unresolved."
)


print()

print(
    "Step 11C critical identity checks: PASS"
)


# ============================================================
# STEP 11 COMPLETE
# ============================================================

print()

print("=" * 80)

print(
    "STEP 11 — FINAL CANONICAL IDENTITY STATE"
)

print("=" * 80)

print(
    f"Economic issuers: "
    f"{len(economic_issuer_master_df):,}"
)

print(
    f"Legal entities: "
    f"{len(legal_entity_master_df):,}"
)

print(
    f"Securities: "
    f"{len(security_master_df):,}"
)

print(
    f"Listings: "
    f"{len(listing_master_df):,}"
)

print(
    f"Identifier history rows: "
    f"{len(identifier_history_df):,}"
)

print(
    f"Unresolved strong issuer duplicates: "
    f"{len(remaining_strong_candidates):,}"
)

print(
    f"Security → economic issuer conflicts: "
    f"{len(security_parent_conflicts):,}"
)

print("=" * 80)

print()

print(
    "Step 11 complete."
)

Economic issuer master: 421
Legal entity master: 493
Security master: 493
Listing master: 538

Canonical dataframe aliases reset from fresh Step 11 output:
  Economic issuers: 421
  Legal entities:   493
  Securities:       493
  Listings:         538

Duplicate audit saved to: /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/economic_issuer_duplicate_candidates.parquet
ECONOMIC ISSUER DUPLICATE AUDIT
Economic issuers: 421
Duplicate candidates: 15


,economic_issuer_id_left,issuer_name_left,economic_issuer_id_right,issuer_name_right,name_similarity,candidate_strength,resolution_reason
1,EI_171C89C6D6C7F8CF4413,Polestar Automotive Holding UK,EI_37751A68347964AB667F,POLESTAR AUTOMOTIVE HOLDING UK PLC,1.000000,EXACT_NORMALISED_NAME,EXACT_NORMALISED_NAME_REQUIRES_REFERENCE_REVIEW
3,EI_3107FA0761A1BE682FAA,Marvell Technology Inc,EI_9873ADE3C58BBD64103C,Marvell Technology Group Ltd,1.000000,EXACT_NORMALISED_NAME,EXACT_NORMALISED_NAME_REQUIRES_REFERENCE_REVIEW
5,EI_3C406EAC76F5E8EF71CA,Hesai Group,EI_AD9BEACD02BCD45CA36F,Hesai Group,1.000000,EXACT_NORMALISED_NAME,EXACT_NORMALISED_NAME_REQUIRES_REFERENCE_REVIEW
9,EI_71A9094755A6CEBE1BE0,TE Connectivity Ltd,EI_828A0E111BDC0BDAE69E,TE CONNECTIVITY PLC,1.000000,EXACT_NORMALISED_NAME,EXACT_NORMALISED_NAME_REQUIRES_REFERENCE_REVIEW
14,EI_F9C90BDADC07812ED62E,Aeva Technologies Inc,EI_FE27CCB4C2CDEFFEA0B3,Via Technologies Inc,0.909091,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE
8,EI_62407F34888F00F32F40,SAIC Motor Corporation Limited,EI_C8BE34EBE3E0C1415CB1,BAIC Motor Corp Ltd,0.900000,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE
6,EI_43093EA9C0AD5FE28C42,"KANDI TECHNOLOGIES GROUP, INC.",EI_5A4803096B115EFAB40A,Niu Technologies,0.882353,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE
10,EI_AC3A0ED3317D91E5EB15,ON Semiconductor Corp,EI_CDCEA9ED3531803D805D,NXP Semiconductors NV,0.882353,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE
12,EI_DB49ACF682923BC672F0,Himax Technologies Inc,EI_FE27CCB4C2CDEFFEA0B3,Via Technologies Inc,0.882353,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE
7,EI_5A4803096B115EFAB40A,Niu Technologies,EI_FE27CCB4C2CDEFFEA0B3,Via Technologies Inc,0.875000,SEMANTIC_REVIEW,SEMANTIC_DUPLICATE_CANDIDATE



NOTE: This audit does not automatically merge economic issuers.
Canonical identity changes remain deterministic/reference-data decisions.

STEP 11C — AUTHORITATIVE ECONOMIC-ISSUER DUPLICATE RESOLUTION
Economic issuers before resolution: 421
Authoritative resolutions registered: 4
Resolutions applied this run: 4
Resolutions already present upstream: 0
Economic issuers after resolution: 417
Legal entities: 493
Securities: 493
Listings: 538
Security → economic issuer conflicts: 0
Orphan economic issuer references: 0
Remaining strong duplicate candidates: 0
Upstream tables remapped in memory: 1
Remapped tables: historical_universe_membership_df

Resolution evidence:


,absorbed_economic_issuer_id,surviving_economic_issuer_id,resolution_type,evidence_type,effective_date,evidence_authority,application_state
0,EI_171C89C6D6C7F8CF4413,EI_37751A68347964AB667F,SAME_ECONOMIC_ISSUER,AUTHORITATIVE_LEGAL_ENTITY_CONTINUITY,NaT,UK Companies House / SEC,PENDING
1,EI_3107FA0761A1BE682FAA,EI_9873ADE3C58BBD64103C,SUCCESSOR_CONTINUITY,SEC_SUCCESSOR_ISSUER,2021-04-20,U.S. SEC,PENDING
2,EI_AD9BEACD02BCD45CA36F,EI_3C406EAC76F5E8EF71CA,SAME_ECONOMIC_ISSUER,AUTHORITATIVE_MULTI_LISTING_CONFIRMATION,NaT,Hesai / SEC / HKEX,PENDING
3,EI_71A9094755A6CEBE1BE0,EI_828A0E111BDC0BDAE69E,SUCCESSOR_CONTINUITY,SEC_REDOMICILIATION_MERGER,2024-09-30,U.S. SEC,PENDING



Saved resolution evidence: /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/economic_issuer_resolution_evidence.parquet
Saved issuer alias history: /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/economic_issuer_alias_history.parquet
Saved duplicate audit: /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/economic_issuer_duplicate_candidates.parquet

Step 11C critical identity checks: PASS

STEP 11 — FINAL CANONICAL IDENTITY STATE
Economic issuers: 417
Legal entities: 493
Securities: 493
Listings: 538
Identifier history rows: 1,159
Unresolved strong issuer duplicates: 0
Security → economic issuer conflicts: 0

Step 11 complete.


In [78]:
# 12. GATED AI THEME CLASSIFICATION

# COST-EFFICIENT SEMANTIC ARCHITECTURE
# ------------------------------------
#
# Final canonical economic issuers from Step 11
#       ↓
# Existing persisted theme_classification.parquet?
#       │
#       ├── YES
#       │     ↓
#       │  Apply Step 11C issuer alias lineage
#       │     ↓
#       │  Validate persisted semantic rows
#       │     ↓
#       │  Deterministically select one valid row
#       │  per current canonical economic issuer
#       │     ↓
#       │  REUSE — ZERO NEW AI COST
#       │
#       └── NO / missing issuer
#               ↓
#       Deterministic exact AI cache
#               ↓ miss
#       GPT-5 nano — NO WEB
#               ↓
#       confident + valid?
#       ├── YES → accept
#       └── NO  → GPT-5.6 Luna + web research
#
# Before any NEW issuer-level AI call:
#
#       COST-SAFETY PREFLIGHT
#               ↓
#       report:
#       - canonical issuers
#       - persisted semantic reuse
#       - exact cache availability
#       - genuinely new fast calls
#       - known new web calls
#
#       ↓
#
# 12B. TARGETED CLASSIFICATION ERROR REPAIR
# ------------------------------------------
#
# Detect only:
#
#     theme_classification_method == "ERROR"
#
#       ↓
# Retry only failed issuers
#       ↓
# GPT-5.6 Luna + web
#       ↓
# Require zero remaining ERROR rows
#
#       ↓
#
# 12C. THEME SUBINDUSTRY HARMONISATION
# -------------------------------------
#
# Current raw subindustries
#       ↓
# Existing persisted taxonomy registry
#       ↓
# Existing raw label?
#       ├── YES → reuse canonical mapping — ZERO AI COST
#       └── NO
#             ↓
#       Are there any genuinely new raw labels?
#             ├── NO → finish
#             └── YES
#                    ↓
#             GPT-5.6 Terra
#             incremental harmonisation only
#
# AI MAY:
# - classify issuer economics semantically;
# - research ambiguous issuers using public web information;
# - assess material exposure to the configured theme;
# - propose raw subindustry labels;
# - repair failed semantic classifications;
# - harmonise genuinely new raw labels.
#
# AI MAY NOT:
# - assign or modify canonical identity;
# - merge economic issuers;
# - alter identifiers;
# - determine PIT timestamps;
# - perform authoritative source routing;
# - perform arithmetic truth;
# - mutate canonical identity tables.
#
# DESIGN PRINCIPLES
# -----------------
# - Configuration, not hard-coding.
# - Persisted semantic work is reusable research capital.
# - Existing valid research is preferred to new AI calls.
# - Exact AI cache is the second reuse layer.
# - Web search is escalation only.
# - Taxonomy is reused incrementally.
# - Raw and canonical subindustries remain separate.
# - Error repair touches only failed rows.
# - No industry-specific executable taxonomy.
# - Parquet remains canonical persisted format.
# ============================================================


# ============================================================
# 12.0 — RESOLVE CURRENT CANONICAL ISSUER MASTER
# ============================================================

def _resolve_existing_df_step12(*candidate_names):

    for name in candidate_names:

        obj = globals().get(name)

        if isinstance(
            obj,
            pd.DataFrame,
        ):

            return obj

    raise NameError(
        "Could not find any expected dataframe variable: "
        + ", ".join(candidate_names)
    )


economic_issuer_master = (
    _resolve_existing_df_step12(
        "economic_issuer_master",
        "economic_issuer_master_df",
    )
    .copy()
)


if (
    "economic_issuer_id"
    not in economic_issuer_master.columns
):

    raise RuntimeError(
        "economic_issuer_master lacks economic_issuer_id."
    )


if economic_issuer_master[
    "economic_issuer_id"
].duplicated().any():

    raise RuntimeError(
        "Step 12 requires one row per canonical economic issuer."
    )


CURRENT_CANONICAL_ISSUER_IDS = set(
    economic_issuer_master[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


# ============================================================
# 12.1 — STORAGE DIRECTORIES
# ============================================================

def _resolve_block1_directory_step12(
    kind: str,
) -> Path:

    candidates = {

        "canonical": [
            "BLOCK1_CANONICAL_DIR",
            "CANONICAL_DIR",
            "canonical_dir",
            "CANONICAL_OUTPUT_DIR",
            "canonical_output_dir",
            "CANONICAL_PATH",
            "canonical_path",
        ],

        "reference_cache": [
            "REFERENCE_CACHE_DIR",
            "reference_cache_dir",
            "REFERENCE_CACHE_PATH",
            "reference_cache_path",
            "REFERENCE_DIR",
            "reference_dir",
            "CACHE_DIR",
            "cache_dir",
        ],
    }


    if kind not in candidates:

        raise ValueError(
            f"Unknown Block 1 directory kind: {kind}"
        )


    for name in candidates[kind]:

        value = globals().get(
            name
        )

        if value is None:

            continue


        path = Path(
            value
        )


        if (
            kind == "canonical"
            and path.name.lower() == "canonical"
        ):

            path = (
                path
                / "block_1"
            )


        path.mkdir(
            parents=True,
            exist_ok=True,
        )

        return path


    project_root = Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "AI-Powered Global PIT Equity Research Engine"
    )


    if kind == "canonical":

        path = (
            project_root
            / "data"
            / "canonical"
            / "block_1"
        )

    else:

        path = (
            project_root
            / "data"
            / "reference_cache"
        )


    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    return path


BLOCK1_CANONICAL_DIR = (
    _resolve_block1_directory_step12(
        "canonical"
    )
)


BLOCK1_REFERENCE_CACHE_DIR = (
    _resolve_block1_directory_step12(
        "reference_cache"
    )
)


# ============================================================
# 12.2 — MODEL + TASK CONFIGURATION
# ============================================================

FAST_CLASSIFICATION_MODEL = (
    RESEARCH_CONFIG
    .get("ai", {})
    .get(
        "fast_theme_classification_model",
        "gpt-5-nano",
    )
)


WEB_RESEARCH_MODEL = (
    RESEARCH_CONFIG
    .get("ai", {})
    .get(
        "theme_web_research_model",
        "gpt-5.6-luna",
    )
)


TAXONOMY_HARMONISATION_MODEL = (
    RESEARCH_CONFIG
    .get("ai", {})
    .get(
        "taxonomy_harmonisation_model",
        "gpt-5.6-terra",
    )
)


FAST_ACCEPT_CONFIDENCE = float(
    RESEARCH_CONFIG
    .get("ai", {})
    .get(
        "fast_classification_accept_confidence",
        0.90,
    )
)


# ------------------------------------------------------------
# Cost-safety guard
# ------------------------------------------------------------
#
# This is NOT an AI enable/disable toggle.
#
# AI remains automatic when required.
#
# This merely prevents an accidental cache/path/schema problem from
# launching hundreds of unexpected new calls.
# ------------------------------------------------------------

MAX_NEW_FAST_CALLS_SAFETY = int(
    RESEARCH_CONFIG
    .get("ai", {})
    .get(
        "max_new_theme_fast_calls_per_run",
        25,
    )
)


FAST_CLASSIFICATION_TASK_TYPE = (
    "THEME_CLASSIFICATION_FAST_GATE"
)

FAST_CLASSIFICATION_PROMPT_VERSION = (
    "theme_fast_gate_v1"
)

FAST_CLASSIFICATION_SCHEMA_VERSION = (
    "1.0"
)


WEB_CLASSIFICATION_TASK_TYPE = (
    "THEME_CLASSIFICATION_WEB_RESEARCH"
)

WEB_CLASSIFICATION_PROMPT_VERSION = (
    "theme_web_materiality_v5_gated"
)

WEB_CLASSIFICATION_SCHEMA_VERSION = (
    "5.0"
)


ERROR_REPAIR_TASK_TYPE = (
    "THEME_CLASSIFICATION_WEB_ERROR_REPAIR"
)

ERROR_REPAIR_PROMPT_VERSION = (
    "theme_web_error_repair_v1"
)

ERROR_REPAIR_SCHEMA_VERSION = (
    "1.0"
)


SUBINDUSTRY_HARMONISATION_TASK_TYPE = (
    "THEME_SUBINDUSTRY_HARMONISATION"
)

SUBINDUSTRY_HARMONISATION_PROMPT_VERSION = (
    "theme_taxonomy_discovery_v5_incremental"
)

SUBINDUSTRY_HARMONISATION_SCHEMA_VERSION = (
    "5.0"
)


OPENAI_RESPONSES_URL = (
    "https://api.openai.com/v1/responses"
)


OPENAI_TIMEOUT_SECONDS = 240

MAX_CONSECUTIVE_IDENTICAL_ERRORS = 3


# ============================================================
# 12.3 — JSON-SAFE SERIALISATION
# ============================================================

def json_safe(
    value: Any,
):

    if value is None:

        return None


    if isinstance(
        value,
        dict,
    ):

        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):

        return [
            json_safe(v)
            for v in value
        ]


    if isinstance(
        value,
        np.ndarray,
    ):

        return [
            json_safe(v)
            for v in value.tolist()
        ]


    if isinstance(
        value,
        Path,
    ):

        return str(value)


    if isinstance(
        value,
        (
            pd.Timestamp,
            datetime,
            date,
        ),
    ):

        return value.isoformat()


    if isinstance(
        value,
        np.generic,
    ):

        return json_safe(
            value.item()
        )


    try:

        missing = pd.isna(
            value
        )

        if (
            isinstance(
                missing,
                (
                    bool,
                    np.bool_,
                ),
            )
            and bool(missing)
        ):

            return None

    except Exception:

        pass


    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):

        return value


    return str(
        value
    )


def json_dumps_safe(
    value,
    **kwargs,
):

    return json.dumps(
        json_safe(
            value
        ),
        **kwargs,
    )


# ============================================================
# 12.4 — OPENAI API KEY
# ============================================================

def get_openai_api_key():

    key = os.getenv(
        "OPENAI_API_KEY"
    )


    if key:

        return key.strip()


    try:

        from google.colab import userdata

        key = userdata.get(
            "OPENAI_API_KEY"
        )

        if key:

            return key.strip()

    except Exception:

        pass


    return None


OPENAI_API_KEY = (
    get_openai_api_key()
)


# ============================================================
# 12.5 — GENERIC CACHE FUNCTIONS
# ============================================================

def load_ai_cache(
    path: Path,
):

    cache = {}


    if not path.exists():

        return cache


    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = line.strip()

            if not line:

                continue


            try:

                record = json.loads(
                    line
                )

                cache[
                    record[
                        "cache_key"
                    ]
                ] = record

            except Exception:

                continue


    return cache


def append_ai_cache_record(
    path: Path,
    record: dict,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.write(
            json_dumps_safe(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


def ai_cache_key(
    task_type,
    schema_version,
    prompt_version,
    model,
    payload,
):

    normalized = (
        json_dumps_safe(
            payload,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False,
        )
    )


    signature = (
        f"{task_type}|"
        f"{schema_version}|"
        f"{prompt_version}|"
        f"{model}|"
        f"{normalized}"
    )


    return hashlib.sha256(
        signature.encode(
            "utf-8"
        )
    ).hexdigest()


# ============================================================
# 12.6 — RESEARCH THEME
# ============================================================

research_theme = (
    RESEARCH_CONFIG
    .get("project", {})
    .get(
        "research_theme"
    )
)


if not research_theme:

    research_theme = (
        globals().get(
            "RESEARCH_THEME"
        )
    )


if (
    not isinstance(
        research_theme,
        str,
    )
    or not research_theme.strip()
):

    raise ValueError(
        "No valid research theme found."
    )


research_theme = (
    research_theme.strip()
)


# ============================================================
# 12.7 — CACHE FILES
# ============================================================

ai_cache_dir = (
    BLOCK1_REFERENCE_CACHE_DIR
    / "ai"
)


ai_cache_dir.mkdir(
    parents=True,
    exist_ok=True,
)


fast_cache_path = (
    ai_cache_dir
    / "theme_fast_gate_cache.jsonl"
)


web_cache_path = (
    ai_cache_dir
    / "theme_web_semantic_cache.jsonl"
)


taxonomy_cache_path = (
    ai_cache_dir
    / "theme_subindustry_taxonomy_cache.jsonl"
)


fast_cache = load_ai_cache(
    fast_cache_path
)


web_cache = load_ai_cache(
    web_cache_path
)


taxonomy_cache = load_ai_cache(
    taxonomy_cache_path
)


# ============================================================
# 12.8 — CANONICAL OUTPUT PATHS
# ============================================================

theme_classification_path = (
    BLOCK1_CANONICAL_DIR
    / "theme_classification.parquet"
)


theme_classification_pre_harmonisation_path = (
    BLOCK1_CANONICAL_DIR
    / "theme_classification_pre_harmonisation.parquet"
)


theme_subindustry_taxonomy_path = (
    BLOCK1_CANONICAL_DIR
    / "theme_subindustry_taxonomy.parquet"
)


issuer_alias_history_path = (
    BLOCK1_CANONICAL_DIR
    / "economic_issuer_alias_history.parquet"
)


# ============================================================
# 12.9 — ISSUER PAYLOAD
# ============================================================

def theme_classification_payload(
    row,
):

    return json_safe(
        {
            "economic_issuer_id":
                row.get(
                    "economic_issuer_id"
                ),

            "issuer_name":
                row.get(
                    "issuer_name"
                ),

            "legal_name":
                row.get(
                    "legal_name"
                ),

            "country_of_domicile":
                row.get(
                    "country_of_domicile"
                ),

            "headquarters_country":
                row.get(
                    "headquarters_country"
                ),

            "research_theme":
                research_theme,
        }
    )


# ============================================================
# 12.10 — STRUCTURED OUTPUT SCHEMAS
# ============================================================

FAST_RESPONSE_SCHEMA = {

    "type": "object",

    "additionalProperties": False,

    "properties": {

        "relevance": {
            "type": "string",
            "enum": [
                "Core",
                "High",
                "Moderate",
                "Peripheral",
            ],
        },

        "subindustry_raw": {
            "type": [
                "string",
                "null",
            ],
        },

        "confidence": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
        },

        "business_description": {
            "type": [
                "string",
                "null",
            ],
        },

        "materiality_reasoning": {
            "type": [
                "string",
                "null",
            ],
        },

        "needs_web_research": {
            "type": "boolean",
        },

        "abstain": {
            "type": "boolean",
        },

        "abstain_reason": {
            "type": [
                "string",
                "null",
            ],
        },
    },

    "required": [
        "relevance",
        "subindustry_raw",
        "confidence",
        "business_description",
        "materiality_reasoning",
        "needs_web_research",
        "abstain",
        "abstain_reason",
    ],
}


WEB_RESPONSE_SCHEMA = {

    "type": "object",

    "additionalProperties": False,

    "properties": {

        "relevance": {
            "type": "string",
            "enum": [
                "Core",
                "High",
                "Moderate",
                "Peripheral",
            ],
        },

        "subindustry_raw": {
            "type": [
                "string",
                "null",
            ],
        },

        "confidence": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
        },

        "business_description": {
            "type": [
                "string",
                "null",
            ],
        },

        "materiality_reasoning": {
            "type": [
                "string",
                "null",
            ],
        },

        "abstain": {
            "type": "boolean",
        },

        "abstain_reason": {
            "type": [
                "string",
                "null",
            ],
        },
    },

    "required": [
        "relevance",
        "subindustry_raw",
        "confidence",
        "business_description",
        "materiality_reasoning",
        "abstain",
        "abstain_reason",
    ],
}


# ============================================================
# 12.11 — PROMPTS
# ============================================================

FAST_SYSTEM_PROMPT = f"""
You are the low-cost semantic classification gate for a global
point-in-time equity research database.

Configured research theme:

{research_theme}

You are NOT browsing the web.

Determine whether the supplied economic issuer can be classified
with HIGH CONFIDENCE using:
- supplied canonical issuer metadata; and
- existing model knowledge.

If there is meaningful uncertainty about what the issuer does,
whether the theme is economically material, or the correct
subindustry, set:

needs_web_research = true

Do not guess from a company name.

This is NOT identity resolution.

Never:
- assign or modify IDs;
- merge issuers;
- alter identifiers;
- determine PIT timestamps.

MATERIALITY

Core:
The theme is the principal business or a defining dominant segment.

High:
The theme is substantial and strategically important.

Moderate:
Exposure is genuine and economically meaningful but secondary.

Peripheral:
The relationship is genuine but economically limited.

Generic service providers must not receive substantial relevance
merely because customers operate in the configured theme.

Set needs_web_research = false only when highly confident about:
1. issuer activity;
2. theme relevance;
3. materiality;
4. descriptive raw subindustry.

Return only structured output.
""".strip()


WEB_SYSTEM_PROMPT = f"""
You are the evidence-backed semantic research layer of a global
point-in-time equity research database.

Configured research theme:

{research_theme}

Use public web information to determine what the issuer actually does
and assess MATERIAL ECONOMIC EXPOSURE to the configured theme.

This is NOT identity resolution.

Never:
- alter canonical IDs;
- merge issuers;
- replace identifiers;
- determine PIT timestamps;
- invent facts.

Prefer:
1. issuer/company sources;
2. regulatory filings;
3. exchanges/government;
4. established financial/business sources.

MATERIALITY

Core:
Principal business or defining dominant segment.

High:
Substantial and strategically important component.

Moderate:
Real and economically meaningful but secondary.

Peripheral:
Genuine but limited.

Do not assign substantial relevance merely because an issuer serves
customers in the theme.

Return a concise descriptive raw subindustry representing the
economic activity responsible for exposure.

If reliable evidence remains insufficient, abstain.

If abstain = false, subindustry_raw must be populated.

Return only structured output.
""".strip()


# ============================================================
# 12.12 — RESPONSES API HELPERS
# ============================================================

def extract_response_output_text(
    response_json,
):

    output_text = (
        response_json.get(
            "output_text"
        )
    )


    if (
        isinstance(
            output_text,
            str,
        )
        and output_text.strip()
    ):

        return output_text.strip()


    texts = []


    for item in response_json.get(
        "output",
        [],
    ):

        if not isinstance(
            item,
            dict,
        ):

            continue


        for content in item.get(
            "content",
            [],
        ):

            if not isinstance(
                content,
                dict,
            ):

                continue


            content_type = (
                content.get(
                    "type"
                )
            )


            if (
                isinstance(
                    content_type,
                    str,
                )
                and content_type
                in {
                    "output_text",
                    "text",
                }
            ):

                text = (
                    content.get(
                        "text"
                    )
                )

                if isinstance(
                    text,
                    str,
                ):

                    texts.append(
                        text
                    )


    return "\n".join(
        texts
    ).strip()


def extract_web_sources(
    response_json,
):

    collected = []


    def add_source(
        title,
        url,
    ):

        if not isinstance(
            url,
            str,
        ):

            return


        url = url.strip()


        if not url:

            return


        clean_title = (
            title.strip()
            if (
                isinstance(
                    title,
                    str,
                )
                and title.strip()
            )
            else url
        )


        collected.append(
            {
                "title":
                    clean_title,

                "url":
                    url,
            }
        )


    def walk(
        obj,
    ):

        if isinstance(
            obj,
            dict,
        ):

            obj_type = (
                obj.get(
                    "type"
                )
            )


            if isinstance(
                obj_type,
                str,
            ):

                if obj_type == "url_citation":

                    add_source(
                        obj.get(
                            "title"
                        ),
                        obj.get(
                            "url"
                        ),
                    )

                elif obj_type == "web_search_result":

                    add_source(
                        obj.get(
                            "title"
                        ),
                        obj.get(
                            "url"
                        ),
                    )


            nested = (
                obj.get(
                    "url_citation"
                )
            )


            if isinstance(
                nested,
                dict,
            ):

                add_source(
                    nested.get(
                        "title"
                    ),
                    nested.get(
                        "url"
                    ),
                )


            annotations = (
                obj.get(
                    "annotations"
                )
            )


            if isinstance(
                annotations,
                list,
            ):

                for annotation in annotations:

                    if not isinstance(
                        annotation,
                        dict,
                    ):

                        continue


                    if (
                        annotation.get(
                            "type"
                        )
                        == "url_citation"
                    ):

                        add_source(
                            annotation.get(
                                "title"
                            ),
                            annotation.get(
                                "url"
                            ),
                        )


                        nested_annotation = (
                            annotation.get(
                                "url_citation"
                            )
                        )


                        if isinstance(
                            nested_annotation,
                            dict,
                        ):

                            add_source(
                                nested_annotation.get(
                                    "title"
                                ),
                                nested_annotation.get(
                                    "url"
                                ),
                            )


            for value in obj.values():

                walk(
                    value
                )


        elif isinstance(
            obj,
            list,
        ):

            for value in obj:

                walk(
                    value
                )


    walk(
        response_json
    )


    unique = []

    seen_urls = set()


    for source in collected:

        url = source[
            "url"
        ]


        if url in seen_urls:

            continue


        seen_urls.add(
            url
        )

        unique.append(
            source
        )


    return unique


def call_structured_response(
    *,
    model,
    instructions,
    payload,
    schema,
    schema_name,
    use_web=False,
):

    if not OPENAI_API_KEY:

        raise RuntimeError(
            "OPENAI_API_KEY not available."
        )


    request_body = {

        "model":
            model,

        "instructions":
            instructions,

        "input":
            json_dumps_safe(
                payload,
                ensure_ascii=False,
                indent=2,
            ),

        "text": {
            "format": {
                "type":
                    "json_schema",

                "name":
                    schema_name,

                "strict":
                    True,

                "schema":
                    schema,
            }
        },
    }


    if use_web:

        request_body[
            "tools"
        ] = [
            {
                "type":
                    "web_search",
            }
        ]


    response = requests.post(

        OPENAI_RESPONSES_URL,

        headers={
            "Authorization":
                f"Bearer {OPENAI_API_KEY}",

            "Content-Type":
                "application/json",
        },

        data=json_dumps_safe(
            request_body,
            ensure_ascii=False,
        ),

        timeout=
            OPENAI_TIMEOUT_SECONDS,
    )


    if response.status_code >= 400:

        raise RuntimeError(
            f"OpenAI HTTP {response.status_code}: "
            f"{response.text[:1500]}"
        )


    response_json = (
        response.json()
    )


    output_text = (
        extract_response_output_text(
            response_json
        )
    )


    if not output_text:

        raise RuntimeError(
            "OpenAI response contained no structured output."
        )


    try:

        result = json.loads(
            output_text
        )

    except Exception as exc:

        raise RuntimeError(
            "Structured output could not be decoded as JSON. "
            f"Preview: {output_text[:1000]}"
        ) from exc


    sources = (
        extract_web_sources(
            response_json
        )
        if use_web
        else []
    )


    return (
        result,
        response_json,
        sources,
    )


# ============================================================
# 12.13 — RESULT VALIDATION
# ============================================================

VALID_RELEVANCE = {
    "Core",
    "High",
    "Moderate",
    "Peripheral",
}


def validate_common_result(
    result,
):

    if (
        result.get(
            "relevance"
        )
        not in VALID_RELEVANCE
    ):

        raise ValueError(
            "Invalid relevance."
        )


    confidence = float(
        result.get(
            "confidence"
        )
    )


    if not (
        0
        <= confidence
        <= 1
    ):

        raise ValueError(
            "Confidence outside [0, 1]."
        )


    if not isinstance(
        result.get(
            "abstain"
        ),
        bool,
    ):

        raise ValueError(
            "abstain must be boolean."
        )


    return True


def fast_result_can_be_accepted(
    result,
):

    validate_common_result(
        result
    )


    if not isinstance(
        result.get(
            "needs_web_research"
        ),
        bool,
    ):

        raise ValueError(
            "needs_web_research must be boolean."
        )


    if result.get(
        "abstain"
    ):

        return False


    if result.get(
        "needs_web_research"
    ):

        return False


    if float(
        result.get(
            "confidence",
            0,
        )
    ) < FAST_ACCEPT_CONFIDENCE:

        return False


    if not (
        isinstance(
            result.get(
                "subindustry_raw"
            ),
            str,
        )
        and result.get(
            "subindustry_raw"
        ).strip()
    ):

        return False


    return True


def validate_web_result(
    result,
):

    validate_common_result(
        result
    )


    if (
        not result.get(
            "abstain"
        )
        and not (
            isinstance(
                result.get(
                    "subindustry_raw"
                ),
                str,
            )
            and result.get(
                "subindustry_raw"
            ).strip()
        )
    ):

        raise ValueError(
            "Non-abstained web classification "
            "must contain subindustry_raw."
        )


    return True


# ============================================================
# 12.14 — LOAD STEP 11C ECONOMIC-ISSUER ALIAS LINEAGE
# ============================================================

economic_issuer_alias_map = {}


if issuer_alias_history_path.exists():

    issuer_alias_history_step12 = (
        pd.read_parquet(
            issuer_alias_history_path
        )
    )


    required_alias_cols = {
        "absorbed_economic_issuer_id",
        "canonical_economic_issuer_id",
    }


    if required_alias_cols.issubset(
        issuer_alias_history_step12.columns
    ):

        economic_issuer_alias_map = dict(
            zip(
                issuer_alias_history_step12[
                    "absorbed_economic_issuer_id"
                ]
                .astype(str),

                issuer_alias_history_step12[
                    "canonical_economic_issuer_id"
                ]
                .astype(str),
            )
        )


def canonicalise_semantic_issuer_id(
    issuer_id,
):

    if pd.isna(
        issuer_id
    ):

        return issuer_id


    issuer_id = str(
        issuer_id
    )


    seen = set()


    while (
        issuer_id
        in economic_issuer_alias_map
    ):

        if issuer_id in seen:

            raise RuntimeError(
                "Cycle detected in economic issuer alias history."
            )


        seen.add(
            issuer_id
        )


        issuer_id = (
            economic_issuer_alias_map[
                issuer_id
            ]
        )


    return issuer_id


# ============================================================
# 12.15 — LOAD + VALIDATE PERSISTED SEMANTIC RESEARCH
# ============================================================

persisted_theme_df = pd.DataFrame()


if theme_classification_path.exists():

    try:

        persisted_theme_df = (
            pd.read_parquet(
                theme_classification_path
            )
        )

    except Exception as exc:

        print(
            "WARNING: Existing theme_classification.parquet "
            f"could not be read: {exc}"
        )

        persisted_theme_df = (
            pd.DataFrame()
        )


def _nonempty_string(
    value,
):

    return bool(
        isinstance(
            value,
            str,
        )
        and value.strip()
    )


def persisted_semantic_row_is_valid(
    row,
):

    issuer_id = row.get(
        "economic_issuer_id"
    )


    if not _nonempty_string(
        str(issuer_id)
        if not pd.isna(issuer_id)
        else None
    ):

        return False


    if (
        str(
            row.get(
                "theme_classification_method",
                ""
            )
        )
        == "ERROR"
    ):

        return False


    if (
        str(
            row.get(
                "research_method",
                ""
            )
        )
        == "ERROR"
    ):

        return False


    relevance = row.get(
        "theme_relevance"
    )


    if relevance not in (
        VALID_RELEVANCE
        | {
            "Unclassified"
        }
    ):

        return False


    confidence = pd.to_numeric(
        pd.Series(
            [
                row.get(
                    "theme_confidence"
                )
            ]
        ),
        errors="coerce",
    ).iloc[0]


    if pd.isna(
        confidence
    ):

        return False


    if not (
        0
        <= float(confidence)
        <= 1
    ):

        return False


    if relevance != "Unclassified":

        if not _nonempty_string(
            row.get(
                "theme_subindustry_raw"
            )
        ):

            return False


    return True


# ============================================================
# 12.16 — REMAP OLD SEMANTIC ROWS TO CURRENT CANONICAL ISSUERS
# ============================================================

persisted_semantic_reuse = {}


if not persisted_theme_df.empty:

    persisted_theme_work = (
        persisted_theme_df.copy()
    )


    persisted_theme_work[
        "_original_economic_issuer_id"
    ] = (
        persisted_theme_work[
            "economic_issuer_id"
        ]
        .astype(str)
    )


    persisted_theme_work[
        "economic_issuer_id"
    ] = (
        persisted_theme_work[
            "economic_issuer_id"
        ]
        .map(
            canonicalise_semantic_issuer_id
        )
    )


    persisted_theme_work = (
        persisted_theme_work[
            persisted_theme_work[
                "economic_issuer_id"
            ]
            .astype(str)
            .isin(
                CURRENT_CANONICAL_ISSUER_IDS
            )
        ]
        .copy()
    )


    persisted_theme_work[
        "_valid_semantic_row"
    ] = (
        persisted_theme_work.apply(
            persisted_semantic_row_is_valid,
            axis=1,
        )
    )


    persisted_theme_work = (
        persisted_theme_work[
            persisted_theme_work[
                "_valid_semantic_row"
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Deterministic duplicate preference after issuer merges:
    #
    # 1. row already belonging to surviving issuer ID;
    # 2. non-abstained classified row;
    # 3. web-researched/repair row;
    # 4. higher confidence;
    # 5. more recent research datetime.
    # --------------------------------------------------------

    persisted_theme_work[
        "_survivor_row"
    ] = (
        persisted_theme_work[
            "_original_economic_issuer_id"
        ]
        .astype(str)
        .eq(
            persisted_theme_work[
                "economic_issuer_id"
            ]
            .astype(str)
        )
        .astype(int)
    )


    persisted_theme_work[
        "_classified_row"
    ] = (
        persisted_theme_work[
            "theme_relevance"
        ]
        .ne(
            "Unclassified"
        )
        .astype(int)
    )


    persisted_theme_work[
        "_web_row"
    ] = (
        persisted_theme_work[
            "research_method"
        ]
        .isin(
            [
                "OPENAI_WEB_RESEARCH",
                "OPENAI_WEB_RESEARCH_REPAIR",
            ]
        )
        .astype(int)
    )


    persisted_theme_work[
        "_confidence_numeric"
    ] = pd.to_numeric(
        persisted_theme_work[
            "theme_confidence"
        ],
        errors="coerce",
    ).fillna(
        -1
    )


    persisted_theme_work[
        "_research_datetime_sort"
    ] = pd.to_datetime(
        persisted_theme_work.get(
            "research_datetime"
        ),
        errors="coerce",
        utc=True,
    )


    persisted_theme_work = (
        persisted_theme_work
        .sort_values(
            [
                "economic_issuer_id",
                "_survivor_row",
                "_classified_row",
                "_web_row",
                "_confidence_numeric",
                "_research_datetime_sort",
            ],
            ascending=[
                True,
                False,
                False,
                False,
                False,
                False,
            ],
            na_position="last",
        )
    )


    persisted_best = (
        persisted_theme_work
        .drop_duplicates(
            subset=[
                "economic_issuer_id"
            ],
            keep="first",
        )
        .copy()
    )


    persisted_semantic_reuse = {
        str(row["economic_issuer_id"]):
            row.to_dict()

        for _, row
        in persisted_best.iterrows()
    }


# ============================================================
# 12.17 — AI COST PREFLIGHT
# ============================================================

issuer_preflight_rows = []


for _, issuer_row in (
    economic_issuer_master
    .iterrows()
):

    base_payload = (
        theme_classification_payload(
            issuer_row
        )
    )


    issuer_id = str(
        base_payload[
            "economic_issuer_id"
        ]
    )


    if issuer_id in persisted_semantic_reuse:

        issuer_preflight_rows.append(
            {
                "economic_issuer_id":
                    issuer_id,

                "preflight_route":
                    "PERSISTED_SEMANTIC_REUSE",

                "new_fast_call_required":
                    False,

                "known_new_web_call_required":
                    False,
            }
        )

        continue


    fast_payload = {

        **base_payload,

        "classification_model":
            FAST_CLASSIFICATION_MODEL,
    }


    fast_key = ai_cache_key(

        FAST_CLASSIFICATION_TASK_TYPE,

        FAST_CLASSIFICATION_SCHEMA_VERSION,

        FAST_CLASSIFICATION_PROMPT_VERSION,

        FAST_CLASSIFICATION_MODEL,

        fast_payload,
    )


    fast_record = (
        fast_cache.get(
            fast_key
        )
    )


    if fast_record is None:

        issuer_preflight_rows.append(
            {
                "economic_issuer_id":
                    issuer_id,

                "preflight_route":
                    "NEW_FAST_CALL",

                "new_fast_call_required":
                    True,

                "known_new_web_call_required":
                    False,
            }
        )

        continue


    try:

        fast_result = (
            fast_record[
                "result"
            ]
        )


        if fast_result_can_be_accepted(
            fast_result
        ):

            issuer_preflight_rows.append(
                {
                    "economic_issuer_id":
                        issuer_id,

                    "preflight_route":
                        "FAST_CACHE_ACCEPT",

                    "new_fast_call_required":
                        False,

                    "known_new_web_call_required":
                        False,
                }
            )

            continue


        web_payload = {

            **base_payload,

            "web_research_model":
                WEB_RESEARCH_MODEL,

            "fast_gate_result":
                fast_result,
        }


        web_key = ai_cache_key(

            WEB_CLASSIFICATION_TASK_TYPE,

            WEB_CLASSIFICATION_SCHEMA_VERSION,

            WEB_CLASSIFICATION_PROMPT_VERSION,

            WEB_RESEARCH_MODEL,

            web_payload,
        )


        web_record = (
            web_cache.get(
                web_key
            )
        )


        issuer_preflight_rows.append(
            {
                "economic_issuer_id":
                    issuer_id,

                "preflight_route":
                    (
                        "WEB_CACHE"
                        if web_record is not None
                        else "NEW_WEB_CALL"
                    ),

                "new_fast_call_required":
                    False,

                "known_new_web_call_required":
                    (
                        web_record is None
                    ),
            }
        )


    except Exception:

        issuer_preflight_rows.append(
            {
                "economic_issuer_id":
                    issuer_id,

                "preflight_route":
                    "INVALID_CACHE_NEW_FAST",

                "new_fast_call_required":
                    True,

                "known_new_web_call_required":
                    False,
            }
        )


issuer_preflight_df = pd.DataFrame(
    issuer_preflight_rows
)


persisted_reuse_count = int(
    issuer_preflight_df[
        "preflight_route"
    ]
    .eq(
        "PERSISTED_SEMANTIC_REUSE"
    )
    .sum()
)


new_fast_preflight_count = int(
    issuer_preflight_df[
        "new_fast_call_required"
    ]
    .sum()
)


known_new_web_preflight_count = int(
    issuer_preflight_df[
        "known_new_web_call_required"
    ]
    .sum()
)


cached_without_new_ai_count = int(
    len(
        issuer_preflight_df
    )
    - persisted_reuse_count
    - new_fast_preflight_count
    - known_new_web_preflight_count
)


print("=" * 80)
print("STEP 12 — AI COST PREFLIGHT")
print("=" * 80)

print(
    f"Research theme: "
    f"{research_theme}"
)

print(
    f"Canonical economic issuers: "
    f"{len(economic_issuer_master):,}"
)

print(
    f"Persisted semantic classifications reusable: "
    f"{persisted_reuse_count:,}"
)

print(
    f"Additional exact-cache routes requiring no new AI: "
    f"{cached_without_new_ai_count:,}"
)

print(
    f"Genuinely new fast-model calls required: "
    f"{new_fast_preflight_count:,}"
)

print(
    f"Known genuinely new web calls required: "
    f"{known_new_web_preflight_count:,}"
)

print(
    f"New-fast-call safety limit: "
    f"{MAX_NEW_FAST_CALLS_SAFETY:,}"
)

print("=" * 80)


display(
    issuer_preflight_df[
        "preflight_route"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "preflight_route"
    )
    .reset_index(
        name="issuer_count"
    )
)


if (
    new_fast_preflight_count
    > MAX_NEW_FAST_CALLS_SAFETY
):

    raise RuntimeError(
        "STEP 12 COST-SAFETY STOP.\n\n"
        f"The preflight found "
        f"{new_fast_preflight_count:,} genuinely new fast-model "
        "calls, exceeding the configured safety limit of "
        f"{MAX_NEW_FAST_CALLS_SAFETY:,}.\n\n"
        "This usually indicates that persisted semantic research "
        "or exact caches are not being reused as expected. "
        "No new issuer-level AI classification loop has run."
    )


print()

print(
    "Cost-safety preflight: PASS"
)

print()


# ============================================================
# 12.18 — CLASSIFICATION LOOP
# ============================================================

theme_rows = []


total_issuers = len(
    economic_issuer_master
)


step12_started = (
    time.time()
)


persisted_reuse_runtime_count = 0

fast_new_count = 0

fast_cache_count = 0

fast_accepted_count = 0

web_escalation_count = 0

web_new_count = 0

web_cache_count = 0

abstain_count = 0

unclassified_count = 0

error_count = 0

research_source_count = 0


consecutive_error_count = 0

last_error_signature = None


print("=" * 80)

print(
    "STEP 12 — GATED AI THEME CLASSIFICATION"
)

print("=" * 80)

print(
    f"Research theme: "
    f"{research_theme}"
)

print(
    f"Economic issuers: "
    f"{total_issuers:,}"
)

print(
    f"Fast no-web model: "
    f"{FAST_CLASSIFICATION_MODEL}"
)

print(
    f"Web escalation model: "
    f"{WEB_RESEARCH_MODEL}"
)

print(
    f"Persisted semantic reuse available: "
    f"{persisted_reuse_count:,}"
)

print("=" * 80)


for i, (_, issuer_row) in enumerate(
    economic_issuer_master.iterrows(),
    start=1,
):

    issuer_started = (
        time.time()
    )


    base_payload = (
        theme_classification_payload(
            issuer_row
        )
    )


    issuer_id = str(
        base_payload[
            "economic_issuer_id"
        ]
    )


    issuer_name = (
        base_payload.get(
            "issuer_name"
        )
        or base_payload.get(
            "legal_name"
        )
        or issuer_id
    )


    # ========================================================
    # LEVEL 1 — REUSE PERSISTED VALIDATED SEMANTIC RESEARCH
    # ========================================================

    if issuer_id in persisted_semantic_reuse:

        existing = (
            persisted_semantic_reuse[
                issuer_id
            ]
        )


        row_out = {

            "economic_issuer_id":
                issuer_id,

            "research_theme":
                research_theme,

            "theme_relevance":
                existing.get(
                    "theme_relevance"
                ),

            "theme_subindustry_raw":
                existing.get(
                    "theme_subindustry_raw"
                ),

            # 12C reapplies canonical mapping.
            "theme_subindustry":
                None,

            "theme_confidence":
                float(
                    existing.get(
                        "theme_confidence",
                        0.0,
                    )
                ),

            "business_description":
                existing.get(
                    "business_description"
                ),

            "classification_reason":
                existing.get(
                    "classification_reason"
                ),

            "theme_classification_method":
                existing.get(
                    "theme_classification_method"
                ),

            "research_method":
                existing.get(
                    "research_method"
                ),

            "research_model":
                existing.get(
                    "research_model"
                ),

            "web_research_used":
                bool(
                    existing.get(
                        "web_research_used",
                        False,
                    )
                ),

            "research_sources_json":
                existing.get(
                    "research_sources_json",
                    "[]",
                ),

            "research_source_count":
                int(
                    existing.get(
                        "research_source_count",
                        0,
                    )
                    or 0
                ),

            "research_datetime":
                existing.get(
                    "research_datetime"
                ),

            "ai_assisted":
                bool(
                    existing.get(
                        "ai_assisted",
                        True,
                    )
                ),

            "ai_task_id":
                existing.get(
                    "ai_task_id"
                ),

            "abstained":
                bool(
                    existing.get(
                        "abstained",
                        False,
                    )
                ),

            "abstain_reason":
                existing.get(
                    "abstain_reason"
                ),

            "classification_error":
                None,

            "semantic_reuse_method":
                "PERSISTED_VALIDATED_REUSE",

            "semantic_reuse_original_issuer_id":
                existing.get(
                    "_original_economic_issuer_id"
                ),
        }


        theme_rows.append(
            row_out
        )


        persisted_reuse_runtime_count += 1


        if (
            row_out[
                "theme_relevance"
            ]
            == "Unclassified"
        ):

            unclassified_count += 1


        if row_out[
            "abstained"
        ]:

            abstain_count += 1


        issuer_runtime = (
            time.time()
            - issuer_started
        )


        elapsed = (
            time.time()
            - step12_started
        )


        pct = (
            100
            * i
            / total_issuers
        )


        print(
            f"[{time.strftime('%H:%M:%S')}] "
            f"✓ {i}/{total_issuers} "
            f"({pct:.1f}%) | "
            f"{issuer_name[:60]}"
        )


        print(
            "    Route: PERSISTED REUSE | "
            f"Method: "
            f"{row_out['theme_classification_method']} | "
            f"Relevance: "
            f"{row_out['theme_relevance']}"
        )


        print(
            f"    Subindustry: "
            f"{row_out['theme_subindustry_raw'] or '—'} | "
            f"Runtime: {issuer_runtime:.2f}s"
        )


        print(
            "-" * 80
        )


        continue


    # ========================================================
    # LEVEL 2 / 3 — EXACT CACHE OR NEW AI
    # ========================================================

    result = None

    research_sources = []

    response_id = None

    classification_method = None

    research_method = None

    research_model = None

    error_message = None

    web_used = False


    try:

        # ====================================================
        # STAGE A — FAST, NO WEB
        # ====================================================

        fast_payload = {

            **base_payload,

            "classification_model":
                FAST_CLASSIFICATION_MODEL,
        }


        fast_key = ai_cache_key(

            FAST_CLASSIFICATION_TASK_TYPE,

            FAST_CLASSIFICATION_SCHEMA_VERSION,

            FAST_CLASSIFICATION_PROMPT_VERSION,

            FAST_CLASSIFICATION_MODEL,

            fast_payload,
        )


        fast_record = (
            fast_cache.get(
                fast_key
            )
        )


        if fast_record is not None:

            fast_result = (
                fast_record[
                    "result"
                ]
            )

            fast_cache_count += 1


        else:

            (
                fast_result,
                fast_response,
                _,
            ) = call_structured_response(

                model=
                    FAST_CLASSIFICATION_MODEL,

                instructions=
                    FAST_SYSTEM_PROMPT,

                payload=
                    fast_payload,

                schema=
                    FAST_RESPONSE_SCHEMA,

                schema_name=
                    "theme_fast_gate",

                use_web=False,
            )


            fast_new_count += 1


            fast_record = {

                "cache_key":
                    fast_key,

                "task_type":
                    FAST_CLASSIFICATION_TASK_TYPE,

                "schema_version":
                    FAST_CLASSIFICATION_SCHEMA_VERSION,

                "prompt_version":
                    FAST_CLASSIFICATION_PROMPT_VERSION,

                "model":
                    FAST_CLASSIFICATION_MODEL,

                "created_datetime":
                    pd.Timestamp.utcnow()
                    .isoformat(),

                "payload":
                    fast_payload,

                "result":
                    fast_result,

                "openai_response_id":
                    fast_response.get(
                        "id"
                    ),
            }


            append_ai_cache_record(
                fast_cache_path,
                fast_record,
            )


            fast_cache[
                fast_key
            ] = fast_record


        if fast_result_can_be_accepted(
            fast_result
        ):

            result = (
                fast_result
            )


            fast_accepted_count += 1


            classification_method = (
                "AI_FAST_VALIDATED"
            )


            research_method = (
                "OPENAI_FAST_NO_WEB"
            )


            research_model = (
                FAST_CLASSIFICATION_MODEL
            )


            response_id = (
                fast_record.get(
                    "openai_response_id"
                )
                or fast_key
            )


        else:

            # =================================================
            # STAGE B — WEB ESCALATION
            # =================================================

            web_used = True

            web_escalation_count += 1


            web_payload = {

                **base_payload,

                "web_research_model":
                    WEB_RESEARCH_MODEL,

                "fast_gate_result":
                    fast_result,
            }


            web_key = ai_cache_key(

                WEB_CLASSIFICATION_TASK_TYPE,

                WEB_CLASSIFICATION_SCHEMA_VERSION,

                WEB_CLASSIFICATION_PROMPT_VERSION,

                WEB_RESEARCH_MODEL,

                web_payload,
            )


            web_record = (
                web_cache.get(
                    web_key
                )
            )


            if web_record is not None:

                result = (
                    web_record[
                        "result"
                    ]
                )


                research_sources = (
                    web_record.get(
                        "research_sources",
                        [],
                    )
                )


                web_cache_count += 1


            else:

                (
                    result,
                    web_response,
                    research_sources,
                ) = call_structured_response(

                    model=
                        WEB_RESEARCH_MODEL,

                    instructions=
                        WEB_SYSTEM_PROMPT,

                    payload=
                        web_payload,

                    schema=
                        WEB_RESPONSE_SCHEMA,

                    schema_name=
                        "theme_web_classification",

                    use_web=True,
                )


                web_new_count += 1


                web_record = {

                    "cache_key":
                        web_key,

                    "task_type":
                        WEB_CLASSIFICATION_TASK_TYPE,

                    "schema_version":
                        WEB_CLASSIFICATION_SCHEMA_VERSION,

                    "prompt_version":
                        WEB_CLASSIFICATION_PROMPT_VERSION,

                    "model":
                        WEB_RESEARCH_MODEL,

                    "created_datetime":
                        pd.Timestamp.utcnow()
                        .isoformat(),

                    "payload":
                        web_payload,

                    "result":
                        result,

                    "research_sources":
                        research_sources,

                    "openai_response_id":
                        web_response.get(
                            "id"
                        ),
                }


                append_ai_cache_record(
                    web_cache_path,
                    web_record,
                )


                web_cache[
                    web_key
                ] = web_record


            validate_web_result(
                result
            )


            classification_method = (
                "AI_WEB_PROPOSED_VALIDATED"
            )


            research_method = (
                "OPENAI_WEB_RESEARCH"
            )


            research_model = (
                WEB_RESEARCH_MODEL
            )


            response_id = (
                web_record.get(
                    "openai_response_id"
                )
                or web_key
            )


            research_source_count += (
                len(
                    research_sources
                )
            )


        consecutive_error_count = 0

        last_error_signature = None


    except Exception as exc:

        error_message = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )


        error_signature = (
            error_message[:500]
        )


        error_count += 1


        if (
            error_signature
            == last_error_signature
        ):

            consecutive_error_count += 1

        else:

            consecutive_error_count = 1

            last_error_signature = (
                error_signature
            )


        result = {

            "relevance":
                "Peripheral",

            "subindustry_raw":
                None,

            "confidence":
                0.0,

            "business_description":
                None,

            "materiality_reasoning":
                None,

            "abstain":
                True,

            "abstain_reason":
                error_message,
        }


        research_sources = []

        classification_method = (
            "ERROR"
        )

        research_method = (
            "ERROR"
        )

        research_model = (
            WEB_RESEARCH_MODEL
            if web_used
            else FAST_CLASSIFICATION_MODEL
        )


        if (
            consecutive_error_count
            >= MAX_CONSECUTIVE_IDENTICAL_ERRORS
        ):

            raise RuntimeError(
                "Step 12 stopped after "
                f"{MAX_CONSECUTIVE_IDENTICAL_ERRORS} "
                "consecutive identical errors.\n"
                f"Repeated error: {error_message}"
            ) from exc


    abstained = bool(
        result.get(
            "abstain"
        )
    )


    if abstained:

        abstain_count += 1

        unclassified_count += 1

        relevance = (
            "Unclassified"
        )

        subindustry_raw = (
            None
        )


        if (
            classification_method
            != "ERROR"
        ):

            classification_method = (
                "AI_WEB_ABSTAINED"
                if web_used
                else "AI_FAST_ABSTAINED"
            )


    else:

        relevance = (
            result[
                "relevance"
            ]
        )

        subindustry_raw = (
            result.get(
                "subindustry_raw"
            )
        )


    theme_rows.append(
        {
            "economic_issuer_id":
                issuer_id,

            "research_theme":
                research_theme,

            "theme_relevance":
                relevance,

            "theme_subindustry_raw":
                subindustry_raw,

            "theme_subindustry":
                None,

            "theme_confidence":
                float(
                    result.get(
                        "confidence",
                        0.0,
                    )
                ),

            "business_description":
                result.get(
                    "business_description"
                ),

            "classification_reason":
                result.get(
                    "materiality_reasoning"
                ),

            "theme_classification_method":
                classification_method,

            "research_method":
                research_method,

            "research_model":
                research_model,

            "web_research_used":
                bool(
                    web_used
                ),

            "research_sources_json":
                json_dumps_safe(
                    research_sources,
                    ensure_ascii=False,
                ),

            "research_source_count":
                len(
                    research_sources
                ),

            "research_datetime":
                pd.Timestamp.utcnow(),

            "ai_assisted":
                (
                    classification_method
                    != "ERROR"
                ),

            "ai_task_id":
                response_id,

            "abstained":
                abstained,

            "abstain_reason":
                result.get(
                    "abstain_reason"
                ),

            "classification_error":
                error_message,

            "semantic_reuse_method":
                (
                    "EXACT_AI_CACHE"
                    if (
                        fast_record is not None
                        and fast_key in fast_cache
                        and fast_new_count == 0
                    )
                    else "AI_PIPELINE"
                ),

            "semantic_reuse_original_issuer_id":
                issuer_id,
        }
    )


    issuer_runtime = (
        time.time()
        - issuer_started
    )


    elapsed = (
        time.time()
        - step12_started
    )


    avg_runtime = (
        elapsed
        / i
    )


    remaining = (
        total_issuers
        - i
    )


    eta_seconds = (
        avg_runtime
        * remaining
    )


    pct = (
        100
        * i
        / total_issuers
    )


    route_label = (
        "WEB"
        if web_used
        else "FAST"
    )


    print(
        f"[{time.strftime('%H:%M:%S')}] "
        f"✓ {i}/{total_issuers} "
        f"({pct:.1f}%) | "
        f"{issuer_name[:60]}"
    )


    print(
        f"    Route: {route_label} | "
        f"Method: {classification_method} | "
        f"Relevance: {relevance}"
    )


    print(
        f"    Subindustry: "
        f"{subindustry_raw or '—'}"
    )


    print(
        f"    Runtime: "
        f"{issuer_runtime:.1f}s | "
        f"Elapsed: "
        f"{elapsed/60:.1f}m | "
        f"ETA: "
        f"~{eta_seconds/60:.1f}m"
    )


    print(
        "-" * 80
    )


# ============================================================
# 12.19 — CLASSIFICATION OUTPUT + QC
# ============================================================

theme_classification_df = (
    pd.DataFrame(
        theme_rows
    )
)


if (
    len(
        theme_classification_df
    )
    != len(
        economic_issuer_master
    )
):

    raise RuntimeError(
        "Step 12 row count does not equal canonical issuer count."
    )


if theme_classification_df[
    "economic_issuer_id"
].duplicated().any():

    raise RuntimeError(
        "Step 12 produced duplicate canonical economic issuers."
    )


theme_ids = set(
    theme_classification_df[
        "economic_issuer_id"
    ]
    .astype(str)
)


if (
    theme_ids
    != CURRENT_CANONICAL_ISSUER_IDS
):

    missing = (
        CURRENT_CANONICAL_ISSUER_IDS
        - theme_ids
    )

    unexpected = (
        theme_ids
        - CURRENT_CANONICAL_ISSUER_IDS
    )

    raise RuntimeError(
        "Step 12 issuer coverage mismatch.\n"
        f"Missing: {sorted(missing)[:20]}\n"
        f"Unexpected: {sorted(unexpected)[:20]}"
    )


step12_runtime = (
    time.time()
    - step12_started
)


print()

print("=" * 80)

print(
    "STEP 12 — GATED AI CLASSIFICATION COMPLETE"
)

print("=" * 80)

print(
    f"Canonical issuers: "
    f"{len(theme_classification_df):,}"
)

print(
    f"Persisted semantic rows reused: "
    f"{persisted_reuse_runtime_count:,}"
)

print(
    f"Fast new calls: "
    f"{fast_new_count:,}"
)

print(
    f"Fast cache hits: "
    f"{fast_cache_count:,}"
)

print(
    f"Accepted without web: "
    f"{fast_accepted_count:,}"
)

print(
    f"Web escalations: "
    f"{web_escalation_count:,}"
)

print(
    f"New web calls: "
    f"{web_new_count:,}"
)

print(
    f"Web cache hits: "
    f"{web_cache_count:,}"
)

print(
    f"Abstentions: "
    f"{abstain_count:,}"
)

print(
    f"Errors: "
    f"{error_count:,}"
)

print(
    f"Runtime: "
    f"{step12_runtime/60:.1f} minutes"
)

print("=" * 80)


display(
    theme_classification_df[
        "theme_relevance"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "theme_relevance"
    )
    .reset_index(
        name="issuer_count"
    )
)


print()


display(
    theme_classification_df[
        "research_method"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "research_method"
    )
    .reset_index(
        name="issuer_count"
    )
)


theme_classification_df.to_parquet(
    theme_classification_pre_harmonisation_path,
    index=False,
)


# ============================================================
# 12B. TARGETED CLASSIFICATION ERROR REPAIR
# ============================================================

ERROR_REPAIR_MODEL = (
    WEB_RESEARCH_MODEL
)


MAX_REPAIR_ATTEMPTS = 5

INITIAL_RETRY_WAIT_SECONDS = 10


error_mask = (
    theme_classification_df[
        "theme_classification_method"
    ]
    .eq(
        "ERROR"
    )
)


error_rows = (
    theme_classification_df.loc[
        error_mask
    ]
    .copy()
)


print()

print("=" * 80)

print(
    "STEP 12B — TARGETED CLASSIFICATION ERROR REPAIR"
)

print("=" * 80)

print(
    f"Rows requiring repair: "
    f"{len(error_rows):,}"
)

print(
    f"Repair model: "
    f"{ERROR_REPAIR_MODEL}"
)

print("=" * 80)


ERROR_REPAIR_SYSTEM_PROMPT = f"""
You are repairing a failed semantic classification in a global
point-in-time equity research database.

Configured research theme:

{research_theme}

Use public web research to establish what the issuer actually does
and determine its MATERIAL ECONOMIC EXPOSURE to the configured theme.

This is not identity resolution.

Never alter canonical identity or identifiers.

Core:
Principal business or defining dominant segment.

High:
Substantial and economically important component.

Moderate:
Genuine and economically meaningful but secondary.

Peripheral:
Genuine but limited.

If abstain = false:
subindustry_raw MUST be a non-empty string.

If reliable classification is not possible:
abstain = true
subindustry_raw = null

Return only structured output.
""".strip()


def call_repair_with_retry(
    payload,
):

    wait_seconds = (
        INITIAL_RETRY_WAIT_SECONDS
    )


    for attempt in range(
        1,
        MAX_REPAIR_ATTEMPTS + 1,
    ):

        try:

            return call_structured_response(

                model=
                    ERROR_REPAIR_MODEL,

                instructions=
                    ERROR_REPAIR_SYSTEM_PROMPT,

                payload=
                    payload,

                schema=
                    WEB_RESPONSE_SCHEMA,

                schema_name=
                    "theme_error_repair",

                use_web=True,
            )


        except Exception as exc:

            message = str(
                exc
            )


            retryable = any(
                code in message

                for code in [
                    "HTTP 429",
                    "HTTP 500",
                    "HTTP 502",
                    "HTTP 503",
                    "HTTP 504",
                ]
            )


            if (
                not retryable
                or attempt
                >= MAX_REPAIR_ATTEMPTS
            ):

                raise


            print(
                f"Retryable error on attempt "
                f"{attempt}/{MAX_REPAIR_ATTEMPTS}. "
                f"Waiting {wait_seconds}s..."
            )


            time.sleep(
                wait_seconds
            )


            wait_seconds *= 2


repair_success_count = 0

repair_failure_count = 0


for repair_i, (_, failed_row) in enumerate(
    error_rows.iterrows(),
    start=1,
):

    issuer_id = str(
        failed_row[
            "economic_issuer_id"
        ]
    )


    issuer_match = (
        economic_issuer_master[
            economic_issuer_master[
                "economic_issuer_id"
            ]
            .astype(str)
            .eq(
                issuer_id
            )
        ]
    )


    if issuer_match.empty:

        raise RuntimeError(
            f"Could not locate issuer {issuer_id}."
        )


    issuer_row = (
        issuer_match.iloc[
            0
        ]
    )


    issuer_name = (
        issuer_row.get(
            "issuer_name"
        )
        or issuer_id
    )


    payload = (
        theme_classification_payload(
            issuer_row
        )
    )


    payload[
        "repair_model"
    ] = (
        ERROR_REPAIR_MODEL
    )


    payload[
        "previous_error"
    ] = (
        failed_row.get(
            "classification_error"
        )
    )


    repair_key = ai_cache_key(

        ERROR_REPAIR_TASK_TYPE,

        ERROR_REPAIR_SCHEMA_VERSION,

        ERROR_REPAIR_PROMPT_VERSION,

        ERROR_REPAIR_MODEL,

        payload,
    )


    print(
        f"[{repair_i}/{len(error_rows)}] "
        f"Repairing: {issuer_name}"
    )


    try:

        cached_repair = (
            web_cache.get(
                repair_key
            )
        )


        if cached_repair is not None:

            result = (
                cached_repair[
                    "result"
                ]
            )

            research_sources = (
                cached_repair.get(
                    "research_sources",
                    [],
                )
            )

            response_id = (
                cached_repair.get(
                    "openai_response_id"
                )
                or repair_key
            )


        else:

            (
                result,
                response_json,
                research_sources,
            ) = call_repair_with_retry(
                payload
            )


            response_id = (
                response_json.get(
                    "id"
                )
            )


            repair_record = {

                "cache_key":
                    repair_key,

                "task_type":
                    ERROR_REPAIR_TASK_TYPE,

                "schema_version":
                    ERROR_REPAIR_SCHEMA_VERSION,

                "prompt_version":
                    ERROR_REPAIR_PROMPT_VERSION,

                "model":
                    ERROR_REPAIR_MODEL,

                "created_datetime":
                    pd.Timestamp.utcnow()
                    .isoformat(),

                "payload":
                    payload,

                "result":
                    result,

                "research_sources":
                    research_sources,

                "openai_response_id":
                    response_id,
            }


            append_ai_cache_record(
                web_cache_path,
                repair_record,
            )


            web_cache[
                repair_key
            ] = repair_record


        validate_web_result(
            result
        )


        abstained = bool(
            result.get(
                "abstain"
            )
        )


        if abstained:

            relevance = (
                "Unclassified"
            )

            subindustry_raw = (
                None
            )

            repaired_method = (
                "AI_WEB_REPAIR_ABSTAINED"
            )

        else:

            relevance = (
                result[
                    "relevance"
                ]
            )

            subindustry_raw = (
                result[
                    "subindustry_raw"
                ]
            )

            repaired_method = (
                "AI_WEB_REPAIR_VALIDATED"
            )


        target_mask = (
            theme_classification_df[
                "economic_issuer_id"
            ]
            .astype(str)
            .eq(
                issuer_id
            )
        )


        theme_classification_df.loc[
            target_mask,
            "theme_relevance"
        ] = relevance


        theme_classification_df.loc[
            target_mask,
            "theme_subindustry_raw"
        ] = subindustry_raw


        theme_classification_df.loc[
            target_mask,
            "theme_subindustry"
        ] = None


        theme_classification_df.loc[
            target_mask,
            "theme_confidence"
        ] = float(
            result.get(
                "confidence",
                0.0,
            )
        )


        theme_classification_df.loc[
            target_mask,
            "business_description"
        ] = result.get(
            "business_description"
        )


        theme_classification_df.loc[
            target_mask,
            "classification_reason"
        ] = result.get(
            "materiality_reasoning"
        )


        theme_classification_df.loc[
            target_mask,
            "theme_classification_method"
        ] = repaired_method


        theme_classification_df.loc[
            target_mask,
            "research_method"
        ] = (
            "OPENAI_WEB_RESEARCH_REPAIR"
        )


        theme_classification_df.loc[
            target_mask,
            "research_model"
        ] = (
            ERROR_REPAIR_MODEL
        )


        theme_classification_df.loc[
            target_mask,
            "web_research_used"
        ] = True


        theme_classification_df.loc[
            target_mask,
            "research_sources_json"
        ] = json_dumps_safe(
            research_sources,
            ensure_ascii=False,
        )


        theme_classification_df.loc[
            target_mask,
            "research_source_count"
        ] = len(
            research_sources
        )


        theme_classification_df.loc[
            target_mask,
            "research_datetime"
        ] = pd.Timestamp.utcnow()


        theme_classification_df.loc[
            target_mask,
            "ai_assisted"
        ] = True


        theme_classification_df.loc[
            target_mask,
            "ai_task_id"
        ] = (
            response_id
            or repair_key
        )


        theme_classification_df.loc[
            target_mask,
            "abstained"
        ] = abstained


        theme_classification_df.loc[
            target_mask,
            "abstain_reason"
        ] = result.get(
            "abstain_reason"
        )


        theme_classification_df.loc[
            target_mask,
            "classification_error"
        ] = None


        theme_classification_df.loc[
            target_mask,
            "semantic_reuse_method"
        ] = (
            "ERROR_REPAIR"
        )


        repair_success_count += 1


        print(
            f"    ✓ {repaired_method} | "
            f"{relevance} | "
            f"{subindustry_raw or '—'}"
        )


    except Exception as exc:

        repair_failure_count += 1


        print(
            f"    ✗ FAILED: "
            f"{type(exc).__name__}: "
            f"{exc}"
        )


remaining_error_mask = (
    theme_classification_df[
        "theme_classification_method"
    ]
    .eq(
        "ERROR"
    )
)


remaining_error_count = int(
    remaining_error_mask.sum()
)


print()

print("=" * 80)

print(
    "STEP 12B — TARGETED ERROR REPAIR COMPLETE"
)

print("=" * 80)

print(
    f"Rows attempted: "
    f"{len(error_rows):,}"
)

print(
    f"Successfully repaired: "
    f"{repair_success_count:,}"
)

print(
    f"Repair failures: "
    f"{repair_failure_count:,}"
)

print(
    f"Remaining ERROR rows: "
    f"{remaining_error_count:,}"
)

print("=" * 80)


if remaining_error_count > 0:

    display(
        theme_classification_df.loc[
            remaining_error_mask,
            [
                "economic_issuer_id",
                "research_model",
                "classification_error",
                "abstain_reason",
            ],
        ]
    )


    raise RuntimeError(
        f"Step 12B finished with "
        f"{remaining_error_count:,} unresolved ERROR rows."
    )


theme_classification_df.to_parquet(
    theme_classification_pre_harmonisation_path,
    index=False,
)


print()

print(
    "Step 12B integrity: PASS"
)


# ============================================================
# 12C. THEME SUBINDUSTRY HARMONISATION
#     + PERSISTED TAXONOMY REUSE
# ============================================================

classified = (
    theme_classification_df[
        (
            theme_classification_df[
                "theme_relevance"
            ]
            != "Unclassified"
        )
        &
        (
            theme_classification_df[
                "theme_subindustry_raw"
            ]
            .notna()
        )
    ]
    .copy()
)


raw_counts = (
    classified[
        "theme_subindustry_raw"
    ]
    .value_counts()
    .rename_axis(
        "raw_subindustry"
    )
    .reset_index(
        name="issuer_count"
    )
)


raw_labels = (
    raw_counts[
        "raw_subindustry"
    ]
    .astype(str)
    .tolist()
)


if not raw_labels:

    raise RuntimeError(
        "Step 12C cannot run because no classified "
        "raw subindustry labels exist."
    )


# ============================================================
# 12C.1 — LOAD EXISTING TAXONOMY REGISTRY
# ============================================================

existing_taxonomy_df = pd.DataFrame()


if theme_subindustry_taxonomy_path.exists():

    try:

        existing_taxonomy_df = (
            pd.read_parquet(
                theme_subindustry_taxonomy_path
            )
        )

    except Exception as exc:

        print(
            "WARNING: Existing taxonomy registry "
            f"could not be read: {exc}"
        )

        existing_taxonomy_df = (
            pd.DataFrame()
        )


existing_raw_to_canonical = {}

existing_mapping_confidence = {}


if not existing_taxonomy_df.empty:

    required_existing_cols = {
        "raw_subindustry",
        "canonical_subindustry",
    }


    if required_existing_cols.issubset(
        existing_taxonomy_df.columns
    ):

        taxonomy_existing_work = (
            existing_taxonomy_df.copy()
        )


        if (
            "research_theme"
            in taxonomy_existing_work.columns
        ):

            taxonomy_existing_work = (
                taxonomy_existing_work[
                    taxonomy_existing_work[
                        "research_theme"
                    ]
                    .astype(str)
                    .eq(
                        research_theme
                    )
                ]
                .copy()
            )


        # Ensure one stable target per raw label.
        conflicting_existing = (
            taxonomy_existing_work
            .groupby(
                "raw_subindustry"
            )[
                "canonical_subindustry"
            ]
            .nunique(
                dropna=True
            )
        )


        conflicting_existing = (
            conflicting_existing[
                conflicting_existing
                > 1
            ]
        )


        if len(
            conflicting_existing
        ) > 0:

            raise RuntimeError(
                "Existing taxonomy contains raw labels "
                "mapped to multiple canonical categories."
            )


        taxonomy_existing_work = (
            taxonomy_existing_work
            .drop_duplicates(
                subset=[
                    "raw_subindustry"
                ],
                keep="last",
            )
        )


        existing_raw_to_canonical = dict(
            zip(
                taxonomy_existing_work[
                    "raw_subindustry"
                ]
                .astype(str),

                taxonomy_existing_work[
                    "canonical_subindustry"
                ]
                .astype(str),
            )
        )


        if (
            "mapping_confidence"
            in taxonomy_existing_work.columns
        ):

            existing_mapping_confidence = dict(
                zip(
                    taxonomy_existing_work[
                        "raw_subindustry"
                    ]
                    .astype(str),

                    pd.to_numeric(
                        taxonomy_existing_work[
                            "mapping_confidence"
                        ],
                        errors="coerce",
                    ),
                )
            )


current_raw_set = set(
    raw_labels
)


existing_reusable_raw = (
    current_raw_set
    & set(
        existing_raw_to_canonical
    )
)


new_raw_labels = sorted(
    current_raw_set
    - set(
        existing_raw_to_canonical
    )
)


existing_canonical_categories = sorted(
    {
        existing_raw_to_canonical[
            raw
        ]

        for raw
        in existing_reusable_raw
    }
)


print()

print("=" * 80)

print(
    "STEP 12C — THEME SUBINDUSTRY HARMONISATION"
)

print("=" * 80)

print(
    f"Research theme: "
    f"{research_theme}"
)

print(
    f"Classified issuers: "
    f"{len(classified):,}"
)

print(
    f"Unique current raw subindustries: "
    f"{len(raw_labels):,}"
)

print(
    f"Existing raw mappings reusable: "
    f"{len(existing_reusable_raw):,}"
)

print(
    f"Genuinely new raw labels: "
    f"{len(new_raw_labels):,}"
)

print(
    f"Existing canonical categories represented: "
    f"{len(existing_canonical_categories):,}"
)

print("=" * 80)


# ============================================================
# 12C.2 — INCREMENTAL TAXONOMY SCHEMA
# ============================================================

INCREMENTAL_TAXONOMY_RESPONSE_SCHEMA = {

    "type": "object",

    "additionalProperties": False,

    "properties": {

        "new_canonical_categories": {

            "type": "array",

            "items": {

                "type": "object",

                "additionalProperties": False,

                "properties": {

                    "canonical_subindustry": {
                        "type": "string",
                    },

                    "description": {
                        "type": "string",
                    },
                },

                "required": [
                    "canonical_subindustry",
                    "description",
                ],
            },
        },


        "mappings": {

            "type": "array",

            "items": {

                "type": "object",

                "additionalProperties": False,

                "properties": {

                    "raw_subindustry": {
                        "type": "string",
                    },

                    "canonical_subindustry": {
                        "type": "string",
                    },

                    "mapping_confidence": {
                        "type": "number",
                        "minimum": 0,
                        "maximum": 1,
                    },
                },

                "required": [
                    "raw_subindustry",
                    "canonical_subindustry",
                    "mapping_confidence",
                ],
            },
        },
    },

    "required": [
        "new_canonical_categories",
        "mappings",
    ],
}


INCREMENTAL_TAXONOMY_SYSTEM_PROMPT = f"""
You are the semantic taxonomy-harmonisation layer of a global
point-in-time equity research database.

Configured research theme:

{research_theme}

You will receive:
1. the existing canonical subindustry category names already used
   successfully for this research theme; and
2. only genuinely NEW raw subindustry labels that do not yet have
   persisted mappings.

Your task is incremental harmonisation.

RULES

1. Prefer mapping a new raw label into an existing canonical category
   whenever the economic activity is genuinely equivalent.

2. Create a new canonical category only when the new raw label
   represents an economically meaningful activity not adequately
   represented by existing categories.

3. Do not create a new category merely because wording differs.

4. Preserve economically meaningful distinctions.

5. Every supplied new raw label must map exactly once.

6. Do not modify or delete existing canonical categories.

7. Do not alter issuer identity, theme relevance or research evidence.

8. Do not use a hardcoded external industry taxonomy.

Return only the requested structured output.
""".strip()


# ============================================================
# 12C.3 — HARMONISE ONLY GENUINELY NEW RAW LABELS
# ============================================================

incremental_mapping_df = pd.DataFrame(
    columns=[
        "raw_subindustry",
        "canonical_subindustry",
        "mapping_confidence",
    ]
)


incremental_new_categories_df = pd.DataFrame(
    columns=[
        "canonical_subindustry",
        "description",
    ]
)


taxonomy_ai_method = (
    "PERSISTED_REUSE_ONLY"
)


if new_raw_labels:

    new_raw_counts = (
        raw_counts[
            raw_counts[
                "raw_subindustry"
            ]
            .astype(str)
            .isin(
                new_raw_labels
            )
        ]
        .copy()
    )


    incremental_payload = {

        "research_theme":
            research_theme,

        "existing_canonical_categories":
            existing_canonical_categories,

        "new_raw_subindustries":
            new_raw_counts.to_dict(
                orient="records"
            ),
    }


    incremental_taxonomy_key = (
        ai_cache_key(

            SUBINDUSTRY_HARMONISATION_TASK_TYPE,

            SUBINDUSTRY_HARMONISATION_SCHEMA_VERSION,

            SUBINDUSTRY_HARMONISATION_PROMPT_VERSION,

            TAXONOMY_HARMONISATION_MODEL,

            incremental_payload,
        )
    )


    incremental_record = (
        taxonomy_cache.get(
            incremental_taxonomy_key
        )
    )


    if incremental_record is not None:

        incremental_result = (
            incremental_record[
                "result"
            ]
        )

        taxonomy_ai_method = (
            "AI_INCREMENTAL_CACHE"
        )


        print(
            "Using cached incremental taxonomy result."
        )


    else:

        (
            incremental_result,
            incremental_response,
            _,
        ) = call_structured_response(

            model=
                TAXONOMY_HARMONISATION_MODEL,

            instructions=
                INCREMENTAL_TAXONOMY_SYSTEM_PROMPT,

            payload=
                incremental_payload,

            schema=
                INCREMENTAL_TAXONOMY_RESPONSE_SCHEMA,

            schema_name=
                "theme_subindustry_taxonomy_incremental",

            use_web=False,
        )


        taxonomy_ai_method = (
            "AI_INCREMENTAL_NEW"
        )


        incremental_record = {

            "cache_key":
                incremental_taxonomy_key,

            "task_type":
                SUBINDUSTRY_HARMONISATION_TASK_TYPE,

            "schema_version":
                SUBINDUSTRY_HARMONISATION_SCHEMA_VERSION,

            "prompt_version":
                SUBINDUSTRY_HARMONISATION_PROMPT_VERSION,

            "model":
                TAXONOMY_HARMONISATION_MODEL,

            "created_datetime":
                pd.Timestamp.utcnow()
                .isoformat(),

            "payload":
                incremental_payload,

            "result":
                incremental_result,

            "openai_response_id":
                incremental_response.get(
                    "id"
                ),
        }


        append_ai_cache_record(
            taxonomy_cache_path,
            incremental_record,
        )


        taxonomy_cache[
            incremental_taxonomy_key
        ] = incremental_record


    incremental_mapping_df = pd.DataFrame(
        incremental_result[
            "mappings"
        ]
    )


    incremental_new_categories_df = pd.DataFrame(
        incremental_result[
            "new_canonical_categories"
        ]
    )


    if (
        incremental_mapping_df[
            "raw_subindustry"
        ]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            "Incremental taxonomy returned duplicate "
            "raw-subindustry mappings."
        )


    mapped_new_raw = set(
        incremental_mapping_df[
            "raw_subindustry"
        ]
        .astype(str)
    )


    if mapped_new_raw != set(
        new_raw_labels
    ):

        missing_new = (
            set(
                new_raw_labels
            )
            - mapped_new_raw
        )

        unexpected_new = (
            mapped_new_raw
            - set(
                new_raw_labels
            )
        )

        raise RuntimeError(
            "Incremental taxonomy coverage mismatch.\n"
            f"Missing: {sorted(missing_new)[:20]}\n"
            f"Unexpected: {sorted(unexpected_new)[:20]}"
        )


    allowed_incremental_targets = (
        set(
            existing_canonical_categories
        )
        |
        set(
            incremental_new_categories_df[
                "canonical_subindustry"
            ]
            .astype(str)
        )
    )


    invalid_incremental_targets = (
        set(
            incremental_mapping_df[
                "canonical_subindustry"
            ]
            .astype(str)
        )
        - allowed_incremental_targets
    )


    if invalid_incremental_targets:

        raise RuntimeError(
            "Incremental taxonomy mappings reference "
            "undefined categories: "
            f"{sorted(invalid_incremental_targets)}"
        )


# ============================================================
# 12C.4 — BUILD COMPLETE CURRENT RAW → CANONICAL MAP
# ============================================================

current_mapping_rows = []


for raw_label in raw_labels:

    if raw_label in existing_raw_to_canonical:

        current_mapping_rows.append(
            {
                "raw_subindustry":
                    raw_label,

                "canonical_subindustry":
                    existing_raw_to_canonical[
                        raw_label
                    ],

                "mapping_confidence":
                    existing_mapping_confidence.get(
                        raw_label,
                        np.nan,
                    ),

                "mapping_origin":
                    "PERSISTED_REUSE",
            }
        )


if not incremental_mapping_df.empty:

    for _, row in (
        incremental_mapping_df
        .iterrows()
    ):

        current_mapping_rows.append(
            {
                "raw_subindustry":
                    str(
                        row[
                            "raw_subindustry"
                        ]
                    ),

                "canonical_subindustry":
                    str(
                        row[
                            "canonical_subindustry"
                        ]
                    ),

                "mapping_confidence":
                    float(
                        row[
                            "mapping_confidence"
                        ]
                    ),

                "mapping_origin":
                    taxonomy_ai_method,
            }
        )


mapping_df = pd.DataFrame(
    current_mapping_rows
)


if (
    mapping_df[
        "raw_subindustry"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Current taxonomy contains duplicate raw mappings."
    )


expected_raw = set(
    raw_labels
)


mapped_raw = set(
    mapping_df[
        "raw_subindustry"
    ]
    .astype(str)
)


if mapped_raw != expected_raw:

    raise RuntimeError(
        "Current taxonomy does not map every raw label exactly once."
    )


raw_to_canonical = dict(
    zip(
        mapping_df[
            "raw_subindustry"
        ],
        mapping_df[
            "canonical_subindustry"
        ],
    )
)


# ============================================================
# 12C.5 — APPLY CANONICAL TAXONOMY
# ============================================================

theme_classification_df[
    "theme_subindustry"
] = (
    theme_classification_df[
        "theme_subindustry_raw"
    ]
    .map(
        raw_to_canonical
    )
)


classified_after = (
    theme_classification_df[
        "theme_relevance"
    ]
    != "Unclassified"
)


missing_canonical = (
    classified_after
    &
    theme_classification_df[
        "theme_subindustry"
    ]
    .isna()
)


if missing_canonical.any():

    display(
        theme_classification_df.loc[
            missing_canonical,
            [
                "economic_issuer_id",
                "theme_relevance",
                "theme_subindustry_raw",
            ],
        ]
    )


    raise RuntimeError(
        f"{missing_canonical.sum():,} classified issuers "
        "lack canonical subindustry."
    )


# ============================================================
# 12C.6 — BUILD CURRENT TAXONOMY REGISTRY
# ============================================================

theme_subindustry_taxonomy = (
    mapping_df
    .merge(
        raw_counts,
        on="raw_subindustry",
        how="left",
    )
)


theme_subindustry_taxonomy[
    "research_theme"
] = (
    research_theme
)


theme_subindustry_taxonomy[
    "taxonomy_method"
] = (
    theme_subindustry_taxonomy[
        "mapping_origin"
    ]
)


theme_subindustry_taxonomy[
    "taxonomy_model"
] = (
    theme_subindustry_taxonomy[
        "mapping_origin"
    ]
    .map(
        lambda x:
            TAXONOMY_HARMONISATION_MODEL
            if str(x).startswith(
                "AI_"
            )
            else pd.NA
    )
)


theme_subindustry_taxonomy[
    "taxonomy_version"
] = (
    SUBINDUSTRY_HARMONISATION_PROMPT_VERSION
)


theme_subindustry_taxonomy[
    "created_datetime"
] = (
    pd.Timestamp.utcnow()
)


# ============================================================
# 12C.7 — FINAL PERSISTENCE
# ============================================================

theme_classification_df.to_parquet(
    theme_classification_path,
    index=False,
)


theme_subindustry_taxonomy.to_parquet(
    theme_subindustry_taxonomy_path,
    index=False,
)


# ============================================================
# 12C.8 — FINAL REPORT
# ============================================================

canonical_category_count = (
    theme_classification_df[
        "theme_subindustry"
    ]
    .nunique(
        dropna=True
    )
)


print()

print("=" * 80)

print(
    "STEP 12C — THEME SUBINDUSTRY HARMONISATION COMPLETE"
)

print("=" * 80)

print(
    f"Raw subindustries: "
    f"{len(raw_labels):,}"
)

print(
    f"Persisted raw mappings reused: "
    f"{len(existing_reusable_raw):,}"
)

print(
    f"New raw labels harmonised: "
    f"{len(new_raw_labels):,}"
)

print(
    f"Canonical subindustries: "
    f"{canonical_category_count:,}"
)

print(
    f"Unmapped classified issuers: "
    f"{missing_canonical.sum():,}"
)

print(
    f"Incremental taxonomy AI method: "
    f"{taxonomy_ai_method}"
)

print()

print(
    "Saved canonical theme classification:"
)

print(
    theme_classification_path
)

print()

print(
    "Saved canonical taxonomy registry:"
)

print(
    theme_subindustry_taxonomy_path
)

print("=" * 80)


display(
    theme_subindustry_taxonomy[
        [
            "raw_subindustry",
            "canonical_subindustry",
            "mapping_confidence",
            "issuer_count",
            "mapping_origin",
        ]
    ]
    .sort_values(
        [
            "canonical_subindustry",
            "issuer_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


# ============================================================
# 12C.9 — FINAL INTEGRITY CHECKS
# ============================================================

assert isinstance(
    research_theme,
    str,
)


assert research_theme.strip()


assert (
    len(
        theme_classification_df
    )
    ==
    len(
        economic_issuer_master
    )
), (
    "Theme classification row count does not equal "
    "canonical issuer count."
)


assert not (
    theme_classification_df[
        "economic_issuer_id"
    ]
    .duplicated()
    .any()
), (
    "Duplicate economic issuers remain in theme classification."
)


assert (
    set(
        theme_classification_df[
            "economic_issuer_id"
        ]
        .astype(str)
    )
    ==
    CURRENT_CANONICAL_ISSUER_IDS
), (
    "Theme classification issuer universe does not exactly match "
    "the Step 11 canonical issuer universe."
)


assert not (
    classified_after
    &
    theme_classification_df[
        "theme_subindustry_raw"
    ]
    .isna()
).any(), (
    "A classified issuer lacks theme_subindustry_raw."
)


assert not (
    classified_after
    &
    theme_classification_df[
        "theme_subindustry"
    ]
    .isna()
).any(), (
    "A classified issuer lacks canonical theme_subindustry."
)


ALLOWED_RESEARCH_METHODS = {
    "OPENAI_FAST_NO_WEB",
    "OPENAI_WEB_RESEARCH",
    "OPENAI_WEB_RESEARCH_REPAIR",
    "ERROR",
}


unexpected_research_methods = sorted(
    set(
        theme_classification_df[
            "research_method"
        ]
        .dropna()
        .astype(str)
    )
    - ALLOWED_RESEARCH_METHODS
)


assert not unexpected_research_methods, (
    "Unexpected research_method values: "
    f"{unexpected_research_methods}"
)


remaining_error_count_final = int(
    theme_classification_df[
        "theme_classification_method"
    ]
    .eq(
        "ERROR"
    )
    .sum()
)


assert (
    remaining_error_count_final
    == 0
), (
    "Theme classification still contains ERROR rows."
)


print()

print("=" * 80)

print(
    "STEP 12 — FINAL SEMANTIC STATE"
)

print("=" * 80)

print(
    f"Canonical economic issuers: "
    f"{len(theme_classification_df):,}"
)

print(
    f"Persisted classifications reused: "
    f"{persisted_reuse_runtime_count:,}"
)

print(
    f"New fast AI calls: "
    f"{fast_new_count:,}"
)

print(
    f"New web AI calls: "
    f"{web_new_count:,}"
)

print(
    f"Step 12B repairs: "
    f"{repair_success_count:,}"
)

print(
    f"Existing taxonomy mappings reused: "
    f"{len(existing_reusable_raw):,}"
)

print(
    f"New taxonomy labels requiring Terra: "
    f"{len(new_raw_labels):,}"
)

print(
    f"Canonical subindustries: "
    f"{canonical_category_count:,}"
)

print(
    f"Remaining classification errors: "
    f"{remaining_error_count_final:,}"
)

print("=" * 80)

print()

print(
    "Step 12 / 12B / 12C integrity checks: PASS"
)

STEP 12 — AI COST PREFLIGHT
Research theme: Automotive
Canonical economic issuers: 417
Persisted semantic classifications reusable: 417
Additional exact-cache routes requiring no new AI: 0
Genuinely new fast-model calls required: 0
Known genuinely new web calls required: 0
New-fast-call safety limit: 25


,preflight_route,issuer_count
0,PERSISTED_SEMANTIC_REUSE,417



Cost-safety preflight: PASS

STEP 12 — GATED AI THEME CLASSIFICATION
Research theme: Automotive
Economic issuers: 417
Fast no-web model: gpt-5-nano
Web escalation model: gpt-5.6-luna
Persisted semantic reuse available: 417
[04:14:59] ✓ 1/417 (0.2%) | International CSRC Investment
    Route: PERSISTED REUSE | Method: AI_WEB_PROPOSED_VALIDATED | Relevance: High
    Subindustry: Carbon black and specialty materials for tire manufacturing | Runtime: 0.00s
--------------------------------------------------------------------------------
[04:14:59] ✓ 2/417 (0.5%) | ABB Ltd
    Route: PERSISTED REUSE | Method: AI_WEB_PROPOSED_VALIDATED | Relevance: Moderate
    Subindustry: Industrial robotics, factory automation, and electrical infrastructure for automotive manufacturing | Runtime: 0.00s
--------------------------------------------------------------------------------
[04:14:59] ✓ 3/417 (0.7%) | BALLARD POWER SYSTEMS INC.
    Route: PERSISTED REUSE | Method: AI_WEB_PROPOSED_VALIDATED | Releva

,theme_relevance,issuer_count
0,Core,202
1,Moderate,80
2,High,76
3,Peripheral,57
4,Unclassified,2


,research_method,issuer_count
0,OPENAI_WEB_RESEARCH,311
1,OPENAI_FAST_NO_WEB,101
2,OPENAI_WEB_RESEARCH_REPAIR,5



STEP 12B — TARGETED CLASSIFICATION ERROR REPAIR
Rows requiring repair: 0
Repair model: gpt-5.6-luna

STEP 12B — TARGETED ERROR REPAIR COMPLETE
Rows attempted: 0
Successfully repaired: 0
Repair failures: 0
Remaining ERROR rows: 0

Step 12B integrity: PASS

STEP 12C — THEME SUBINDUSTRY HARMONISATION
Research theme: Automotive
Classified issuers: 415
Unique current raw subindustries: 391
Existing raw mappings reusable: 391
Genuinely new raw labels: 0
Existing canonical categories represented: 22

STEP 12C — THEME SUBINDUSTRY HARMONISATION COMPLETE
Raw subindustries: 391
Persisted raw mappings reused: 391
New raw labels harmonised: 0
Canonical subindustries: 22
Unmapped classified issuers: 0
Incremental taxonomy AI method: PERSISTED_REUSE_ONLY

Saved canonical theme classification:
/content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/theme_classification.parquet

Saved canonical taxonomy registry:
/content/drive/MyDrive/Colab Notebooks

,raw_subindustry,canonical_subindustry,mapping_confidence,issuer_count,mapping_origin
14,Automotive LiDAR and autonomous-driving sensin...,ADAS and Autonomous Driving,1.00,1,PERSISTED_REUSE
28,Autonomous sidewalk delivery robotics,ADAS and Autonomous Driving,0.85,1,PERSISTED_REUSE
38,Automotive AI compute and autonomous-driving p...,ADAS and Autonomous Driving,0.95,1,PERSISTED_REUSE
45,Autonomous vehicles and delivery robots,ADAS and Autonomous Driving,0.90,1,PERSISTED_REUSE
65,Automotive LiDAR / Autonomous Driving Sensors,ADAS and Autonomous Driving,1.00,1,PERSISTED_REUSE
...,...,...,...,...,...
350,EV battery cells / Lithium-ion batteries,Vehicle Batteries,1.00,1,PERSISTED_REUSE
358,"Automotive auxiliary, start-stop, parking, and...",Vehicle Batteries,1.00,1,PERSISTED_REUSE
369,Electric-vehicle lithium-ion power batteries,Vehicle Batteries,1.00,1,PERSISTED_REUSE
371,Electric-vehicle and commercial-vehicle lithiu...,Vehicle Batteries,1.00,1,PERSISTED_REUSE



STEP 12 — FINAL SEMANTIC STATE
Canonical economic issuers: 417
Persisted classifications reused: 417
New fast AI calls: 0
New web AI calls: 0
Step 12B repairs: 0
Existing taxonomy mappings reused: 391
New taxonomy labels requiring Terra: 0
Canonical subindustries: 22
Remaining classification errors: 0

Step 12 / 12B / 12C integrity checks: PASS


In [80]:
# 13. SOURCE ROUTING TABLE AND CANONICAL HISTORICAL UNIVERSE MEMBERSHIP

def build_source_routing_table(listings: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in listings.itertuples(index=False):
        route = route_from_mic(row.exchange_mic, row.listing_country)
        rows.append({
            "economic_issuer_id": row.economic_issuer_id,
            "security_id": row.security_id,
            "listing_id": row.listing_id,
            "ticker": row.ticker,
            "exchange_mic": row.exchange_mic,
            "listing_country": row.listing_country,
            **route,
            "valid_from": row.valid_from,
            "valid_to": row.valid_to,
        })
    out = ensure_columns(pd.DataFrame(rows), ROUTING_COLUMNS)
    invalid = set(out["primary_source_engine"].dropna()) - VALID_SOURCE_ENGINES
    if invalid:
        raise AssertionError(f"Unexpected source engines: {invalid}")
    return out

source_routing_table_df = build_source_routing_table(listing_master_df)

def build_historical_universe_membership(x: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in x.itertuples(index=False):
        is_manual = getattr(row, "source_system", None) == "MANUAL_CONFIG"
        rows.append({
            "research_project_id": RESEARCH_PROJECT_ID,
            "fund_id": getattr(row, "fund_id", pd.NA),
            "fund_ticker": getattr(row, "fund_ticker", pd.NA),
            "security_id": row.security_id,
            "listing_id": row.listing_id,
            "economic_issuer_id": row.economic_issuer_id,
            "membership_start_date": getattr(row, "membership_start_date", pd.NaT),
            "membership_end_date": getattr(row, "membership_end_date", pd.NaT),
            "first_observed_date": getattr(row, "snapshot_date", pd.NaT),
            "last_observed_date": getattr(row, "snapshot_date", pd.NaT),
            "weight": getattr(row, "reported_weight", pd.NA),
            "shares": getattr(row, "balance", pd.NA),
            "market_value": getattr(row, "market_value", pd.NA),
            "source_filing_id": getattr(row, "source_filing_id", pd.NA),
            "source_available_datetime": getattr(row, "available_datetime", pd.NaT),
            "source_observation_id": getattr(row, "source_observation_id", pd.NA),
            "universe_source_type": "MANUAL" if is_manual else "ETF",
        })
    return ensure_columns(pd.DataFrame(rows), UNIVERSE_MEMBERSHIP_COLUMNS)

historical_universe_membership_df = build_historical_universe_membership(resolved_identity_df)

print("Routing rows:", len(source_routing_table_df))
print("Universe membership rows:", len(historical_universe_membership_df))
display(source_routing_table_df["primary_source_engine"].value_counts(dropna=False).to_frame())


Routing rows: 538
Universe membership rows: 8144


,count
primary_source_engine,
STRUCTURED_XBRL,196
CHINA_HK,103
EAST_ASIA,86
RESIDUAL_GLOBAL,77
EUROPE,76


In [90]:
# 14. DIAGNOSTICS, COLLISIONS, UNRESOLVED CASES AND CONTRACT VALIDATION

UNIQUE_IDENTIFIER_POLICY = {
    "LEI": "LEGAL_ENTITY",
    "ISIN": "SECURITY",
    "CUSIP": "SECURITY",
}


# ============================================================
# 14.1 — UNIQUE IDENTIFIER COLLISION REPORT
# ============================================================

def build_identifier_collision_report(
    history: pd.DataFrame,
) -> pd.DataFrame:

    columns = [
        "identifier_type",
        "identifier_value",
        "entity_count",
        "entity_types",
        "entities",
        "first_observed",
        "last_observed",
    ]

    if history.empty:
        return pd.DataFrame(columns=columns)

    h = history.loc[
        history["identifier_type"].isin(
            UNIQUE_IDENTIFIER_POLICY
        )
    ].copy()

    if h.empty:
        return pd.DataFrame(columns=columns)

    h["identifier_value"] = h.apply(
        lambda r: usable_identifier(
            r["identifier_value"],
            r["identifier_type"],
        ),
        axis=1,
    )

    h = h.dropna(
        subset=["identifier_value"]
    )

    h = h.loc[
        h.apply(
            lambda r:
                r["entity_type"]
                ==
                UNIQUE_IDENTIFIER_POLICY.get(
                    r["identifier_type"]
                ),
            axis=1,
        )
    ]

    g = (
        h.groupby(
            [
                "identifier_type",
                "identifier_value",
            ],
            dropna=False,
        )
        .agg(
            entity_count=(
                "entity_id",
                "nunique",
            ),
            entity_types=(
                "entity_type",
                lambda s:
                    "|".join(
                        sorted(set(map(str, s)))
                    ),
            ),
            entities=(
                "entity_id",
                lambda s:
                    "|".join(
                        sorted(set(map(str, s)))
                    ),
            ),
            first_observed=(
                "first_observed_date",
                "min",
            ),
            last_observed=(
                "last_observed_date",
                "max",
            ),
        )
        .reset_index()
    )

    return g.loc[
        g["entity_count"] > 1
    ].copy()


identifier_collision_report_df = (
    build_identifier_collision_report(
        identifier_history_df
    )
)


# ============================================================
# 14.2 — UNRESOLVED CANONICAL IDENTITY
# ============================================================

identity_unresolved_mask = (
    resolved_identity_df[
        "security_identity_resolution_method"
    ].eq("SOURCE_OBSERVATION")
    |
    resolved_identity_df[
        "security_id"
    ].isna()
    |
    resolved_identity_df[
        "canonical_security_legal_entity_id"
    ].isna()
    |
    resolved_identity_df[
        "economic_issuer_id"
    ].isna()
)

identity_unresolved_df = (
    resolved_identity_df.loc[
        identity_unresolved_mask
    ].copy()
)


# ============================================================
# 14.3 — SOURCE IDENTITY EVIDENCE GAPS
# ============================================================

source_identity_evidence_missing_df = (
    resolved_identity_df.loc[
        resolved_identity_df[
            "issuer_name"
        ].isna()
        &
        resolved_identity_df[
            "issuer_lei"
        ].isna()
    ].copy()
)


# ============================================================
# 14.4 — SECURITY → ONE ECONOMIC ISSUER INVARIANT
# ============================================================

_security_issuer_counts = (
    resolved_identity_df
    .groupby("security_id")[
        "economic_issuer_id"
    ]
    .nunique()
)

security_economic_issuer_conflicts_df = (
    _security_issuer_counts[
        _security_issuer_counts > 1
    ]
    .rename("economic_issuer_count")
    .reset_index()
)


# ============================================================
# 14.5 — LISTING + ROUTING DIAGNOSTICS
# ============================================================

listing_unresolved_df = (
    resolved_identity_df.loc[
        resolved_identity_df[
            "exchange_mic"
        ].isna()
    ].copy()
)

routing_unresolved_df = (
    source_routing_table_df.loc[
        source_routing_table_df[
            "listing_country"
        ].isna()
    ].copy()
)


# ============================================================
# 14.6 — BLOCK 1 QUALITY SUMMARY
# ============================================================

quality_metrics = [
    (
        "identity_evidence_rows",
        len(resolved_identity_df),
    ),
    (
        "economic_issuer_count",
        economic_issuer_master_df[
            "economic_issuer_id"
        ].nunique(),
    ),
    (
        "legal_entity_count",
        legal_entity_master_df[
            "legal_entity_id"
        ].nunique(),
    ),
    (
        "security_count",
        security_master_df[
            "security_id"
        ].nunique(),
    ),
    (
        "listing_count",
        listing_master_df[
            "listing_id"
        ].nunique(),
    ),
    (
        "unresolved_identity_rows",
        len(identity_unresolved_df),
    ),
    (
        "source_identity_evidence_missing_rows",
        len(source_identity_evidence_missing_df),
    ),
    (
        "unresolved_listing_rows",
        len(listing_unresolved_df),
    ),
    (
        "identifier_collision_rows",
        len(identifier_collision_report_df),
    ),
    (
        "routing_unresolved_rows",
        len(routing_unresolved_df),
    ),
    (
        "membership_rows",
        len(historical_universe_membership_df),
    ),
    (
        "security_rows_cusip_to_isin_bridge",
        identity_repair_stats.get(
            "security_rows_cusip_to_isin_bridge",
            0,
        ),
    ),
    (
        "ambiguous_cusip_values_quarantined",
        identity_repair_stats.get(
            "ambiguous_cusip_values_quarantined",
            0,
        ),
    ),
    (
        "external_lei_records",
        identity_repair_stats.get(
            "external_lei_records",
            0,
        ),
    ),
    (
        "external_lei_validated_rows",
        identity_repair_stats.get(
            "external_lei_validated_rows",
            0,
        ),
    ),
    (
        "external_lei_provisional_rows",
        identity_repair_stats.get(
            "external_lei_provisional_rows",
            0,
        ),
    ),
    (
        "external_lei_quarantined_rows",
        identity_repair_stats.get(
            "external_lei_quarantined_rows",
            0,
        ),
    ),
    (
        "identity_semantic_review_count",
        identity_repair_stats.get(
            "identity_semantic_review_count",
            0,
        ),
    ),
    (
        "identity_semantic_review_cache_hits",
        identity_repair_stats.get(
            "identity_semantic_review_cache_hits",
            0,
        ),
    ),
    (
        "security_economic_issuer_conflicts",
        len(
            security_economic_issuer_conflicts_df
        ),
    ),
    (
        "invalid_cusip_rows_excluded_from_identity",
        identity_repair_stats.get(
            "invalid_cusip_rows_excluded_from_identity",
            0,
        ),
    ),
    (
        "invalid_isin_rows_excluded_from_identity",
        identity_repair_stats.get(
            "invalid_isin_rows_excluded_from_identity",
            0,
        ),
    ),
]

block_1_quality_summary_df = (
    pd.DataFrame(
        quality_metrics,
        columns=[
            "metric",
            "value",
        ],
    )
)


# ============================================================
# 14.7 — CANONICAL KEY + FOREIGN-KEY CONTRACTS
# ============================================================

assert_unique_non_null(
    economic_issuer_master_df,
    "economic_issuer_id",
    "economic_issuer_master",
)

assert_unique_non_null(
    legal_entity_master_df,
    "legal_entity_id",
    "legal_entity_master",
)

assert_unique_non_null(
    security_master_df,
    "security_id",
    "security_master",
)

assert_unique_non_null(
    listing_master_df,
    "listing_id",
    "listing_master",
)

assert_foreign_key(
    legal_entity_master_df,
    "economic_issuer_id",
    economic_issuer_master_df,
    "economic_issuer_id",
    "legal_entity_master",
    allow_null=False,
)

assert_foreign_key(
    security_master_df,
    "legal_entity_id",
    legal_entity_master_df,
    "legal_entity_id",
    "security_master",
    allow_null=False,
)

assert_foreign_key(
    security_master_df,
    "economic_issuer_id",
    economic_issuer_master_df,
    "economic_issuer_id",
    "security_master",
    allow_null=False,
)

assert_foreign_key(
    listing_master_df,
    "security_id",
    security_master_df,
    "security_id",
    "listing_master",
    allow_null=False,
)

assert_foreign_key(
    historical_universe_membership_df,
    "security_id",
    security_master_df,
    "security_id",
    "historical_universe_membership",
    allow_null=False,
)

assert_foreign_key(
    historical_universe_membership_df,
    "listing_id",
    listing_master_df,
    "listing_id",
    "historical_universe_membership",
    allow_null=False,
)

if (
    historical_universe_membership_df[
        "source_available_datetime"
    ]
    .isna()
    .any()
):
    print(
        "WARNING: some membership observations lack "
        "a source availability timestamp."
    )


# ============================================================
# 14B. ADDITION — DUPLICATE + THEME SEMANTIC QC
# ============================================================

print()
print("=" * 72)
print("ADDITIONAL IDENTITY + THEME QC")
print("=" * 72)


# ============================================================
# 14B.1 — ECONOMIC-ISSUER DUPLICATE CANDIDATES
# ============================================================
#
# Step 11B preserves candidate history.
#
# Step 11C may subsequently absorb one candidate issuer into
# another canonical issuer.
#
# Therefore historical strong candidates are not automatically
# unresolved strong candidates.
# ============================================================

duplicate_candidate_count = len(
    economic_issuer_duplicate_candidates
)

STRONG_DUPLICATE_STRENGTHS = {
    "STRONG_REFERENCE_EVIDENCE",
    "EXACT_NORMALISED_NAME",
    "STRONG_NAME_SIMILARITY",
}

if duplicate_candidate_count:

    all_strong_duplicate_candidates = (
        economic_issuer_duplicate_candidates.loc[
            economic_issuer_duplicate_candidates[
                "candidate_strength"
            ].isin(
                STRONG_DUPLICATE_STRENGTHS
            )
        ].copy()
    )

else:

    all_strong_duplicate_candidates = (
        pd.DataFrame(
            columns=
                economic_issuer_duplicate_candidates.columns
        )
    )

historical_strong_duplicate_candidate_count = (
    len(
        all_strong_duplicate_candidates
    )
)

final_canonical_issuer_ids = set(
    economic_issuer_master_df[
        "economic_issuer_id"
    ]
    .dropna()
    .astype(str)
)


# ------------------------------------------------------------
# Load Step 11C economic-issuer alias history if available.
# ------------------------------------------------------------

economic_issuer_alias_map_qc = {}

alias_history_candidates = []

for candidate_name in [
    "economic_issuer_alias_history",
    "economic_issuer_alias_history_df",
]:

    candidate_obj = globals().get(
        candidate_name
    )

    if isinstance(
        candidate_obj,
        pd.DataFrame,
    ):
        alias_history_candidates.append(
            candidate_obj.copy()
        )


if not alias_history_candidates:

    try:
        _block1_canonical_dir_qc = Path(
            BLOCK1_CANONICAL_DIR
        )

    except Exception:
        _block1_canonical_dir_qc = Path(
            "/content/drive/MyDrive/Colab Notebooks/"
            "AI-Powered Global PIT Equity Research Engine/"
            "data/canonical/block_1"
        )

    _alias_history_path_qc = (
        _block1_canonical_dir_qc
        / "economic_issuer_alias_history.parquet"
    )

    if _alias_history_path_qc.exists():

        try:
            alias_history_candidates.append(
                pd.read_parquet(
                    _alias_history_path_qc
                )
            )

        except Exception as exc:
            print(
                "WARNING: economic issuer alias history "
                "could not be loaded for QC:"
            )
            print(
                f"    {type(exc).__name__}: {exc}"
            )


if alias_history_candidates:

    economic_issuer_alias_history_qc = (
        alias_history_candidates[0].copy()
    )

    required_alias_columns = {
        "absorbed_economic_issuer_id",
        "canonical_economic_issuer_id",
    }

    if required_alias_columns.issubset(
        economic_issuer_alias_history_qc.columns
    ):

        _alias_rows_qc = (
            economic_issuer_alias_history_qc.loc[
                economic_issuer_alias_history_qc[
                    "absorbed_economic_issuer_id"
                ].notna()
                &
                economic_issuer_alias_history_qc[
                    "canonical_economic_issuer_id"
                ].notna()
            ]
            .copy()
        )

        economic_issuer_alias_map_qc = dict(
            zip(
                _alias_rows_qc[
                    "absorbed_economic_issuer_id"
                ].astype(str),
                _alias_rows_qc[
                    "canonical_economic_issuer_id"
                ].astype(str),
            )
        )


def resolve_final_economic_issuer_id_qc(
    issuer_id,
):

    if pd.isna(issuer_id):
        return None

    current = str(issuer_id)

    seen = set()

    while (
        current
        in economic_issuer_alias_map_qc
    ):

        if current in seen:
            raise RuntimeError(
                "Cycle detected in economic issuer "
                "alias history during Step 14 QC."
            )

        seen.add(current)

        current = str(
            economic_issuer_alias_map_qc[
                current
            ]
        )

    return current


resolved_strong_duplicate_rows = []
unresolved_strong_duplicate_rows = []

for _, candidate_row in (
    all_strong_duplicate_candidates
    .iterrows()
):

    left_original = str(
        candidate_row[
            "economic_issuer_id_left"
        ]
    )

    right_original = str(
        candidate_row[
            "economic_issuer_id_right"
        ]
    )

    left_final = (
        resolve_final_economic_issuer_id_qc(
            left_original
        )
    )

    right_final = (
        resolve_final_economic_issuer_id_qc(
            right_original
        )
    )

    left_original_still_canonical = (
        left_original
        in final_canonical_issuer_ids
    )

    right_original_still_canonical = (
        right_original
        in final_canonical_issuer_ids
    )

    same_final_issuer = (
        left_final is not None
        and right_final is not None
        and left_final == right_final
    )

    one_or_both_absorbed = not (
        left_original_still_canonical
        and right_original_still_canonical
    )

    candidate_resolved = (
        same_final_issuer
        or one_or_both_absorbed
    )

    out_row = (
        candidate_row.to_dict()
    )

    out_row[
        "final_economic_issuer_id_left"
    ] = left_final

    out_row[
        "final_economic_issuer_id_right"
    ] = right_final

    out_row[
        "left_original_still_canonical"
    ] = left_original_still_canonical

    out_row[
        "right_original_still_canonical"
    ] = right_original_still_canonical

    out_row[
        "duplicate_resolution_status"
    ] = (
        "RESOLVED"
        if candidate_resolved
        else "UNRESOLVED"
    )

    if candidate_resolved:
        resolved_strong_duplicate_rows.append(
            out_row
        )

    else:
        unresolved_strong_duplicate_rows.append(
            out_row
        )


resolved_strong_duplicate_candidates = (
    pd.DataFrame(
        resolved_strong_duplicate_rows
    )
)

strong_duplicate_candidates = (
    pd.DataFrame(
        unresolved_strong_duplicate_rows
    )
)

resolved_strong_duplicate_candidate_count = (
    len(
        resolved_strong_duplicate_candidates
    )
)

strong_duplicate_candidate_count = (
    len(
        strong_duplicate_candidates
    )
)


# ============================================================
# 14B.2 — SECURITY → ONE ECONOMIC ISSUER
# ============================================================

security_issuer_counts = (
    security_master_df
    .groupby(
        "security_id"
    )[
        "economic_issuer_id"
    ]
    .nunique(
        dropna=True
    )
)

security_economic_issuer_conflicts = (
    security_issuer_counts[
        security_issuer_counts > 1
    ]
)

security_economic_issuer_conflict_count = (
    len(
        security_economic_issuer_conflicts
    )
)


# ============================================================
# 14B.3 — THEME CLASSIFICATION COVERAGE
# ============================================================

theme_total = len(
    theme_classification_df
)

theme_unclassified_count = int(
    (
        theme_classification_df[
            "theme_relevance"
        ]
        == "Unclassified"
    ).sum()
)

theme_error_count = int(
    (
        theme_classification_df[
            "theme_classification_method"
        ]
        == "ERROR"
    ).sum()
)

theme_classified_count = (
    theme_total
    - theme_unclassified_count
)


# ============================================================
# 14B.4 — CANONICAL SUBINDUSTRY COVERAGE
# ============================================================

classified_mask = (
    theme_classification_df[
        "theme_relevance"
    ]
    != "Unclassified"
)

classified_missing_raw = int(
    (
        classified_mask
        &
        theme_classification_df[
            "theme_subindustry_raw"
        ].isna()
    ).sum()
)

classified_missing_canonical = int(
    (
        classified_mask
        &
        theme_classification_df[
            "theme_subindustry"
        ].isna()
    ).sum()
)

raw_subindustry_count = (
    theme_classification_df[
        "theme_subindustry_raw"
    ]
    .nunique(
        dropna=True
    )
)

canonical_subindustry_count = (
    theme_classification_df[
        "theme_subindustry"
    ]
    .nunique(
        dropna=True
    )
)


# ============================================================
# 14B.5 — RESEARCH PROVENANCE STATUS
# ============================================================
#
# Deterministic QC classification only.
#
# This makes no AI or web calls.
#
# WEB_SOURCES_CAPTURED:
#   Web research was used and at least one source was captured.
#
# WEB_SOURCES_NOT_CAPTURED_LEGACY:
#   Successful web classification exists, but the historical
#   response did not persist extracted source URLs.
#
# NO_WEB_RESEARCH:
#   Fast semantic classification intentionally used no web.
#
# UNCLASSIFIED:
#   Semantic pipeline abstained / left issuer unclassified.
#
# OTHER:
#   Anything outside the expected production states.
# ============================================================

theme_classification_qc_df = (
    theme_classification_df.copy()
)

research_sources_empty_mask = (
    theme_classification_qc_df[
        "research_sources_json"
    ]
    .fillna("[]")
    .astype(str)
    .str.strip()
    .isin(
        [
            "",
            "[]",
            "null",
            "None",
            "<NA>",
        ]
    )
)

web_method_mask = (
    theme_classification_qc_df[
        "research_method"
    ]
    .isin(
        [
            "OPENAI_WEB_RESEARCH",
            "OPENAI_WEB_RESEARCH_REPAIR",
        ]
    )
)

fast_method_mask = (
    theme_classification_qc_df[
        "research_method"
    ]
    .eq(
        "OPENAI_FAST_NO_WEB"
    )
)

unclassified_qc_mask = (
    theme_classification_qc_df[
        "theme_relevance"
    ]
    .eq(
        "Unclassified"
    )
)

theme_classification_qc_df[
    "research_provenance_status"
] = "OTHER"

theme_classification_qc_df.loc[
    unclassified_qc_mask,
    "research_provenance_status",
] = "UNCLASSIFIED"

theme_classification_qc_df.loc[
    (
        ~unclassified_qc_mask
        &
        fast_method_mask
    ),
    "research_provenance_status",
] = "NO_WEB_RESEARCH"

theme_classification_qc_df.loc[
    (
        ~unclassified_qc_mask
        &
        web_method_mask
        &
        ~research_sources_empty_mask
    ),
    "research_provenance_status",
] = "WEB_SOURCES_CAPTURED"

theme_classification_qc_df.loc[
    (
        ~unclassified_qc_mask
        &
        web_method_mask
        &
        research_sources_empty_mask
    ),
    "research_provenance_status",
] = "WEB_SOURCES_NOT_CAPTURED_LEGACY"


research_provenance_summary_df = (
    theme_classification_qc_df[
        "research_provenance_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "research_provenance_status"
    )
    .reset_index(
        name="row_count"
    )
)


def _provenance_count(
    status,
):
    return int(
        (
            theme_classification_qc_df[
                "research_provenance_status"
            ]
            == status
        ).sum()
    )


web_sources_captured_count = (
    _provenance_count(
        "WEB_SOURCES_CAPTURED"
    )
)

web_sources_not_captured_legacy_count = (
    _provenance_count(
        "WEB_SOURCES_NOT_CAPTURED_LEGACY"
    )
)

no_web_research_count = (
    _provenance_count(
        "NO_WEB_RESEARCH"
    )
)

unclassified_provenance_count = (
    _provenance_count(
        "UNCLASSIFIED"
    )
)

other_provenance_count = (
    _provenance_count(
        "OTHER"
    )
)


# Preserve explicit rows for inspection / downstream audit.

web_sources_not_captured_legacy_df = (
    theme_classification_qc_df.loc[
        theme_classification_qc_df[
            "research_provenance_status"
        ].eq(
            "WEB_SOURCES_NOT_CAPTURED_LEGACY"
        )
    ].copy()
)

unexpected_research_provenance_df = (
    theme_classification_qc_df.loc[
        theme_classification_qc_df[
            "research_provenance_status"
        ].eq(
            "OTHER"
        )
    ].copy()
)


# ============================================================
# 14B.6 — PROVENANCE CONSISTENCY CHECKS
# ============================================================

web_researched_classified_rows = int(
    (
        classified_mask
        &
        web_method_mask
    ).sum()
)

fast_no_web_classified_rows = int(
    (
        classified_mask
        &
        fast_method_mask
    ).sum()
)

expected_web_total = (
    web_sources_captured_count
    +
    web_sources_not_captured_legacy_count
)

assert (
    expected_web_total
    ==
    web_researched_classified_rows
), (
    "Research provenance accounting mismatch: "
    "captured + legacy-missing web provenance does not "
    "equal the number of web-researched classified rows."
)

assert (
    no_web_research_count
    ==
    fast_no_web_classified_rows
), (
    "Research provenance accounting mismatch: "
    "NO_WEB_RESEARCH does not equal fast no-web "
    "classified rows."
)

assert (
    unclassified_provenance_count
    ==
    theme_unclassified_count
), (
    "Research provenance accounting mismatch: "
    "UNCLASSIFIED provenance count does not equal "
    "theme unclassified count."
)


# ============================================================
# 14B.7 — ADDITIONAL QC SUMMARY
# ============================================================

additional_qc = pd.DataFrame(
    [
        {
            "metric":
                "economic_issuer_duplicate_candidates",
            "value":
                duplicate_candidate_count,
            "critical":
                False,
        },
        {
            "metric":
                "historical_strong_economic_issuer_duplicate_candidates",
            "value":
                historical_strong_duplicate_candidate_count,
            "critical":
                False,
        },
        {
            "metric":
                "resolved_strong_economic_issuer_duplicate_candidates",
            "value":
                resolved_strong_duplicate_candidate_count,
            "critical":
                False,
        },
        {
            "metric":
                "unresolved_strong_economic_issuer_duplicate_candidates",
            "value":
                strong_duplicate_candidate_count,
            "critical":
                True,
        },
        {
            "metric":
                "security_economic_issuer_conflicts",
            "value":
                security_economic_issuer_conflict_count,
            "critical":
                True,
        },
        {
            "metric":
                "theme_classification_rows",
            "value":
                theme_total,
            "critical":
                False,
        },
        {
            "metric":
                "theme_classified_rows",
            "value":
                theme_classified_count,
            "critical":
                False,
        },
        {
            "metric":
                "theme_unclassified_rows",
            "value":
                theme_unclassified_count,
            "critical":
                False,
        },
        {
            "metric":
                "theme_error_rows",
            "value":
                theme_error_count,
            "critical":
                True,
        },
        {
            "metric":
                "classified_missing_raw_subindustry",
            "value":
                classified_missing_raw,
            "critical":
                True,
        },
        {
            "metric":
                "classified_missing_canonical_subindustry",
            "value":
                classified_missing_canonical,
            "critical":
                True,
        },
        {
            "metric":
                "raw_subindustry_count",
            "value":
                raw_subindustry_count,
            "critical":
                False,
        },
        {
            "metric":
                "canonical_subindustry_count",
            "value":
                canonical_subindustry_count,
            "critical":
                False,
        },
        {
            "metric":
                "web_researched_classified_rows",
            "value":
                web_researched_classified_rows,
            "critical":
                False,
        },
        {
            "metric":
                "web_sources_captured",
            "value":
                web_sources_captured_count,
            "critical":
                False,
        },
        {
            "metric":
                "web_sources_not_captured_legacy",
            "value":
                web_sources_not_captured_legacy_count,
            "critical":
                False,
        },
        {
            "metric":
                "fast_no_web_classified_rows",
            "value":
                fast_no_web_classified_rows,
            "critical":
                False,
        },
        {
            "metric":
                "unclassified_provenance_rows",
            "value":
                unclassified_provenance_count,
            "critical":
                False,
        },
        {
            "metric":
                "unexpected_research_provenance_rows",
            "value":
                other_provenance_count,
            "critical":
                True,
        },
    ]
)

display(
    additional_qc
)


print()
print(
    "RESEARCH PROVENANCE STATUS"
)

display(
    research_provenance_summary_df
)


# ============================================================
# 14B.8 — CRITICAL IDENTITY ASSERTIONS
# ============================================================

if (
    security_economic_issuer_conflict_count
    > 0
):
    raise AssertionError(
        "CRITICAL QC FAILURE: "
        f"{security_economic_issuer_conflict_count:,} "
        "securities map to multiple economic issuers."
    )


if (
    strong_duplicate_candidate_count
    > 0
):

    print()
    print(
        "BLOCK 1 FREEZE WARNING:"
    )

    print(
        f"{strong_duplicate_candidate_count:,} "
        "strong economic-issuer duplicate candidates "
        "remain unresolved in the FINAL canonical graph."
    )

    display_columns = [
        col
        for col in [
            "economic_issuer_id_left",
            "issuer_name_left",
            "economic_issuer_id_right",
            "issuer_name_right",
            "name_similarity",
            "candidate_strength",
            "resolution_reason",
            "final_economic_issuer_id_left",
            "final_economic_issuer_id_right",
            "duplicate_resolution_status",
        ]
        if col
        in strong_duplicate_candidates.columns
    ]

    display(
        strong_duplicate_candidates[
            display_columns
        ]
    )


# ============================================================
# 14B.9 — CRITICAL THEME + PROVENANCE ASSERTIONS
# ============================================================

if (
    theme_error_count
    > 0
):
    raise AssertionError(
        "CRITICAL QC FAILURE: "
        f"{theme_error_count:,} "
        "theme-classification errors remain."
    )


if (
    classified_missing_raw
    > 0
):
    raise AssertionError(
        "CRITICAL QC FAILURE: "
        f"{classified_missing_raw:,} "
        "classified issuers lack "
        "theme_subindustry_raw."
    )


if (
    classified_missing_canonical
    > 0
):
    raise AssertionError(
        "CRITICAL QC FAILURE: "
        f"{classified_missing_canonical:,} "
        "classified issuers lack canonical "
        "theme_subindustry."
    )


if (
    other_provenance_count
    > 0
):

    print()

    print(
        "UNEXPECTED RESEARCH PROVENANCE ROWS:"
    )

    _unexpected_columns = [
        col
        for col in [
            "economic_issuer_id",
            "theme_relevance",
            "theme_classification_method",
            "research_method",
            "research_model",
            "web_research_used",
            "research_source_count",
            "research_sources_json",
            "research_provenance_status",
        ]
        if col
        in unexpected_research_provenance_df.columns
    ]

    display(
        unexpected_research_provenance_df[
            _unexpected_columns
        ]
    )

    raise AssertionError(
        "CRITICAL QC FAILURE: "
        f"{other_provenance_count:,} theme rows have "
        "an unexpected research provenance state."
    )


# ============================================================
# 14B.10 — FINAL STATUS
# ============================================================

print()

print(
    "Security → economic issuer invariant: "
    "PASS"
)

print(
    "Theme semantic pipeline integrity: "
    "PASS"
)

print(
    "Research provenance accounting: "
    "PASS"
)

if (
    strong_duplicate_candidate_count
    == 0
):
    print(
        "Strong economic-issuer duplicate audit: "
        "PASS"
    )

else:
    print(
        "Strong economic-issuer duplicate audit: "
        "REVIEW REQUIRED"
    )


print(
    "Historical strong duplicate candidates:",
    historical_strong_duplicate_candidate_count,
)

print(
    "Resolved strong duplicate candidates:",
    resolved_strong_duplicate_candidate_count,
)

print(
    "Unresolved strong duplicate candidates:",
    strong_duplicate_candidate_count,
)

print(
    "Web sources captured:",
    web_sources_captured_count,
)

print(
    "Legacy web-source capture gaps:",
    web_sources_not_captured_legacy_count,
)

print(
    "Fast/no-web classifications:",
    no_web_research_count,
)

print(
    "Unclassified:",
    unclassified_provenance_count,
)

print(
    "Unexpected provenance states:",
    other_provenance_count,
)

print(
    "Canonical contracts validated."
)

print(
    "Genuine unique-identifier collisions:",
    len(
        identifier_collision_report_df
    ),
)

print(
    "Unresolved canonical identity rows:",
    len(
        identity_unresolved_df
    ),
)

print(
    "Source identity evidence missing but "
    "canonically resolved:",
    len(
        source_identity_evidence_missing_df
    ),
)

print(
    "Routing unresolved rows:",
    len(
        routing_unresolved_df
    ),
)

print(
    "Security → economic issuer conflicts:",
    len(
        security_economic_issuer_conflicts_df
    ),
)

print(
    "GLEIF quarantine rows:",
    len(
        identity_quarantine_df
    ),
)


# ============================================================
# 14B.11 — FINAL HARD CONTRACT
# ============================================================

assert (
    len(
        security_economic_issuer_conflicts_df
    )
    == 0
), (
    "A security maps to multiple economic issuers."
)

assert (
    strong_duplicate_candidate_count
    == 0
), (
    "Strong unresolved economic-issuer duplicate "
    "candidates remain in the final canonical graph."
)

assert (
    theme_error_count
    == 0
), (
    "Theme-classification ERROR rows remain."
)

assert (
    classified_missing_raw
    == 0
), (
    "Classified issuers lack raw subindustry."
)

assert (
    classified_missing_canonical
    == 0
), (
    "Classified issuers lack canonical subindustry."
)

assert (
    other_provenance_count
    == 0
), (
    "Unexpected research provenance states remain."
)


print()
print("=" * 72)
print("BLOCK 1 CRITICAL QC: PASS")
print("=" * 72)

display(
    block_1_quality_summary_df
)



ADDITIONAL IDENTITY + THEME QC


,metric,value,critical
0,economic_issuer_duplicate_candidates,15,False
1,historical_strong_economic_issuer_duplicate_ca...,4,False
2,resolved_strong_economic_issuer_duplicate_cand...,4,False
3,unresolved_strong_economic_issuer_duplicate_ca...,0,True
4,security_economic_issuer_conflicts,0,True
5,theme_classification_rows,417,False
6,theme_classified_rows,415,False
7,theme_unclassified_rows,2,False
8,theme_error_rows,0,True
9,classified_missing_raw_subindustry,0,True



RESEARCH PROVENANCE STATUS


,research_provenance_status,row_count
0,WEB_SOURCES_CAPTURED,306
1,NO_WEB_RESEARCH,101
2,WEB_SOURCES_NOT_CAPTURED_LEGACY,8
3,UNCLASSIFIED,2



Security → economic issuer invariant: PASS
Theme semantic pipeline integrity: PASS
Research provenance accounting: PASS
Strong economic-issuer duplicate audit: PASS
Historical strong duplicate candidates: 4
Resolved strong duplicate candidates: 4
Unresolved strong duplicate candidates: 0
Web sources captured: 306
Legacy web-source capture gaps: 8
Fast/no-web classifications: 101
Unclassified: 2
Unexpected provenance states: 0
Canonical contracts validated.
Genuine unique-identifier collisions: 0
Unresolved canonical identity rows: 0
Source identity evidence missing but canonically resolved: 2
Routing unresolved rows: 0
Security → economic issuer conflicts: 0
GLEIF quarantine rows: 2

BLOCK 1 CRITICAL QC: PASS


,metric,value
0,identity_evidence_rows,8144
1,economic_issuer_count,417
2,legal_entity_count,493
3,security_count,493
4,listing_count,538
5,unresolved_identity_rows,0
6,source_identity_evidence_missing_rows,2
7,unresolved_listing_rows,5729
8,identifier_collision_rows,0
9,routing_unresolved_rows,0


In [96]:
# 15. PERSIST PARQUET OUTPUTS AND WRITE THE BLOCK MANIFEST

def parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        # Convert mixed Python objects to strings only when Arrow cannot infer safely.
        if out[c].dtype == "object":
            non_null = out[c].dropna()
            if len(non_null) and any(isinstance(v, (list, dict, set, tuple)) for v in non_null.head(100)):
                out[c] = out[c].map(
                    lambda v: json.dumps(v, ensure_ascii=False, default=str)
                    if isinstance(v, (list, dict, set, tuple)) else v
                )
    return out

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def persist_dataframe(
    name: str,
    df: pd.DataFrame,
    directory: Path,
    overwrite: bool = True,
) -> dict:
    path = directory / f"{name}.parquet"
    if path.exists() and not overwrite:
        raise FileExistsError(path)
    safe = parquet_safe(df)
    safe.to_parquet(path, index=False)
    return {
        "table_name": name,
        "path": str(path),
        "row_count": int(len(safe)),
        "column_count": int(len(safe.columns)),
        "columns": list(safe.columns),
        "file_size_bytes": int(path.stat().st_size),
        "sha256": file_sha256(path),
        "created_at_utc": utc_now_iso(),
    }

canonical_tables = {
    "economic_issuer_master": economic_issuer_master_df,
    "legal_entity_master": legal_entity_master_df,
    "security_master": security_master_df,
    "listing_master": listing_master_df,
    "identifier_history": identifier_history_df,
    "historical_universe_membership": historical_universe_membership_df,
    "theme_classification": theme_classification_df,
    "source_routing_table": source_routing_table_df,
}

diagnostic_tables = {
    "identity_unresolved": identity_unresolved_df,
    "listing_unresolved": listing_unresolved_df,
    "identifier_collisions": identifier_collision_report_df,
    "routing_unresolved": routing_unresolved_df,
    "block_1_quality_summary": block_1_quality_summary_df,
    "nport_download_log": nport_download_log_df,
    "nport_snapshot_index": etf_snapshot_index_df,
    "identity_reference_evidence": identity_reference_evidence_df,
    "identity_quarantine": identity_quarantine_df,
    "security_economic_issuer_conflicts": security_economic_issuer_conflicts_df,
}

raw_tables = {
    "nport_filing_history": etf_filing_history_df,
    "nport_holdings_standardised": etf_holdings_standardised_df,
}
if PERSIST_RAW_NPORT_HOLDINGS:
    raw_tables["nport_holdings_raw"] = etf_holdings_raw_df

manifest_tables = []

for name, df in canonical_tables.items():
    manifest_tables.append(persist_dataframe(name, df, CANONICAL_DIR, OVERWRITE_OUTPUTS))

for name, df in diagnostic_tables.items():
    manifest_tables.append(persist_dataframe(name, df, INTERIM_DIR, OVERWRITE_OUTPUTS))

for name, df in raw_tables.items():
    manifest_tables.append(persist_dataframe(name, df, RAW_DIR, OVERWRITE_OUTPUTS))

manifest = {
    "block": 1,
    "block_name": "Research Configuration, Universe & Identity Engine",
    "schema_version": SCHEMA_VERSION,
    "identity_namespace_version": IDENTITY_NAMESPACE_VERSION,
    "created_at_utc": utc_now_iso(),
    "run_id": RUN_ID,
    "research_project_id": RESEARCH_PROJECT_ID,
    "research_theme": RESEARCH_THEME,
    "config_hash": CONFIG_HASH,
    "config": RESEARCH_CONFIG,
    "project_root": str(PROJECT_ROOT),
    "canonical_output_directory": str(CANONICAL_DIR),
    "source": "SEC Form N-PORT structured-data bulk files plus optional manual securities; GLEIF public API/cache for legal-entity reference evidence",
    "identity_reference_data": {
        "provider": "GLEIF",
        "api_key_required": False,
        "network_enabled": GLEIF_NETWORK_ENABLED,
        "cache_directory": str(GLEIF_CACHE_DIR),
    },
    "date_range": {
        "start_date": str(START_DATE.date()),
        "end_date": str(END_DATE.date()),
    },
    "target_funds": TARGET_FUNDS,
    "identity_hierarchy": [
        "ECONOMIC_ISSUER",
        "LEGAL_ENTITY",
        "SECURITY",
        "LISTING",
    ],
    "canonical_tables": list(canonical_tables),
    "diagnostic_tables": list(diagnostic_tables),
    "tables": manifest_tables,
}

BLOCK_MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

print("Published canonical outputs:")
for rec in manifest_tables:
    if rec["table_name"] in canonical_tables:
        print(f"  {rec['table_name']}: {rec['row_count']:,} rows -> {rec['path']}")

print("\nManifest:", BLOCK_MANIFEST_PATH)


Published canonical outputs:
  economic_issuer_master: 417 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/economic_issuer_master.parquet
  legal_entity_master: 493 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/legal_entity_master.parquet
  security_master: 493 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/security_master.parquet
  listing_master: 538 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/listing_master.parquet
  identifier_history: 1,159 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/data/canonical/block_1/identifier_history.parquet
  historical_universe_membership: 8,144 rows -> /content/drive/MyDrive/Colab Notebooks/AI-Powered Global PIT Equity Research Engine/d

In [97]:
# ============================================================
# BLOCK 1 COMPLETION SOUND
# ============================================================

from IPython.display import Audio, display

sample_rate = 44_100
duration = 0.6
frequency = 880

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
tone = 0.35 * np.sin(2 * np.pi * frequency * t)

print("✅ Block 1 - Research Configuration, Universe & Identity Engine complete.")
display(Audio(tone, rate=sample_rate, autoplay=True))

✅ Block 1 - Research Configuration, Universe & Identity Engine complete.


## Published contract

On successful completion, the notebook publishes the following authoritative canonical datasets:

- `economic_issuer_master.parquet`
- `legal_entity_master.parquet`
- `security_master.parquet`
- `listing_master.parquet`
- `identifier_history.parquet`
- `historical_universe_membership.parquet`
- `theme_classification.parquet`
- `source_routing_table.parquet`
- `block_1_manifest.json`

Downstream source engines should consume these outputs rather than independently reconstructing issuer, security, listing, theme, or routing identity.

### Identity governance

A legal-entity identifier such as an LEI anchors a legal entity, not automatically an economic issuer. Multiple legal entities may be merged into one economic issuer only through explicit, versionable, validated relationship evidence. Weak semantic similarity may generate a review candidate but must not silently alter canonical identity.

### Point-in-time governance

`source_available_datetime` records when a universe observation became publicly available. It must never be replaced by a holdings snapshot date or reporting-period end date. Historical membership intervals are therefore based on information availability rather than hindsight.


### Identity reference-data policy

Legal-entity resolution is industry-agnostic. Observed LEIs are enriched from the public GLEIF API and cached under `data/reference_cache/gleif`. No GLEIF API key is required. External evidence validates or quarantines source identifiers; it never introduces company-specific hard-coded mappings. A security may have multiple legal entities through time, but must resolve to one economic issuer. OpenAI is invoked automatically only when deterministic semantic logic is insufficient; it remains non-authoritative and can never assign canonical identity truth.
